# Detailed implementation archive

The streamlined report is [practical_1.ipynb](practical_1.ipynb). This notebook preserves the original detailed experiment implementation, cache/resume code and historical policies. All training switches remain false. Its original report content follows.

# Practical 1: Text classification with SNLI

How much can simple text representations tell us about the relationship between
two sentences? We investigate **entailment, neutral and contradiction** using
BPE tokens, sparse features, logistic regression and two small neural networks.

Our questions are:

- Do token counts help compared with token presence?
- Does local token order help when we replace unigrams with bigrams?
- What do hidden layers add, and when do they overfit?
- How much do tokenizer and training settings affect the result?

All implementation code used during Run All is written in the notebook;
there are no local Python-helper imports, source-file execution or notebook
loader shortcuts. The long-run CV/analysis functions are defined here too,
with explicit switches controlling their execution.

We follow the explanation → example → model → experiment → interpretation style
of [this course practical](https://github.com/S-Chaudhuri/uva_nlp1_2025/blob/main/NLP1_2025_Practical_2_%28student%29.ipynb).
The task and requirements remain those of [our Project 1](project.ipynb).

**Where we are:** the baseline models, paired cross-validation and Bayesian
comparisons have been run. Joint tokenizer/model tuning is **implemented in
section 6 and its short SNLI search completed in 7.97 minutes**. The existing 62.75% best mean
CV accuracy remains a baseline result. New tuned CV/Bayesian evaluation and
sections 9–11 remain outstanding.

Run the notebook from top to bottom using the repository's `.venv` kernel.
Short demonstrations run here; the larger experiments are read from their saved,
checked reports. Commands and provenance are collected in the appendix.

## 1. Getting started

First, locate the repository and import the packages we need. The implementation is written in this notebook: data preparation, BPE,
feature construction, classifier classes, training, scoring, CV and Bayesian
analysis are executable definitions in the cells below. Imports are limited
to standard-library and third-party packages.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "part1" / "project.ipynb").is_file():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository root or part1.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository:", REPO_ROOT)
print("Kernel Python:", sys.executable)
print("Environment prefix:", sys.prefix)

In [ ]:
import json
import os
import sys
import platform
import random
import subprocess
import hashlib
import tempfile
import warnings
import traceback
import threading
import gc
import re
import fcntl
from array import array
from collections import Counter
from contextlib import contextmanager
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib import metadata
from importlib.metadata import version
from itertools import chain, combinations
from pathlib import Path
from time import perf_counter, time_ns
from typing import Any, Callable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import scipy
import scipy.stats as st
from scipy import sparse
from scipy.stats import t
import torch
from torch import nn
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import joblib
import psutil
import nbformat
import baycomp
import baycomp.single
import tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from datasets import ClassLabel, Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk
from IPython.display import Image, display

### Configuration and reproducibility

The following functions are defined here and called by this notebook. The
`.py` files remain available for the archived command-line workflow; notebook
execution does not import or execute them. New notebook-generated artifacts
record the hash of these code cells, keeping baseline artifacts separate.

In [ ]:
NOTEBOOK_PATH = REPO_ROOT / "part1/practical_1_details.ipynb"
DEFAULT_CONFIG = REPO_ROOT / "part1/configs/snli_project1.json"
FROZEN_CONFIG = REPO_ROOT / "part1/configs/snli_project1_frozen_bigram.json"
ARTIFACT_FOLDERS = ("manifests", "figures", "metrics", "predictions", "checkpoints", "folds", "features")

def notebook_source_sha256():
    """Stable source identity: saved code cells only, excluding outputs."""
    saved = nbformat.read(NOTEBOOK_PATH, 4)
    source = "\n\n".join(cell.source for cell in saved.cells if cell.cell_type == "code")
    return hashlib.sha256(source.encode("utf-8")).hexdigest()

### Seeds and workspace

Record the environment and reuse the configured data/output locations.

In [ ]:
def set_random_seeds(seed: int) -> None:
    """Seed the global generators used by the tutorial's training code.

    Dedicated generators, CV splits, and sampling still need their own explicit
    seeds. This does not promise identical results across platforms/devices.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def _setup_git_output(*arguments: str) -> str:
    return subprocess.run(['git', *arguments], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()

In [ ]:
def prepare_workspace(config_path: str | Path=DEFAULT_CONFIG) -> dict[str, Any]:
    """Read the config, seed libraries, and capture a local setup manifest.

    Configuration paths are relative to the repository root. Returned artifact
    paths are absolute so the notebook also works when launched in ``part1``.
    Rerunning replaces the latest setup manifest; later training runs will need
    their own manifests rather than using this setup record as a run history.
    """
    config_path = Path(config_path)
    if not config_path.is_absolute():
        config_path = REPO_ROOT / config_path
    config_bytes = config_path.read_bytes()
    config = json.loads(config_bytes)
    set_random_seeds(config['seeds']['model'])
    output_dir = REPO_ROOT / config['paths']['output_dir']
    data_dir = REPO_ROOT / config['paths']['data_dir']
    paths = {'output': output_dir, 'data': data_dir}
    paths.update({name: output_dir / name for name in ARTIFACT_FOLDERS})
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    environment = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True).stdout
    environment_path = paths['manifests'] / 'environment.txt'
    environment_path.write_text(environment, encoding='utf-8')
    git_status = _setup_git_output('status', '--porcelain')
    manifest = {'stage': 'workspace_setup', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'experiment_name': config['experiment_name'], 'configuration': config, 'config_sha256': hashlib.sha256(config_bytes).hexdigest(), 'dataset_loaded': False, 'python': {'executable': sys.executable, 'version': platform.python_version(), 'prefix': sys.prefix}, 'hardware': {'platform': platform.platform(), 'machine': platform.machine(), 'processor': platform.processor(), 'logical_cpu_count': os.cpu_count(), 'configured_device': config['device'], 'cuda_available': torch.cuda.is_available()}, 'packages': {distribution.metadata['Name']: distribution.version for distribution in metadata.distributions()}, 'repository': {'commit': _setup_git_output('rev-parse', 'HEAD'), 'dirty': bool(git_status), 'status': git_status.splitlines()}, 'artifacts': {'setup_manifest': str(paths['manifests'] / 'setup.json'), 'environment': str(environment_path)}}
    manifest_path = Path(manifest['artifacts']['setup_manifest'])
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return {'config': config, 'paths': paths, 'manifest': manifest}

In [ ]:
workspace = prepare_workspace()
config, manifest, paths = workspace["config"], workspace["manifest"], workspace["paths"]
baseline_config = json.loads(FROZEN_CONFIG.read_text())
set_random_seeds(0)
print(f"Python {manifest['python']['version']} | PyTorch {torch.__version__} | CPU")

## 2. Preparing the data

Each SNLI example contains a **premise**, a **hypothesis**, and one of three
labels. Entailment means the hypothesis follows from the premise; contradiction
means it conflicts with it; neutral means neither conclusion is supported.

We load the pinned Hugging Face dataset, remove invalid labels (`-1`) and blank
text, and retain original row identifiers. An offline-cache message means the
loader is reusing the downloaded dataset. It does not mean the data is absent.

In [ ]:
SPLITS = ("train", "validation", "test")
LABEL_NAMES = ("entailment", "neutral", "contradiction")
TEXT_COLUMNS = ("premise", "hypothesis")
ID_COLUMNS = ("example_id", "source_split", "source_row")
CLEANING_VERSION = "v1"

### Cleaning and original row IDs

Keep valid labelled text and preserve its source split and row number.

In [ ]:
def validate_label_mapping(dataset: Dataset) -> None:
    """Reject an unexpected schema rather than silently changing label meaning."""
    required = {*TEXT_COLUMNS, 'label'}
    if not required.issubset(dataset.column_names):
        raise ValueError(f'SNLI requires columns {sorted(required)}.')
    label = dataset.features.get('label')
    if not isinstance(label, ClassLabel) or label.names != list(LABEL_NAMES):
        raise ValueError(f'Unexpected SNLI label mapping: {label!r}')

In [ ]:
def _data_has_text(value: Any) -> bool:
    return isinstance(value, str) and bool(value.strip())

In [ ]:
def clean_split(dataset: Dataset, split: str) -> tuple[Dataset, pd.DataFrame]:
    """Filter unusable rows, retaining original text, labels, order, and row IDs.

    Invalid labels take precedence in the removal audit when a row also has
    empty text, so exclusion reasons are mutually exclusive. Duplicates remain.
    """
    validate_label_mapping(dataset)
    frame = dataset.to_pandas()
    frame['source_row'] = np.arange(len(frame), dtype=np.int64)
    frame['source_split'] = split
    frame['example_id'] = [f'{split}:{row}' for row in frame['source_row']]
    valid_label = frame['label'].isin(range(len(LABEL_NAMES)))
    valid_text = frame['premise'].map(_data_has_text) & frame['hypothesis'].map(_data_has_text)
    keep = valid_label & valid_text
    removed = frame.loc[~keep, [*ID_COLUMNS, 'label']].copy()
    removed['reason'] = np.where(~valid_label.loc[~keep], 'invalid_label', 'missing_or_empty_text')
    features = Features({**dataset.features, 'source_row': Value('int64'), 'source_split': Value('string'), 'example_id': Value('string')})
    cleaned = Dataset.from_pandas(frame.loc[keep], features=features, preserve_index=False)
    return (cleaned, removed.reset_index(drop=True))

### Data audit and loading

Compute the split/label/length/overlap summaries, save the cleaned dataset,
and return the tables and figures used below.

In [ ]:
def build_data_report(raw: DatasetDict, cleaned: DatasetDict, removed: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Summarize labels, lengths, duplicates, and exact cross-split overlap.

    Lengths use Python whitespace splitting for exploration, not a learned BPE
    tokenizer. Overlap uses exact original strings, without normalization.
    """
    frames = {split: cleaned[split].to_pandas() for split in SPLITS}
    summaries, labels, lengths, duplicate_groups = ([], [], [], [])
    premise_sets: dict[str, set[str]] = {}
    pair_sets: dict[str, set[tuple[str, str]]] = {}
    for split, frame in frames.items():
        source = raw[split].to_pandas()
        exclusions = removed.loc[removed['source_split'] == split, 'reason']
        premise_counts = frame['premise'].value_counts()
        repeated_pairs = frame.loc[frame.duplicated(list(TEXT_COLUMNS), keep=False)]
        conflicts = 0
        for (premise, hypothesis), group in repeated_pairs.groupby(list(TEXT_COLUMNS), sort=False):
            unique_labels = sorted((int(label) for label in group['label'].unique()))
            conflicts += len(unique_labels) > 1
            duplicate_groups.append({'split': split, 'premise': premise, 'hypothesis': hypothesis, 'rows': len(group), 'distinct_labels': len(unique_labels), 'labels': json.dumps(unique_labels), 'example_ids': json.dumps(group['example_id'].tolist())})
        summaries.append({'split': split, 'raw_rows': len(source), 'invalid_label_rows': int((exclusions == 'invalid_label').sum()), 'invalid_text_rows_after_label_filter': int((exclusions == 'missing_or_empty_text').sum()), 'raw_empty_or_missing_premise': int((~source['premise'].map(_data_has_text)).sum()), 'raw_empty_or_missing_hypothesis': int((~source['hypothesis'].map(_data_has_text)).sum()), 'clean_rows': len(frame), 'unique_premises': frame['premise'].nunique(), 'repeated_premise_groups': int((premise_counts > 1).sum()), 'rows_in_repeated_premise_groups': int(premise_counts[premise_counts > 1].sum()), 'largest_premise_group': int(premise_counts.max()) if len(frame) else 0, 'extra_duplicate_pair_rows': int(frame.duplicated(list(TEXT_COLUMNS)).sum()), 'conflicting_label_pair_groups': conflicts})
        counts = frame['label'].value_counts().reindex(range(len(LABEL_NAMES)), fill_value=0)
        for label, name in enumerate(LABEL_NAMES):
            labels.append({'split': split, 'label': label, 'label_name': name, 'count': int(counts[label]), 'proportion': float(counts[label] / len(frame)) if len(frame) else 0.0})
        per_row = frame[['example_id']].copy()
        per_row['split'] = split
        for field in TEXT_COLUMNS:
            per_row[f'{field}_characters'] = frame[field].map(len)
            per_row[f'{field}_whitespace_tokens'] = frame[field].map(lambda text: len(text.split()))
        lengths.append(per_row)
        premise_sets[split] = set(frame['premise'])
        pair_sets[split] = set(zip(frame['premise'], frame['hypothesis']))
    row_lengths = pd.concat(lengths, ignore_index=True)
    length_summary = []
    for split in SPLITS:
        split_lengths = row_lengths.loc[row_lengths['split'] == split]
        for field in TEXT_COLUMNS:
            for unit in ('characters', 'whitespace_tokens'):
                values = split_lengths[f'{field}_{unit}']
                length_summary.append({'split': split, 'field': field, 'unit': unit, 'min': values.min(), 'mean': values.mean(), 'median': values.median(), 'p95': values.quantile(0.95), 'max': values.max()})
    overlaps = []
    for left, right in combinations(SPLITS, 2):
        shared_premises = premise_sets[left] & premise_sets[right]
        shared_pairs = pair_sets[left] & pair_sets[right]
        overlaps.append({'left_split': left, 'right_split': right, 'shared_unique_premises': len(shared_premises), 'left_rows_with_shared_premise': int(frames[left]['premise'].isin(shared_premises).sum()), 'right_rows_with_shared_premise': int(frames[right]['premise'].isin(shared_premises).sum()), 'shared_unique_pairs': len(shared_pairs)})
    examples = frames['train'].drop_duplicates('label').sort_values('label').copy()
    examples['label_name'] = examples['label'].map(dict(enumerate(LABEL_NAMES)))
    return {'split_summary': pd.DataFrame(summaries), 'label_counts': pd.DataFrame(labels), 'length_summary': pd.DataFrame(length_summary), 'row_lengths': row_lengths, 'split_overlap': pd.DataFrame(overlaps), 'duplicate_pairs': pd.DataFrame(duplicate_groups, columns=['split', 'premise', 'hypothesis', 'rows', 'distinct_labels', 'labels', 'example_ids']), 'removed_rows': removed, 'training_examples': examples[['example_id', 'premise', 'hypothesis', 'label', 'label_name']]}

In [ ]:
def plot_data_report(tables: dict[str, pd.DataFrame], figure_dir: Path) -> dict[str, Path]:
    """Save label proportions and full-range training-length distributions."""
    figure_dir.mkdir(parents=True, exist_ok=True)
    proportions = tables['label_counts'].pivot(index='split', columns='label_name', values='proportion').reindex(index=list(SPLITS), columns=list(LABEL_NAMES))
    fig, ax = plt.subplots(figsize=(8, 4))
    proportions.plot.bar(ax=ax, rot=0, color=['#3478a5', '#db9c36', '#b45454'])
    ax.set(xlabel='Official split (cleaned)', ylabel='Class proportion', title='SNLI class balance')
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.legend(title='Label', bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    class_path = figure_dir / 'snli_class_balance.png'
    fig.savefig(class_path, dpi=160)
    plt.close(fig)
    training_lengths = tables['row_lengths'].loc[tables['row_lengths']['split'] == 'train']
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, field, color in zip(axes, TEXT_COLUMNS, ['#3478a5', '#db9c36']):
        values = training_lengths[f'{field}_whitespace_tokens']
        ax.hist(values, bins=40, color=color, edgecolor='white')
        ax.axvline(values.median(), color='#333333', linestyle='--', label='Median')
        ax.set(xlabel='Whitespace-separated tokens', ylabel='Training examples', title=field.title())
        ax.legend()
    fig.suptitle('SNLI training text lengths (exploratory units, not BPE)')
    fig.tight_layout()
    length_path = figure_dir / 'snli_training_lengths.png'
    fig.savefig(length_path, dpi=160)
    plt.close(fig)
    return {'class_balance': class_path, 'training_lengths': length_path}

In [ ]:
def prepare_snli_report(config_path: str | Path=DEFAULT_CONFIG) -> dict[str, Any]:
    """Load the source, save cleaned splits, and persist the structural audit."""
    workspace = prepare_workspace(config_path)
    config, paths = (workspace['config'], workspace['paths'])
    source = config['dataset']
    raw = load_dataset(source['repo_id'], revision=source['revision'], cache_dir=str(paths['data'] / 'huggingface'))
    if not isinstance(raw, DatasetDict) or set(raw) != set(SPLITS):
        raise ValueError('Expected the official train, validation, and test splits.')
    cleaned, audits = (DatasetDict(), [])
    for split in SPLITS:
        cleaned[split], audit = clean_split(raw[split], split)
        audits.append(audit)
    removed = pd.concat(audits, ignore_index=True)
    tables = build_data_report(raw, cleaned, removed)
    prepared_dir = paths['data'] / 'snli' / source['revision'] / f'cleaned-{CLEANING_VERSION}'
    cleaned.save_to_disk(str(prepared_dir))
    report_dir = paths['metrics'] / 'data_audit'
    report_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(report_dir / f'{name}.csv', index=False)
    figures = plot_data_report(tables, paths['figures'])
    manifest = {'stage': 'data_inspection', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'dataset': source, 'dataset_loaded': True, 'config_sha256': workspace['manifest']['config_sha256'], 'repository': workspace['manifest']['repository'], 'label_mapping': dict(enumerate(LABEL_NAMES)), 'cleaning_version': CLEANING_VERSION, 'cleaning_policy': {'exclude': ['label outside {0, 1, 2}', 'missing or whitespace-only sentence'], 'retain': ['original split', 'original text', 'original gold label', 'duplicate pairs'], 'id_scheme': 'official_split:original_zero_based_row', 'overlap_matching': 'exact strings, without normalization', 'example_selection': 'first retained training row of each class'}, 'splits': tables['split_summary'].to_dict(orient='records'), 'source_fingerprints': {split: raw[split]._fingerprint for split in SPLITS}, 'cleaned_fingerprints': {split: cleaned[split]._fingerprint for split in SPLITS}, 'artifacts': {'prepared_dataset': str(prepared_dir), 'tables': str(report_dir), 'figures': {name: str(path) for name, path in figures.items()}}, 'test_usage': 'Structural/count audit only; no predictions, scoring, or model selection.'}
    manifest_path = paths['manifests'] / 'data_inspection.json'
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    return {'raw': raw, 'cleaned': cleaned, 'tables': tables, 'figures': figures, 'manifest': manifest, 'manifest_path': manifest_path, 'prepared_dir': prepared_dir}

In [ ]:
from datasets import disable_progress_bars as disable_dataset_progress
from huggingface_hub.utils import disable_progress_bars as disable_hub_progress

disable_dataset_progress()
disable_hub_progress()
snli_report = prepare_snli_report()
dataset, data_tables = snli_report["cleaned"], snli_report["tables"]
label_names = dict(enumerate(LABEL_NAMES))
display(dataset)
print("Labels:", label_names)

### Looking at an example

Read the premise before the hypothesis. What would make a hypothesis neutral
rather than contradictory? The following examples come from the training split.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(data_tables["training_examples"])

### How much data do we have?

The cleaned splits contain 549,367 training, 9,842 validation and 9,824 test
pairs. Removed rows are recorded rather than silently discarded. These are
source split sizes; the leakage-aware development pools are defined below.

In [ ]:
display(data_tables["split_summary"][[
    "split", "raw_rows", "invalid_label_rows", "clean_rows",
]])

### Label balance and sentence lengths

The training labels are close to balanced, so accuracy is an interpretable
primary metric. Macro-F1 checks class-level performance, and NLL measures the
probability assigned to the gold label. Structural test-split counts below are
part of the data audit; no test model scores are computed.

In [ ]:
training_balance = data_tables["label_counts"].query("split == 'train'").copy()
training_balance["percentage"] = 100 * training_balance["proportion"]
display(training_balance[["label_name", "count", "percentage"]].round(2))
display(Image(filename=str(snli_report["figures"]["class_balance"])))

Longer text can create more feature occurrences. These exploratory lengths use
whitespace splitting; BPE segmentation can produce a different number of tokens.

In [ ]:
display(data_tables["length_summary"].round(2))
display(Image(filename=str(snli_report["figures"]["training_lengths"])))

### Repeated premises

Several hypotheses can share one premise. Splitting those examples across a
training and evaluation fold would let the same premise appear on both sides.
We therefore split by premise groups. The original official splits also share
a small number of premises; our development pools exclude these overlaps.

In [ ]:
quality_columns = [
    "split", "unique_premises", "repeated_premise_groups",
    "rows_in_repeated_premise_groups", "largest_premise_group",
    "extra_duplicate_pair_rows", "conflicting_label_pair_groups",
]
display(data_tables["split_summary"][quality_columns])
display(data_tables["split_overlap"])

## 3. Training, validation and evaluation

The splits have different jobs:

| Data | Purpose |
| --- | --- |
| Eligible official training rows | Fit tokenizers, feature inventories and classifiers |
| Eligible official validation rows | Choose tokenizer/model settings and checkpoints |
| Outer CV held-out groups | Compare systems after that fold's selection |
| Official test | Evaluate the final selected pipeline once |

There are 549,253 eligible training rows and 9,839 eligible validation rows after
premise-overlap exclusions. The full validation pool is available for the next
tuning stage. The older baseline used a 60,005-row training subset; its small
pilot used only 1,202 validation examples. These choices are historical, not a
requirement to keep that budget or training schedule.

In [ ]:
PROTOCOL_VERSION = "v1"
POOL_FLAGS = ("cv_pool", "pilot_validation", "final_refit", "final_test")
ROW_COLUMNS = ["pool_index", "clean_index", *ID_COLUMNS, "premise_group", "label"]

### Grouping and leakage-aware pools

The complete selection and fold-construction code follows. The old budget
option is retained to reproduce the baseline.

In [ ]:
def _evaluation_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

In [ ]:
def select_complete_groups(frame: pd.DataFrame, target_rows: int, seed: int) -> pd.DataFrame:
    """Sample a label-independent group prefix, preserving original row order.

    Include the last complete group needed to reach the target. The actual row
    count can exceed the target by less than the last group's size.
    """
    if not isinstance(target_rows, int) or isinstance(target_rows, bool) or (not 1 <= target_rows <= len(frame)):
        raise ValueError('Group-sampling target must be between one and the eligible row count.')
    sizes = frame.groupby('premise_group', sort=True).size()
    shuffled = sizes.iloc[np.random.default_rng(seed).permutation(len(sizes))]
    count = int(np.searchsorted(shuffled.cumsum().to_numpy(), target_rows)) + 1
    return frame.loc[frame['premise_group'].isin(shuffled.index[:count])].copy().reset_index(drop=True)

In [ ]:
def apply_cv_group_budget(pools: dict[str, pd.DataFrame], settings: dict[str, Any]) -> None:
    """Derive a separate budgeted CV selection without changing holdout pools."""
    selected = select_complete_groups(pools['cv_pool'], settings['target_rows'], settings['seed'])
    kept = set(selected['example_id'])
    excluded = pools['cv_pool'].loc[~pools['cv_pool']['example_id'].isin(kept)].copy()
    excluded = excluded[['clean_index', *ID_COLUMNS, 'premise_group']]
    excluded['pool'], excluded['reason'] = ('cv_pool', 'not_selected_by_fixed_group_budget')
    pools['excluded_rows'] = pd.concat([pools['excluded_rows'], excluded], ignore_index=True)
    selected['pool_index'] = np.arange(len(selected), dtype=np.int64)
    pools['cv_pool'] = selected[ROW_COLUMNS]
    pools['membership']['cv_pool'] = pools['membership']['example_id'].isin(kept)
    mask = pools['pool_summary']['pool'].eq('cv_pool')
    pools['pool_summary'].loc[mask, 'rows'] = len(selected)
    pools['pool_summary'].loc[mask, 'premise_groups'] = selected['premise_group'].nunique()

In [ ]:
def make_development_pools(cleaned: DatasetDict) -> dict[str, pd.DataFrame]:
    """Preserve reference splits and derive selections with holdout priority.

    CV uses training rows whose exact premise is absent from both official
    holdouts. Pilot validation excludes test premises. Final refitting may use
    original train + validation, excluding test premises from both. Selection
    does not inspect validation/test labels, or any hypotheses.
    """
    if set(cleaned) != set(SPLITS):
        raise ValueError('Expected cleaned train, validation, and test splits.')
    frames = {}
    for split in SPLITS:
        frame = cleaned[split].select_columns([*ID_COLUMNS, 'premise']).to_pandas()
        if frame.empty or frame.isna().any().any():
            raise ValueError(f'{split}: expected nonempty, cleaned source metadata.')
        if not frame['source_split'].eq(split).all():
            raise ValueError(f'{split}: inconsistent original split identifiers.')
        expected_ids = split + ':' + frame['source_row'].astype(str)
        if not frame['example_id'].equals(expected_ids.rename('example_id')):
            raise ValueError(f'{split}: inconsistent original row identifiers.')
        frame.insert(0, 'clean_index', np.arange(len(frame), dtype=np.int64))
        frames[split] = frame
    membership = pd.concat(frames.values(), ignore_index=True)
    if membership['example_id'].duplicated().any():
        raise ValueError('Source example IDs must be globally unique.')
    premise_ids = {text: hashlib.sha256(text.encode('utf-8')).hexdigest() for text in membership['premise'].unique()}
    membership['premise_group'] = membership['premise'].map(premise_ids)
    in_validation = membership['premise'].isin(frames['validation']['premise'])
    in_test = membership['premise'].isin(frames['test']['premise'])
    is_train = membership['source_split'].eq('train')
    is_validation = membership['source_split'].eq('validation')
    membership['cv_pool'] = is_train & ~in_validation & ~in_test
    membership['pilot_validation'] = is_validation & ~in_test
    membership['final_refit'] = (is_train | is_validation) & ~in_test
    membership['final_test'] = membership['source_split'].eq('test')
    exclusions = []
    for pool, candidates in (('cv_pool', is_train), ('pilot_validation', is_validation), ('final_refit', is_train | is_validation)):
        mask = candidates & ~membership[pool]
        excluded = membership.loc[mask, ['clean_index', *ID_COLUMNS, 'premise_group']].copy()
        excluded['pool'] = pool
        excluded['reason'] = [';'.join((reason for applies, reason in ((pool == 'cv_pool' and in_validation.loc[row], 'premise_in_official_validation'), (in_test.loc[row], 'premise_in_official_test')) if applies)) for row in excluded.index]
        exclusions.append(excluded)
    train_labels = np.asarray(cleaned['train']['label'])
    cv_pool = membership.loc[membership['cv_pool']].copy().reset_index(drop=True)
    cv_pool.insert(0, 'pool_index', np.arange(len(cv_pool), dtype=np.int64))
    cv_pool['label'] = train_labels[cv_pool['clean_index'].to_numpy()]
    if not cv_pool['label'].isin(range(len(LABEL_NAMES))).all():
        raise ValueError('The CV pool must contain only cleaned labels 0, 1, 2.')
    summary = []
    for pool in POOL_FLAGS:
        selected = membership.loc[membership[pool]]
        summary.append({'pool': pool, 'rows': len(selected), 'premise_groups': selected['premise_group'].nunique()})
    groups = {pool: set(membership.loc[membership[pool], 'premise_group']) for pool in POOL_FLAGS}
    if groups['cv_pool'] & groups['pilot_validation'] or groups['final_refit'] & groups['final_test']:
        raise ValueError('Development/holdout premise overlap remains after exclusion.')
    return {'cv_pool': cv_pool[ROW_COLUMNS], 'membership': membership.drop(columns='premise'), 'excluded_rows': pd.concat(exclusions, ignore_index=True), 'pool_summary': pd.DataFrame(summary)}

In [ ]:
def validate_folds(assignments: pd.DataFrame, cv_pool: pd.DataFrame, num_folds: int, seeds: list[int]) -> pd.DataFrame:
    """Check row alignment, coverage, class presence, and group separation."""
    if assignments['example_id'].duplicated().any():
        raise ValueError('Each CV example must have exactly one assignment row.')
    if not assignments[ROW_COLUMNS].equals(cv_pool[ROW_COLUMNS]):
        raise ValueError("Saved folds do not match the CV pool's rows/order/labels/groups.")
    expected_columns = ROW_COLUMNS + [f'fold_seed_{seed}' for seed in seeds]
    if assignments.columns.tolist() != expected_columns:
        raise ValueError('Unexpected fold columns or repetition order.')
    summaries = []
    labels = assignments['label'].to_numpy()
    for repetition, seed in enumerate(seeds, start=1):
        column = f'fold_seed_{seed}'
        folds = assignments[column]
        if not pd.api.types.is_integer_dtype(folds) or not folds.isin(range(num_folds)).all():
            raise ValueError(f'Seed {seed}: every row needs one valid integer held-out fold.')
        if assignments.groupby('premise_group')[column].nunique().gt(1).any():
            raise ValueError(f'Seed {seed}: a premise group crosses held-out folds.')
        coverage = np.zeros(len(assignments), dtype=np.int8)
        for fold in range(num_folds):
            heldout = folds.to_numpy() == fold
            coverage += heldout
            train_counts = np.bincount(labels[~heldout], minlength=len(LABEL_NAMES))
            heldout_counts = np.bincount(labels[heldout], minlength=len(LABEL_NAMES))
            if not train_counts.all() or not heldout_counts.all():
                raise ValueError(f'Seed {seed}, fold {fold}: a partition is empty or lacks a class.')
            row = {'repetition': repetition, 'seed': seed, 'fold': fold, 'train_rows': int((~heldout).sum()), 'heldout_rows': int(heldout.sum()), 'train_groups': assignments.loc[~heldout, 'premise_group'].nunique(), 'heldout_groups': assignments.loc[heldout, 'premise_group'].nunique(), 'shared_premise_groups': 0}
            for label, name in enumerate(LABEL_NAMES):
                row[f'train_{name}'] = int(train_counts[label])
                row[f'heldout_{name}'] = int(heldout_counts[label])
                row[f'heldout_{name}_proportion'] = float(heldout_counts[label] / heldout.sum())
            summaries.append(row)
        if not np.all(coverage == 1):
            raise ValueError(f'Seed {seed}: held-out coverage is not exactly once per row.')
    return pd.DataFrame(summaries)

In [ ]:
def build_grouped_folds(cv_pool: pd.DataFrame, num_folds: int, seeds: list[int], *, verbose: bool=False) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Assign complete exact-premise groups to approximately stratified folds."""
    if num_folds < 2 or cv_pool['premise_group'].nunique() < num_folds:
        raise ValueError('CV needs at least two folds and at least one group per fold.')
    if not seeds or len(set(seeds)) != len(seeds):
        raise ValueError('Provide a nonempty list of distinct CV split seeds.')
    assignments = cv_pool[ROW_COLUMNS].copy()
    for seed in seeds:
        if verbose:
            print(f'Preparing {num_folds} grouped folds for seed {seed}...', flush=True)
        splitter = StratifiedGroupKFold(n_splits=num_folds, shuffle=True, random_state=seed)
        fold_ids = np.full(len(cv_pool), -1, dtype=np.int16)
        for fold, (_, heldout) in enumerate(splitter.split(np.zeros(len(cv_pool)), cv_pool['label'], groups=cv_pool['premise_group'])):
            fold_ids[heldout] = fold
        assignments[f'fold_seed_{seed}'] = fold_ids
    return (assignments, validate_folds(assignments, cv_pool, num_folds, seeds))

In [ ]:
def prepare_evaluation_protocol(config_path: str | Path=DEFAULT_CONFIG, *, verbose: bool=False) -> dict[str, Any]:
    """Save once and reuse checked fold assignments for the same data/design.

    A changed pool, seed, splitter version, or evaluation policy gets a new
    protocol directory. Training hyperparameters do not change the folds.
    Checksums reject modified/incomplete saved artifacts instead of overwriting.
    """
    workspace = prepare_workspace(config_path)
    config, paths = (workspace['config'], workspace['paths'])
    prepared_dir = paths['data'] / 'snli' / config['dataset']['revision'] / f'cleaned-{CLEANING_VERSION}'
    if not prepared_dir.is_dir():
        raise FileNotFoundError(f'Run Step 2 first: python -m part1.snli_data --config {config_path}')
    cleaned = load_from_disk(str(prepared_dir))
    if not isinstance(cleaned, DatasetDict):
        raise ValueError('Expected the saved cleaned DatasetDict from Step 2.')
    pools = make_development_pools(cleaned)
    cv_config = config['cross_validation']
    if cv_config['group_by'] != 'premise' or cv_config['splitter'] != 'StratifiedGroupKFold' or (not cv_config['shuffle']):
        raise ValueError('This protocol requires shuffled, exact-premise StratifiedGroupKFold.')
    if cv_config['pool'] == 'fixed_group_subset':
        apply_cv_group_budget(pools, cv_config['subset'])
    elif cv_config['pool'] != 'all_eligible_training_rows':
        raise ValueError('Expected all_eligible_training_rows or fixed_group_subset.')
    seeds, num_folds = (config['seeds']['cv_splits'], cv_config['num_folds'])
    policy = config['evaluation']['pool_policy']
    if policy != 'test_priority_exclude_train_overlap_with_both_holdouts_v1':
        raise ValueError(f'Unsupported pool policy: {policy}')
    specification = {'protocol_version': PROTOCOL_VERSION, 'dataset': config['dataset'], 'cleaning_version': CLEANING_VERSION, 'cleaned_fingerprints': {split: cleaned[split]._fingerprint for split in SPLITS}, 'cross_validation': cv_config, 'split_seeds': seeds, 'evaluation': config['evaluation'], 'sklearn_version': sklearn.__version__}
    protocol_id = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:16]
    protocol_dir = paths['folds'] / protocol_id
    manifest_path = protocol_dir / 'protocol.json'
    reused = protocol_dir.exists()
    if reused:
        if not manifest_path.is_file():
            raise ValueError(f'Incomplete protocol directory: {protocol_dir}. No saved folds were overwritten.')
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['specification'] != specification:
            raise ValueError('Saved protocol specification does not match the current data/design.')
        for name, digest in manifest['artifact_sha256'].items():
            path = protocol_dir / name
            if not path.is_file() or _evaluation_sha256(path) != digest:
                raise ValueError(f'Saved evaluation artifact changed or is missing: {path}')
        assignments = pd.read_parquet(protocol_dir / 'fold_assignments.parquet')
        fold_summary = validate_folds(assignments, pools['cv_pool'], num_folds, seeds)
    else:
        assignments, fold_summary = build_grouped_folds(pools['cv_pool'], num_folds, seeds, verbose=verbose)
        protocol_dir.mkdir()
        assignments.to_parquet(protocol_dir / 'fold_assignments.parquet', index=False)
        pools['membership'].to_parquet(protocol_dir / 'pool_membership.parquet', index=False)
        for name, frame in (('excluded_rows', pools['excluded_rows']), ('pool_summary', pools['pool_summary']), ('fold_summary', fold_summary)):
            frame.to_csv(protocol_dir / f'{name}.csv', index=False)
        manifest = {'stage': 'evaluation_design', 'protocol_id': protocol_id, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'specification': specification, 'creation_config_sha256': workspace['manifest']['config_sha256'], 'repository': workspace['manifest']['repository'], 'implementation_sha256': notebook_source_sha256(), 'prepared_dataset': str(prepared_dir), 'pools': pools['pool_summary'].to_dict(orient='records'), 'checks': {'unique_example_ids': True, 'original_row_order_preserved': True, 'premise_disjoint_cv_partitions': True, 'one_heldout_assignment_per_row_per_repetition': True, 'all_classes_present_in_each_partition': True, 'cv_disjoint_from_official_holdout_premises': True, 'pilot_validation_disjoint_from_test_premises': True, 'final_refit_disjoint_from_test_premises': True, 'cleaned_official_reference_splits_unchanged': True}, 'models_trained': False, 'test_usage': 'Premises and source IDs only for exclusions; no test labels/hypotheses or model scores.', 'selection_policy': {'pilot': 'Tune limited settings using eligible training groups and pilot validation; fix before reported CV.', 'reported_cv': 'Fit tokenizer/features/model on each outer training partition only; use fixed epochs from pilot.', 'pairing': 'All systems/baselines reuse the saved pool_index order and fold_seed columns.', 'budget_change': 'If the pilot requires fewer complete groups, save a separately identified subset/protocol before comparisons.', 'final_refit': 'Freeze selected system; use original cleaned train + validation excluding test premises; test once.'}, 'artifact_sha256': {path.name: _evaluation_sha256(path) for path in sorted(protocol_dir.iterdir()) if path.is_file()}}
        manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    latest_path = paths['manifests'] / 'evaluation_design.json'
    latest_path.write_text(json.dumps({'protocol_id': protocol_id, 'protocol_manifest': str(manifest_path), 'current_config_sha256': workspace['manifest']['config_sha256']}, indent=2) + '\n', encoding='utf-8')
    return {**pools, 'assignments': assignments, 'fold_summary': fold_summary, 'manifest': manifest, 'manifest_path': manifest_path, 'protocol_dir': protocol_dir, 'reused_folds': reused}

In [ ]:
evaluation_protocol = prepare_evaluation_protocol()
display(evaluation_protocol["pool_summary"])
assert all(evaluation_protocol["manifest"]["checks"].values())

### Cross-validation

Our baseline uses five grouped folds, repeated with split seeds `[0, 3, 5]`.
Each row is held out once per repetition. All systems see matching partitions,
and each tokenizer is learned inside its training fold. A shared premise and
hypothesis vocabulary is used within that fold.

The table below describes the saved full-data reference design. Section 7 reads
the separate completed subset experiment. Different folds may learn different
BPE coordinates; the models are fitted with their own fold's mappings.

In [ ]:
fold_summary = evaluation_protocol["fold_summary"]
display(fold_summary[[
    "repetition", "seed", "fold", "train_rows", "heldout_rows", "shared_premise_groups",
]])
assert fold_summary["shared_premise_groups"].eq(0).all()

## 4. From sentences to features

### BPE vocabulary

BPE starts with small units and merges frequent adjacent units. The vocabulary
size and minimum merge frequency control segmentation; they are hyperparameters,
not properties fixed by SNLI. A BPE token may be a whole word or a subword.

For worked examples, load the tokenizer and matrices from **baseline fold 0**.
Its tokenizer was fitted on 48,003 training pairs, with vocabulary target 5,000.
Using its saved configuration reproduces the baseline; section 6 defines the
larger vocabulary search we still need to run.

In [ ]:
FEATURE_VERSION = "v2"
REPRESENTATIONS = ("indicators", "unigram_counts", "bigram_counts")

### Training the BPE tokenizer

This is the tokenizer-training function used by the feature pipeline. It
learns merges from the supplied training sentences only.

In [ ]:
def train_tokenizer(corpus, vocab_size, min_frequency, pretokenizer=Metaspace(), unk_token='[UNK]', normalization='none'):
    """
    corpus: a list of texts (each a string)
    vocab_size: number of distinct BPE tokens
    min_frequency: we only merge token pairs that are frequent enough
    pretokenizer: a simple tokenizer that we use before BPE tokenization
    
        You probably don't need to change the pretokenizer, unless you are working with
        languages that really don't use any form of blank space 
        (then you could, for example, pretokenize at character level)
    """
    tokenizer = Tokenizer(BPE(unk_token=unk_token))
    if normalization == 'lowercase':
        tokenizer.normalizer = tokenizers.normalizers.Lowercase()
    elif normalization != 'none':
        raise ValueError('Choose none or lowercase normalization.')
    tokenizer.pre_tokenizer = pretokenizer
    trainer = BpeTrainer(vocab_size=vocab_size, min_frequency=min_frequency, special_tokens=[unk_token], show_progress=False)
    tokenizer.train_from_iterator(corpus, trainer=trainer)
    return tokenizer

### BPE tokens and retained adjacent pairs

Store variable-length token sequences, tokenize the two roles separately,
and learn the frequency-ranked bigram inventory.

In [ ]:
def _features_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
@dataclass
class TokenizedPairs:
    """Compact ragged token arrays; offsets mark sentence boundaries per role."""
    premise_ids: np.ndarray
    premise_offsets: np.ndarray
    hypothesis_ids: np.ndarray
    hypothesis_offsets: np.ndarray

    def __post_init__(self):
        for ids, offsets in self.roles():
            if ids.ndim != 1 or offsets.ndim != 1 or (not len(offsets)):
                raise ValueError('Token IDs and offsets must be flat arrays with an initial offset.')
            if ids.dtype != np.int32 or offsets.dtype != np.int64:
                raise ValueError('Expected int32 token IDs and int64 sentence offsets.')
            if offsets[0] != 0 or offsets[-1] != len(ids) or np.any(np.diff(offsets) < 0):
                raise ValueError('Invalid sentence offsets.')
        if len(self.premise_offsets) != len(self.hypothesis_offsets):
            raise ValueError('Premise/hypothesis sentence counts differ.')

    def __len__(self) -> int:
        return len(self.premise_offsets) - 1

    def roles(self):
        return ((self.premise_ids, self.premise_offsets), (self.hypothesis_ids, self.hypothesis_offsets))

    def pair(self, row: int) -> tuple[np.ndarray, np.ndarray]:
        if not 0 <= row < len(self):
            raise IndexError(row)
        return (self.premise_ids[self.premise_offsets[row]:self.premise_offsets[row + 1]], self.hypothesis_ids[self.hypothesis_offsets[row]:self.hypothesis_offsets[row + 1]])

    def save(self, path: str | Path) -> None:
        np.savez_compressed(path, premise_ids=self.premise_ids, premise_offsets=self.premise_offsets, hypothesis_ids=self.hypothesis_ids, hypothesis_offsets=self.hypothesis_offsets)

    @classmethod
    def load(cls, path: str | Path) -> 'TokenizedPairs':
        with np.load(path, allow_pickle=False) as data:
            return cls(**{name: data[name] for name in cls.__dataclass_fields__})

In [ ]:
def tokenize_pairs(tokenizer: Tokenizer, premises: Sequence[str], hypotheses: Sequence[str], *, batch_size: int=2048) -> TokenizedPairs:
    """Encode roles independently in bounded batches, without padding/truncation.

    Blank inference sentences map to an empty sequence/zero role block. Nonblank
    input is passed unchanged; whitespace handling belongs to Metaspace.
    """
    if len(premises) != len(hypotheses) or batch_size < 1:
        raise ValueError('Equal role lengths and a positive batch size are required.')
    buffers = []
    for sentences in (premises, hypotheses):
        ids, offsets = (array('i'), array('q', [0]))
        for start in range(0, len(sentences), batch_size):
            texts = list(sentences[start:start + batch_size])
            if any((not isinstance(text, str) for text in texts)):
                raise TypeError('Each premise/hypothesis must be a string.')
            texts = [text if text.strip() else '' for text in texts]
            for encoding in tokenizer.encode_batch(texts, add_special_tokens=False):
                ids.extend(encoding.ids)
                offsets.append(len(ids))
        buffers.extend((np.asarray(ids, dtype=np.int32), np.asarray(offsets, dtype=np.int64)))
    return TokenizedPairs(*buffers)

In [ ]:
def learn_bigrams(tokens: TokenizedPairs, min_frequency: int, max_features: int) -> tuple[list[tuple[int, int]], list[int]]:
    """Rank training-only within-sentence BPE bigrams by occurrence counts.

    Ties use ascending token-ID pairs. Count both roles in one shared inventory,
    never creating a bigram across sentences, rows, or role boundaries.
    """
    if min_frequency < 1 or max_features < 0:
        raise ValueError('Bigram minimum frequency must be positive; cap must be nonnegative.')
    counts = Counter()
    for ids, offsets in tokens.roles():
        for start, end in zip(offsets[:-1], offsets[1:]):
            sentence = ids[start:end]
            counts.update(zip(sentence[:-1], sentence[1:]))
    selected = sorted(((pair, count) for pair, count in counts.items() if count >= min_frequency), key=lambda item: (-item[1], item[0]))[:max_features]
    return ([tuple(map(int, pair)) for pair, _ in selected], [int(count) for _, count in selected])

### The feature encoder

Here are the actual count/indicator construction, column layouts and fixed
inference transformations—not a separate demonstration implementation.

In [ ]:
class PairFeatureEncoder:
    """A fitted tokenizer/inventory; transformation never learns new coordinates.

    Model inputs are [P unigrams; H unigrams] for indicators/unigram counts,
    or [P bigrams; H bigrams] for bigram-only counts. An internal combined
    count cache holds both inventories so all three inputs share tokenization.
    Values are float32 CSR counts or indicators, without dense intermediate rows.
    """

    def __init__(self, tokenizer: Tokenizer, bigrams: list[tuple[int, int]], bigram_counts: list[int], unk_token: str='[UNK]'):
        self.tokenizer = tokenizer
        self.vocab_size = tokenizer.get_vocab_size()
        self.bigrams = tuple((tuple(pair) for pair in bigrams))
        self.bigram_counts = tuple(bigram_counts)
        self.unk_token = unk_token
        if len(self.bigrams) != len(self.bigram_counts) or len(set(self.bigrams)) != len(self.bigrams):
            raise ValueError('Bigram inventory/counts must align and contain no duplicates.')
        if any((len(pair) != 2 or any((not 0 <= token < self.vocab_size for token in pair)) for pair in self.bigrams)):
            raise ValueError('Bigram IDs must refer to the fitted vocabulary.')
        if any((count < 1 for count in self.bigram_counts)) or tokenizer.token_to_id(unk_token) is None:
            raise ValueError('Expected positive training bigram counts and an unknown token.')
        self._bigram_ids = {pair: self.vocab_size + i for i, pair in enumerate(self.bigrams)}

    def num_features(self, representation: str) -> int:
        if representation not in REPRESENTATIONS:
            raise ValueError(f'Unknown representation: {representation}')
        return 2 * (len(self.bigrams) if representation == 'bigram_counts' else self.vocab_size)

    def count_matrix(self, tokens: TokenizedPairs) -> sparse.csr_matrix:
        """Build the internal cache [P unigrams; P bigrams; H unigrams; H bigrams].

        Select a model input with representations(); this combined cache is
        never itself a representation in the current comparison.
        """
        for ids, _ in tokens.roles():
            if len(ids) and (ids.min() < 0 or ids.max() >= self.vocab_size):
                raise ValueError("Token IDs are outside this encoder's fitted vocabulary.")
        width = self.vocab_size + len(self.bigrams)
        values, indices, indptr = (array('f'), array('i'), array('q', [0]))
        for row in range(len(tokens)):
            coordinates = {}
            for offset, sentence in zip((0, width), tokens.pair(row)):
                coordinates.update(((offset + int(token), count) for token, count in Counter(sentence).items()))
                for pair, count in Counter(zip(sentence[:-1], sentence[1:])).items():
                    feature = self._bigram_ids.get(pair)
                    if feature is not None:
                        coordinates[offset + feature] = count
            for coordinate, count in sorted(coordinates.items()):
                indices.append(coordinate)
                values.append(count)
            indptr.append(len(indices))
        return sparse.csr_matrix((np.asarray(values, dtype=np.float32), np.asarray(indices, dtype=np.int32), np.asarray(indptr, dtype=np.int64)), shape=(len(tokens), 2 * width))

    def representations(self, counts: sparse.csr_matrix) -> dict[str, sparse.csr_matrix]:
        if counts.shape[1] != 2 * (self.vocab_size + len(self.bigrams)):
            raise ValueError('Count matrix has the wrong feature dimension.')
        width = counts.shape[1] // 2
        unigrams = sparse.hstack((counts[:, :self.vocab_size], counts[:, width:width + self.vocab_size]), format='csr', dtype=np.float32)
        indicators = unigrams.copy()
        indicators.data.fill(1)
        bigrams = sparse.hstack((counts[:, self.vocab_size:width], counts[:, width + self.vocab_size:]), format='csr', dtype=np.float32)
        return {'indicators': indicators, 'unigram_counts': unigrams, 'bigram_counts': bigrams}

    def transform(self, premises: Sequence[str], hypotheses: Sequence[str], representation: str) -> sparse.csr_matrix:
        self.num_features(representation)
        tokens = tokenize_pairs(self.tokenizer, premises, hypotheses)
        return self.representations(self.count_matrix(tokens))[representation]

    def feature_name(self, coordinate: int, representation: str) -> str:
        dimension = self.num_features(representation)
        if not 0 <= coordinate < dimension:
            raise IndexError(coordinate)
        role, local = divmod(coordinate, dimension // 2)
        if representation == 'bigram_counts':
            name = repr(tuple((self.tokenizer.id_to_token(token) for token in self.bigrams[local])))
        else:
            name = repr(self.tokenizer.id_to_token(local))
        return f'{('P' if role == 0 else 'H')}: {name}'

    def save(self, directory: str | Path) -> None:
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        self.tokenizer.save(str(directory / 'tokenizer.json'))
        (directory / 'encoder.json').write_text(json.dumps({'feature_version': FEATURE_VERSION, 'unk_token': self.unk_token, 'bigrams': self.bigrams, 'bigram_counts': self.bigram_counts, 'bigram_tie_break': 'frequency_descending_then_token_id_pair_ascending', 'count_cache_layout': 'P_unigrams,P_bigrams,H_unigrams,H_bigrams', 'representation_layouts': {'indicators': 'P_unigrams,H_unigrams', 'unigram_counts': 'P_unigrams,H_unigrams', 'bigram_counts': 'P_bigrams,H_bigrams'}}, indent=2) + '\n', encoding='utf-8')

    @classmethod
    def load(cls, directory: str | Path) -> 'PairFeatureEncoder':
        directory = Path(directory)
        metadata = json.loads((directory / 'encoder.json').read_text(encoding='utf-8'))
        if metadata['feature_version'] != FEATURE_VERSION:
            raise ValueError('Unsupported saved feature version.')
        return cls(Tokenizer.from_file(str(directory / 'tokenizer.json')), metadata['bigrams'], metadata['bigram_counts'], metadata['unk_token'])

In [ ]:
def fit_pair_features(premises: Sequence[str], hypotheses: Sequence[str], tokenizer_config: dict[str, Any], bigram_config: dict[str, Any]) -> tuple[PairFeatureEncoder, TokenizedPairs]:
    """Fit only on the supplied training sentences; also return their encoding."""
    if len(premises) != len(hypotheses) or not len(premises):
        raise ValueError('Fitting requires nonempty, equally sized training roles.')
    if any((not isinstance(text, str) or not text.strip() for text in chain(premises, hypotheses))):
        raise ValueError('Fitting requires cleaned, nonblank training sentences.')
    if tokenizer_config['algorithm'] != 'BPE' or tokenizer_config['pretokenizer'] != 'Metaspace':
        raise ValueError("This implementation uses the tutorial's BPE/Metaspace helper.")
    if tokenizer_config.get('normalization', 'none') not in ('none', 'lowercase'):
        raise ValueError('Choose none or lowercase normalization; retain punctuation.')
    if tokenizer_config['vocab_size'] < 1 or tokenizer_config['min_frequency'] < 1:
        raise ValueError('BPE vocabulary target and merge minimum frequency must be positive.')
    unk = tokenizer_config.get('unk_token', '[UNK]')
    tokenizer = train_tokenizer(chain.from_iterable(zip(premises, hypotheses)), vocab_size=tokenizer_config['vocab_size'], min_frequency=tokenizer_config['min_frequency'], pretokenizer=Metaspace(replacement='▁', prepend_scheme='always', split=True), unk_token=unk, normalization=tokenizer_config.get('normalization', 'none'))
    tokens = tokenize_pairs(tokenizer, premises, hypotheses)
    bigrams, counts = learn_bigrams(tokens, **bigram_config)
    return (PairFeatureEncoder(tokenizer, bigrams, counts, unk), tokens)

In [ ]:
def matrix_summary(matrices: dict[str, dict[str, sparse.csr_matrix]]) -> pd.DataFrame:
    rows = []
    for partition, representations in matrices.items():
        for name, matrix in representations.items():
            rows.append({'partition': partition, 'representation': name, 'rows': matrix.shape[0], 'columns': matrix.shape[1], 'nonzeros': matrix.nnz, 'dtype': str(matrix.dtype), 'csr_bytes': matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes, 'estimated_dense_bytes': matrix.shape[0] * matrix.shape[1] * matrix.dtype.itemsize})
    return pd.DataFrame(rows)

In [ ]:
def prepare_fold_features(config_path: str | Path=DEFAULT_CONFIG, *, seed: int=0, fold: int=0, verbose: bool=False) -> dict[str, Any]:
    """Fit/cache one outer fold, or verify/reuse its existing fitted artifacts.

    Combined unigram/bigram counts are cached internally once; indicators,
    unigram-only counts and bigram-only counts are selected for model inputs.
    Official validation/test sentences are never tokenized here.
    """
    started = perf_counter()
    workspace = prepare_workspace(config_path)
    config, paths = (workspace['config'], workspace['paths'])
    protocol = prepare_evaluation_protocol(config_path)
    if seed not in config['seeds']['cv_splits'] or not 0 <= fold < config['cross_validation']['num_folds']:
        raise ValueError('Choose a seed/fold from the saved evaluation protocol.')
    if config['representations'] != list(REPRESENTATIONS):
        raise ValueError(f'Expected the three planned representations: {REPRESENTATIONS}')
    rows = protocol['assignments'][ROW_COLUMNS].copy()
    rows['partition'] = np.where(protocol['assignments'][f'fold_seed_{seed}'].eq(fold), 'heldout', 'train')
    partitions = {name: rows.loc[rows['partition'].eq(name)].reset_index(drop=True) for name in ('train', 'heldout')}
    specification = {'feature_version': FEATURE_VERSION, 'protocol_id': protocol['manifest']['protocol_id'], 'seed': seed, 'fold': fold, 'tokenizer': config['tokenizer'], 'bigram_features': config['bigram_features'], 'representations': list(REPRESENTATIONS), 'tokenizers_version': tokenizers.__version__, 'scipy_version': scipy.__version__, 'implementation_sha256': notebook_source_sha256(), 'tutorial_helper_sha256': notebook_source_sha256()}
    feature_id = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:16]
    directory = paths['features'] / specification['protocol_id'] / f'seed_{seed}_fold_{fold}' / feature_id
    manifest_path = directory / 'manifest.json'
    reused = directory.exists()
    if reused:
        if not manifest_path.is_file():
            raise ValueError(f'Incomplete feature cache: {directory}. Existing files were not overwritten.')
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['specification'] != specification:
            raise ValueError('Saved features do not match this fold/configuration.')
        for name, digest in manifest['artifact_sha256'].items():
            path = directory / name
            if not path.is_file() or _features_sha256(path) != digest:
                raise ValueError(f'Saved feature artifact changed or is missing: {path}')
        if not pd.read_parquet(directory / 'rows.parquet').equals(rows):
            raise ValueError('Cached feature rows do not match the saved fold order.')
        encoder = PairFeatureEncoder.load(directory)
        tokenized = {name: TokenizedPairs.load(directory / f'{name}_tokens.npz') for name in partitions}
        counts = {name: sparse.load_npz(directory / f'{name}_counts.npz').tocsr() for name in partitions}
        if verbose:
            print(f'Reused features for seed {seed}, fold {fold}.', flush=True)
    else:
        if verbose:
            print(f'Fitting BPE/inventory on {len(partitions['train']):,} training pairs (seed {seed}, fold {fold})...', flush=True)
        source = load_from_disk(protocol['manifest']['prepared_dataset'])['train']
        frame = source.select(partitions['train']['clean_index'].tolist()).select_columns(['premise', 'hypothesis']).to_pandas()
        fit_started = perf_counter()
        encoder, train_tokens = fit_pair_features(frame['premise'].tolist(), frame['hypothesis'].tolist(), config['tokenizer'], config['bigram_features'])
        fitted_at = perf_counter()
        frame = source.select(partitions['heldout']['clean_index'].tolist()).select_columns(['premise', 'hypothesis']).to_pandas()
        tokenized = {'train': train_tokens, 'heldout': tokenize_pairs(encoder.tokenizer, frame['premise'].tolist(), frame['hypothesis'].tolist())}
        del frame
        if verbose:
            print('Building float32 CSR counts from the shared tokenization...', flush=True)
        counts = {name: encoder.count_matrix(tokens) for name, tokens in tokenized.items()}
        directory.mkdir(parents=True)
        encoder.save(directory)
        rows.to_parquet(directory / 'rows.parquet', index=False)
        for name in partitions:
            tokenized[name].save(directory / f'{name}_tokens.npz')
            sparse.save_npz(directory / f'{name}_counts.npz', counts[name])
        manifest = {'stage': 'sentence_pair_features', 'feature_id': feature_id, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'specification': specification, 'creation_config_sha256': workspace['manifest']['config_sha256'], 'repository': workspace['manifest']['repository'], 'actual_vocab_size': encoder.vocab_size, 'bigram_types': len(encoder.bigrams), 'training_rows': len(partitions['train']), 'heldout_rows': len(partitions['heldout']), 'normalization': 'No case folding, punctuation removal, stemming, or negation rewrite; Metaspace handles whitespace.', 'fit_corpus': 'Training premise then hypothesis per row, including repeated text; both roles share one vocabulary/inventory.', 'bigram_selection': 'Within-sentence occurrence counts in both training roles; frequency descending, token-ID-pair ascending.', 'feature_layout': 'P_unigrams,P_bigrams,H_unigrams,H_bigrams; drop bigram blocks for unigram spaces.', 'fit_seconds': fitted_at - fit_started, 'preparation_seconds': perf_counter() - started, 'test_usage': 'No official validation/test text is tokenized or used to fit these features.', 'classifier_trained': False, 'artifact_sha256': {path.name: _features_sha256(path) for path in sorted(directory.iterdir()) if path.is_file()}}
        manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    for name in partitions:
        if len(tokenized[name]) != len(partitions[name]) or counts[name].shape != (len(partitions[name]), 2 * (encoder.vocab_size + len(encoder.bigrams))) or counts[name].dtype != np.float32:
            raise ValueError(f'{name}: tokenization/counts do not align with this fold.')
    matrices = {name: encoder.representations(matrix) for name, matrix in counts.items()}
    return {'encoder': encoder, 'tokens': tokenized, 'matrices': matrices, 'rows': partitions, 'summary': matrix_summary(matrices), 'manifest': manifest, 'manifest_path': manifest_path, 'feature_dir': directory, 'reused_features': reused}

In [ ]:
fold_features = prepare_fold_features(FROZEN_CONFIG, seed=0, fold=0)
pair_encoder, feature_matrices = fold_features["encoder"], fold_features["matrices"]
print("BPE vocabulary:", pair_encoder.vocab_size)
print("Retained bigrams:", len(pair_encoder.bigrams))
print("Training / held-out pairs:", *(len(fold_features["rows"][p]) for p in ("train", "heldout")))

Let's inspect the vocabulary. Token IDs are column addresses, not quantities:
ID 400 does not mean more than ID 20. The `▁` marker represents a word boundary.

In [ ]:
vocabulary = pair_encoder.tokenizer.get_vocab()
display(pd.DataFrame(sorted(vocabulary.items(), key=lambda item: item[1])[:20],
                     columns=["BPE token", "ID"]))

### Tokenizing a sentence pair

We segment premise and hypothesis separately with the same fitted tokenizer.
The following are actual training examples from the chosen fold.

In [ ]:
training_examples = dataset["train"].select(
    fold_features["rows"]["train"]["clean_index"].head(2).tolist()
).to_pandas()
segmented = []
for row in training_examples.itertuples():
    for role in ("premise", "hypothesis"):
        text = getattr(row, role)
        segmented.append({"role": role, "text": text,
                          "BPE tokens": pair_encoder.tokenizer.encode(text).tokens})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(segmented))

### Preparing the model input

A vocabulary is not yet a sentence representation. We turn occurrences into a
fixed-length vector, keeping separate blocks for the two sentence roles:

| Representation | Values | Input layout |
| --- | --- | --- |
| Indicators | 1 if a token occurs, otherwise 0 | `[premise unigrams; hypothesis unigrams]` |
| Unigram counts | Number of occurrences of each token | `[premise unigrams; hypothesis unigrams]` |
| Bigram counts | Number of occurrences of each retained adjacent token pair | `[premise bigrams; hypothesis bigrams]` |

For the baseline, `V = B = 5000`, so all inputs have 10,000 coordinates. The
internal cache contains both inventories; the model receives only the blocks
for its selected representation. Pure bigrams have **no unigram fallback**.

In [ ]:
def show_nonzero_features(matrix, representation):
    """Show the active coordinates in one sparse example."""
    row = matrix.getrow(0)
    return pd.DataFrame([
        {"coordinate": int(i), "feature": pair_encoder.feature_name(int(i), representation),
         "value": float(value)}
        for i, value in zip(row.indices, row.data)
    ])

premises, hypotheses = ["the dog dog"], ["the man"]
counts = pair_encoder.transform(premises, hypotheses, "unigram_counts")
indicators = pair_encoder.transform(premises, hypotheses, "indicators")
comparison = show_nonzero_features(counts, "unigram_counts").rename(columns={"value": "count"})
comparison["indicator"] = [float(indicators[0, i]) for i in comparison["coordinate"]]
display(comparison)
assert counts.data.max() >= 2 and np.all(indicators.data == 1)

The repeated `dog` increases counts, while presence remains one. Now inspect
bigram coordinates. They refer to adjacent **BPE tokens**, which need not be two
whole words. Pairs outside the retained training inventory contribute nothing.

In [ ]:
bigram_input = pair_encoder.transform(premises, hypotheses, "bigram_counts")
display(show_nonzero_features(bigram_input, "bigram_counts"))
print("Sparse input shape:", bigram_input.shape)

### What happens when word order changes?

The next examples keep the same word counts but change who does what. Unigrams
cannot distinguish them. Retained bigrams can preserve some local order;
discarded pairs and long-distance relations remain limitations.

In [ ]:
order_candidates = [
    ("the dog is chasing the man", "the man is chasing the dog"),
    ("the man is following the woman", "the woman is following the man"),
    ("the boy is watching the girl", "the girl is watching the boy"),
]
order_checks = []
for first, second in order_candidates:
    spaces = {name: pair_encoder.transform([first, second], ["an event", "an event"], name) for name in REPRESENTATIONS}
    order_checks.append({
        "first": first, "second": second,
        "indicators_equal": (spaces["indicators"][0] != spaces["indicators"][1]).nnz == 0,
        "unigram_counts_equal": (spaces["unigram_counts"][0] != spaces["unigram_counts"][1]).nnz == 0,
        "changed_bigram_coordinates": (spaces["bigram_counts"][0] != spaces["bigram_counts"][1]).nnz,
    })
order_checks = pd.DataFrame(order_checks)
display(order_checks)
assert order_checks["unigram_counts_equal"].all()
assert order_checks["indicators_equal"].all()

distinguished = order_checks.loc[order_checks["changed_bigram_coordinates"] > 0]
if len(distinguished):
    chosen = distinguished.iloc[0]
    bigram_vectors = pair_encoder.transform([chosen["first"], chosen["second"]], ["an event", "an event"], "bigram_counts")
    changed = (bigram_vectors[0] - bigram_vectors[1]).indices
    display(pd.DataFrame([
        {"feature": pair_encoder.feature_name(int(coordinate), "bigram_counts"),
         "first_count": float(bigram_vectors[0, coordinate]), "second_count": float(bigram_vectors[1, coordinate])}
        for coordinate in changed
    ]))
else:
    print("The current capped inventory drops the distinguishing bigrams in all three examples.")

### Sparse storage

Most coordinates are zero, so we store CSR matrices rather than dense arrays.
Rows are examples, columns are role-specific features, and labels are kept in
the same row order. Matrix size and memory cost change when we tune vocabulary
or bigram inventory size.

In [ ]:
summary = fold_features["summary"].copy()
summary["csr_MiB"] = (summary["csr_bytes"] / 2**20).round(2)
summary["dense_GiB_if_allocated"] = (summary["estimated_dense_bytes"] / 2**30).round(2)
display(summary[[
    "partition", "representation", "rows", "columns", "nonzeros", "dtype",
    "csr_MiB", "dense_GiB_if_allocated",
]])

for partition, spaces in feature_matrices.items():
    assert set(spaces) == set(REPRESENTATIONS)
    assert spaces["indicators"].shape[1] == 2 * pair_encoder.vocab_size
    assert spaces["bigram_counts"].shape[1] == 2 * len(pair_encoder.bigrams)
    assert (spaces["indicators"].data == 1).all()
print("Dimensions and binary indicator values verified.")

## 5. Classifiers

The input is a sparse vector; the output is a probability distribution over
`[entailment, neutral, contradiction]`. Prediction uses the largest probability.
We begin with a linear model, then introduce nonlinear hidden layers.

### Logistic regression

For input $x$, logistic regression computes $z=xW+b$ and applies softmax.
With concatenated bags, the premise and hypothesis contribute additively to
each class score. There is no explicit token alignment feature.

Load a saved baseline model to inspect its weights and predict one example.
The bundle carries its own tokenizer and feature mapping, keeping coordinates
consistent with its weights.

In [ ]:
MODEL_VERSION = "v1"
CLASSIFIERS = ("logistic", "mlp_one_hidden", "mlp_two_hidden")
CLASS_IDS = np.arange(len(LABEL_NAMES))

### Probability and sparse-input utilities

Validate inputs and class order, convert sparse minibatches and compute the
same accuracy/NLL/F1/confusion outputs used by training and saved reports.

In [ ]:
def _models_matrix(X) -> sparse.csr_matrix:
    if not sparse.isspmatrix_csr(X):
        raise TypeError('Expected a SciPy CSR matrix; full feature matrices must stay sparse.')
    if not X.shape[1] or not np.isfinite(X.data).all():
        raise ValueError('Expected a positive input dimension and finite feature values.')
    return X.astype(np.float32, copy=False)

In [ ]:
def _models_labels(y, rows: int, *, require_all: bool=False) -> np.ndarray:
    values = np.asarray(y)
    if values.ndim != 1 or len(values) != rows or (not np.issubdtype(values.dtype, np.integer)):
        raise ValueError('Expected one integer gold label per row.')
    if not np.isin(values, CLASS_IDS).all() or (require_all and set(values.tolist()) != set(CLASS_IDS.tolist())):
        raise ValueError('Training needs classes 0, 1, 2; evaluation labels must lie in that set.')
    return values.astype(np.int64, copy=False)

In [ ]:
def validate_probabilities(probabilities, rows: int | None=None) -> np.ndarray:
    values = np.asarray(probabilities, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(LABEL_NAMES) or (rows is not None and len(values) != rows):
        raise ValueError('Expected probability columns in label order [0, 1, 2].')
    if not np.isfinite(values).all() or np.any(values < 0) or np.any(values > 1):
        raise ValueError('Probabilities must be finite and lie in [0, 1].')
    if not np.allclose(values.sum(axis=1), 1.0, rtol=1e-06, atol=1e-07):
        raise ValueError('Each probability row must sum to one.')
    return values

In [ ]:
def score_probabilities(y, probabilities) -> dict[str, Any]:
    """Apply Step 3's fixed class order, argmax ties, and reporting conventions."""
    probabilities = validate_probabilities(probabilities)
    if not len(probabilities):
        raise ValueError('Cannot score an empty evaluation set.')
    labels = _models_labels(y, len(probabilities))
    predicted = probabilities.argmax(axis=1)
    epsilon = np.finfo(np.float64).eps
    nll = -np.log(np.clip(probabilities[np.arange(len(labels)), labels], epsilon, 1 - epsilon)).mean()
    return {'accuracy': float(accuracy_score(labels, predicted)), 'nll': float(nll), 'macro_f1': float(f1_score(labels, predicted, labels=CLASS_IDS, average='macro', zero_division=0)), 'per_class': classification_report(labels, predicted, labels=CLASS_IDS, target_names=list(LABEL_NAMES), zero_division=0, output_dict=True), 'confusion_matrix': confusion_matrix(labels, predicted, labels=CLASS_IDS).tolist(), 'num_examples': len(labels)}

In [ ]:
def select_input_features(X: sparse.csr_matrix, input_mode: str='pair') -> sparse.csr_matrix:
    X = _models_matrix(X)
    if input_mode == 'pair':
        return X
    if input_mode == 'hypothesis_only' and X.shape[1] % 2 == 0:
        return X[:, X.shape[1] // 2:]
    raise ValueError('Expected pair or hypothesis_only mode with equal sentence-role blocks.')

In [ ]:
def scipy_batch_to_torch(X: sparse.csr_matrix, device: str='cpu') -> torch.Tensor:
    """Convert one sparse batch to COO; the feature matrix is never densified."""
    coo = X.tocoo(copy=False)
    indices = torch.from_numpy(np.vstack((coo.row, coo.col)).astype(np.int64, copy=False))
    values = torch.from_numpy(np.asarray(coo.data, dtype=np.float32))
    return torch.sparse_coo_tensor(indices, values, size=coo.shape, check_invariants=True).coalesce().to(device)

### Classifier classes

The sparse first layer and one-/two-hidden-layer definitions are executable
notebook classes. Loaded state dictionaries instantiate these classes.

In [ ]:
class SparseMLP(nn.Module):
    """One-hidden-layer MLP that accepts sparse input tensors.

    Inputs: feature vectors of size `input_dim` (e.g. bag-of-words).
    Process: sparse matmul into a dense hidden layer, ReLU + dropout, linear (num_classes) and log-softmax outputs
        log-softmax, rather than softmax, for numerical stability
    Output: logprobs over `num_classes`.
    """

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.2):
        """
        input_dim: size of the input feature space (e.g. vocabulary size)
        hidden_dim: width of the hidden layer
        num_classes: number of output labels
        dropout: fraction of hidden units zeroed during training (regularisation)
        """
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, hidden_dim))
        self.bias = nn.Parameter(torch.zeros(hidden_dim))
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Sequential(nn.Linear(hidden_dim, num_classes), nn.LogSoftmax(dim=-1))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        """Map a sparse batch [batch, input_dim] to class log-probabilities [batch, num_classes]."""
        x = torch.sparse.mm(x, self.weight)
        x = x + self.bias
        x = self.activation(x)
        x = self.dropout(x)
        return self.output(x)

### Model construction and prediction

Wrap each model with prediction and metadata interfaces.

In [ ]:
class PairMLP(SparseMLP):
    """Tutorial sparse first layer, with one or two ReLU/dropout hidden stages."""

    def __init__(self, input_dim: int, hidden_dims: Sequence[int], dropout: float):
        if input_dim < 1 or len(hidden_dims) not in (1, 2) or any((width < 1 for width in hidden_dims)):
            raise ValueError('Expected a positive input dimension and one/two positive hidden widths.')
        if not 0 <= dropout < 1:
            raise ValueError('Dropout must lie in [0, 1).')
        super().__init__(input_dim, hidden_dims[0], len(LABEL_NAMES), dropout)
        if len(hidden_dims) == 2:
            self.output = nn.Sequential(nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dims[1], len(LABEL_NAMES)), nn.LogSoftmax(dim=-1))

In [ ]:
def build_neural_model(kind: str, input_dim: int, settings: dict[str, Any]) -> PairMLP:
    if kind not in CLASSIFIERS[1:]:
        raise ValueError(f'Unknown neural architecture: {kind}')
    dimensions = settings['hidden_dims'][kind]
    if len(dimensions) != (1 if kind == 'mlp_one_hidden' else 2):
        raise ValueError('Architecture name and hidden-layer count disagree.')
    return PairMLP(input_dim, dimensions, settings['dropout'])

In [ ]:
@dataclass
class FittedClassifier:
    kind: str
    estimator: Any
    input_dim: int
    settings: dict[str, Any]
    history: list[dict[str, Any]]
    fit_info: dict[str, Any]

    @property
    def parameter_count(self) -> int:
        if self.kind == 'logistic':
            return int(self.estimator.coef_.size + self.estimator.intercept_.size)
        if self.kind == 'class_prior':
            return len(self.estimator)
        return sum((parameter.numel() for parameter in self.estimator.parameters()))

    def predict_log_proba(self, X: sparse.csr_matrix, *, batch_size: int=1024) -> np.ndarray:
        X = _models_matrix(X)
        if X.shape[1] != self.input_dim or batch_size < 1:
            raise ValueError('Input dimension must match training, and batch size must be positive.')
        if not X.shape[0]:
            return np.empty((0, len(LABEL_NAMES)), dtype=np.float64)
        if self.kind == 'logistic':
            if not np.array_equal(self.estimator.classes_, CLASS_IDS):
                raise ValueError('Classifier class order is not [0, 1, 2].')
            return np.asarray(self.estimator.predict_log_proba(X), dtype=np.float64)
        if self.kind == 'class_prior':
            with np.errstate(divide='ignore'):
                return np.tile(np.log(self.estimator), (X.shape[0], 1))
        model = self.estimator
        model.eval()
        device = str(model.weight.device)
        batches = []
        with torch.no_grad():
            for start in range(0, X.shape[0], batch_size):
                output = model(scipy_batch_to_torch(X[start:start + batch_size], device))
                batches.append(output.cpu().numpy().astype(np.float64))
        return np.concatenate(batches)

    def predict_proba(self, X: sparse.csr_matrix, *, batch_size: int=1024) -> np.ndarray:
        return validate_probabilities(np.exp(self.predict_log_proba(X, batch_size=batch_size)), X.shape[0])

    def predict(self, X: sparse.csr_matrix) -> np.ndarray:
        return self.predict_proba(X).argmax(axis=1)

### Fitting logistic regression and the MLPs

This is the full CSR minibatch training implementation, including shuffling,
NLL, gradient clearing, backward passes, optimizer updates and training history.
The historical fixed-epoch interface remains the baseline; validation-driven
candidate selection is still the new work described in section 6.

In [ ]:
def fit_classifier(kind: str, X: sparse.csr_matrix, y, config: dict[str, Any], *, seed: int | None=None, num_epochs: int | None=None, epoch_observer: Callable[[FittedClassifier], None] | None=None) -> FittedClassifier:
    """Fit with a temporary configured CPU thread count; restore it afterward.

    An observer may log independent pilot-validation curves. Its return value
    cannot alter training length or select a checkpoint. Reported CV must not
    pass outer held-out scores through an observer for model selection.
    """
    original_threads = torch.get_num_threads()
    threads = config['neural_training'].get('num_threads') if kind in CLASSIFIERS[1:] else None
    if threads is not None and (not isinstance(threads, int) or threads < 1):
        raise ValueError('num_threads must be a positive integer.')
    if threads is not None:
        torch.set_num_threads(threads)
    try:
        return _models_fit_classifier(kind, X, y, config, seed=seed, num_epochs=num_epochs, epoch_observer=epoch_observer)
    finally:
        if threads is not None:
            torch.set_num_threads(original_threads)

In [ ]:
def _models_fit_classifier(kind: str, X: sparse.csr_matrix, y, config: dict[str, Any], *, seed: int | None=None, num_epochs: int | None=None, epoch_observer: Callable[[FittedClassifier], None] | None=None) -> FittedClassifier:
    """Fit from CSR using training labels only; save the final fixed-epoch state.

    ``num_epochs`` can explicitly override the unset pilot value for software
    checks. Reported runs must supply the frozen pilot-selected value. There is
    no outer validation argument and no checkpoint selection on held-out scores.
    """
    X = _models_matrix(X)
    if not X.shape[0] or kind not in (*CLASSIFIERS, 'class_prior'):
        raise ValueError('Choose a supported classifier and nonempty training matrix.')
    labels = _models_labels(y, X.shape[0], require_all=kind != 'class_prior')
    seed = config['seeds']['model'] if seed is None else seed
    set_random_seeds(seed)
    started = perf_counter()
    common = {'seed': seed, 'training_rows': X.shape[0], 'class_order': CLASS_IDS.tolist()}
    if kind == 'class_prior':
        counts = np.bincount(labels, minlength=len(LABEL_NAMES))
        return FittedClassifier(kind, counts / counts.sum(), X.shape[1], {}, [], {**common, 'class_counts': counts.tolist(), 'fit_seconds': perf_counter() - started})
    if kind == 'logistic':
        settings = dict(config['logistic_training'])
        if settings['solver'] != 'saga':
            raise ValueError("Use the tutorial's sparse-friendly SAGA solver for this comparison.")
        model = LogisticRegression(**settings, l1_ratio=0.0, fit_intercept=True, random_state=seed)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(X, labels)
        convergence = [str(warning.message) for warning in caught if issubclass(warning.category, ConvergenceWarning)]
        for warning in caught:
            if not issubclass(warning.category, ConvergenceWarning):
                warnings.warn(str(warning.message), warning.category)
        return FittedClassifier(kind, model, X.shape[1], settings, [], {**common, 'regularization': 'L2 (l1_ratio=0); C is inverse strength', 'converged': not convergence, 'convergence_warnings': convergence, 'iterations': model.n_iter_.tolist(), 'fit_seconds': perf_counter() - started})
    settings = dict(config['neural_training'])
    epochs = settings['num_epochs'] if num_epochs is None else num_epochs
    if not isinstance(epochs, int) or isinstance(epochs, bool) or epochs < 1:
        raise ValueError('Set positive fixed num_epochs after the pilot, or explicitly override for a software check.')
    if settings['optimizer'] != 'AdamW' or settings['batch_size'] < 1:
        raise ValueError('Expected AdamW and a positive sparse batch size.')
    settings['num_epochs'] = epochs
    device = config['device']
    model = build_neural_model(kind, X.shape[1], settings).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['learning_rate'], weight_decay=settings['weight_decay'])
    criterion = nn.NLLLoss()
    rng = np.random.default_rng(seed)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        order = rng.permutation(X.shape[0])
        total_loss, total_correct = (0.0, 0)
        epoch_started = perf_counter()
        for start in range(0, len(order), settings['batch_size']):
            indices = order[start:start + settings['batch_size']]
            features = scipy_batch_to_torch(X[indices], device)
            target = torch.as_tensor(labels[indices], dtype=torch.long, device=device)
            optimizer.zero_grad(set_to_none=True)
            log_probabilities = model(features)
            loss = criterion(log_probabilities, target)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite neural training loss.')
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(indices)
            total_correct += int((log_probabilities.argmax(dim=1) == target).sum().item())
        history.append({'epoch': epoch, 'train_nll_online': total_loss / X.shape[0], 'train_accuracy_online': total_correct / X.shape[0], 'seconds': perf_counter() - epoch_started})
        if epoch_observer is not None:
            epoch_observer(FittedClassifier(kind, model, X.shape[1], settings, history, {**common, 'epochs_completed': epoch, 'checkpoint_policy': 'epoch_observer_view'}))
    model.eval()
    return FittedClassifier(kind, model, X.shape[1], settings, history, {**common, 'device': device, 'batch_order_seed': seed, 'checkpoint_policy': 'last_fixed_epoch', 'epochs_completed': epochs, 'weight_decay_scope': 'all parameters, including biases', 'fit_seconds': perf_counter() - started, 'torch_cpu_threads': torch.get_num_threads()})

### Saving and loading complete pipelines

Keep weights and their own tokenizer/feature mappings together. New saves
record notebook source provenance.

In [ ]:
@dataclass
class PredictionBundle:
    classifier: FittedClassifier
    encoder: PairFeatureEncoder
    representation: str
    input_mode: str
    metadata: dict[str, Any]

    def predict_proba(self, premises: Sequence[str], hypotheses: Sequence[str]) -> np.ndarray:
        if len(premises) != len(hypotheses):
            raise ValueError('Premise/hypothesis lengths differ.')
        effective_premises = [''] * len(hypotheses) if self.input_mode == 'hypothesis_only' else premises
        matrix = self.encoder.transform(effective_premises, hypotheses, self.representation)
        return self.classifier.predict_proba(select_input_features(matrix, self.input_mode))

    def predict(self, premises: Sequence[str], hypotheses: Sequence[str]) -> np.ndarray:
        return self.predict_proba(premises, hypotheses).argmax(axis=1)

In [ ]:
def save_prediction_bundle(directory: str | Path, classifier: FittedClassifier, encoder: PairFeatureEncoder, representation: str, *, input_mode: str='pair', context: dict[str, Any] | None=None) -> Path:
    expected_dim = encoder.num_features(representation)
    if input_mode == 'hypothesis_only':
        expected_dim //= 2
    elif input_mode != 'pair':
        raise ValueError('Unknown input mode.')
    if classifier.input_dim != expected_dim:
        raise ValueError('Classifier and saved preprocessing dimensions disagree.')
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=False)
    encoder.save(directory / 'preprocessing')
    if classifier.kind == 'logistic':
        joblib.dump(classifier.estimator, directory / 'model.joblib')
    elif classifier.kind == 'class_prior':
        np.save(directory / 'prior.npy', classifier.estimator, allow_pickle=False)
    else:
        state = {name: tensor.detach().cpu() for name, tensor in classifier.estimator.state_dict().items()}
        torch.save(state, directory / 'model.pt')
    metadata = {'model_version': MODEL_VERSION, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'kind': classifier.kind, 'representation': representation, 'input_mode': input_mode, 'input_dim': classifier.input_dim, 'label_names': list(LABEL_NAMES), 'class_order': CLASS_IDS.tolist(), 'tie_break': 'smallest_label_id', 'settings': classifier.settings, 'fit_info': classifier.fit_info, 'history': classifier.history, 'parameter_count': classifier.parameter_count, 'context': context or {}, 'torch_version': torch.__version__, 'sklearn_version': sklearn.__version__, 'implementation_sha256': notebook_source_sha256(), 'tutorial_mlp_sha256': notebook_source_sha256(), 'artifact_sha256': {str(path.relative_to(directory)): _features_sha256(path) for path in sorted(directory.rglob('*')) if path.is_file()}}
    path = directory / 'bundle.json'
    path.write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
    return path

In [ ]:
def load_prediction_bundle(directory: str | Path, *, device: str='cpu') -> PredictionBundle:
    directory = Path(directory)
    metadata = json.loads((directory / 'bundle.json').read_text(encoding='utf-8'))
    if metadata['model_version'] != MODEL_VERSION or metadata['label_names'] != list(LABEL_NAMES) or metadata['class_order'] != CLASS_IDS.tolist():
        raise ValueError('Unsupported bundle version or label mapping.')
    if metadata['input_mode'] not in ('pair', 'hypothesis_only'):
        raise ValueError('Unsupported saved input mode.')
    for name, digest in metadata['artifact_sha256'].items():
        path = directory / name
        if not path.is_file() or _features_sha256(path) != digest:
            raise ValueError(f'Saved model/preprocessing artifact changed or is missing: {path}')
    kind = metadata['kind']
    if kind == 'logistic':
        estimator = joblib.load(directory / 'model.joblib')
    elif kind == 'class_prior':
        estimator = np.load(directory / 'prior.npy', allow_pickle=False)
    elif kind in CLASSIFIERS[1:]:
        estimator = build_neural_model(kind, metadata['input_dim'], metadata['settings'])
        estimator.load_state_dict(torch.load(directory / 'model.pt', map_location=device, weights_only=True))
        estimator = estimator.to(device)
        estimator.eval()
    else:
        raise ValueError(f'Unknown saved classifier: {kind}')
    classifier = FittedClassifier(kind, estimator, metadata['input_dim'], metadata['settings'], metadata['history'], metadata['fit_info'])
    encoder = PairFeatureEncoder.load(directory / 'preprocessing')
    expected = encoder.num_features(metadata['representation']) // (2 if metadata['input_mode'] == 'hypothesis_only' else 1)
    if expected != classifier.input_dim:
        raise ValueError('Saved model/preprocessing dimensions disagree.')
    return PredictionBundle(classifier, encoder, metadata['representation'], metadata['input_mode'], metadata)

In [ ]:
def run_model_smoke(config_path: str | Path=DEFAULT_CONFIG) -> dict[str, Any]:
    """Exercise all nine systems + baselines on synthetic, arbitrarily labelled text.

    No SNLI examples/scores or epoch choices are involved. Two epochs merely
    exercise neural training/reload; the experiment's num_epochs stays unset.
    """
    workspace = prepare_workspace(config_path)
    config, paths = (workspace['config'], workspace['paths'])
    destination = paths['checkpoints'] / 'implementation_smoke'
    destination.mkdir(exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix='run_', dir=destination))
    train_p = [f'item {color}' for _ in range(4) for color in ('red', 'green', 'blue')]
    train_h = [f'{color} {color}' for _ in range(4) for color in ('red', 'green', 'blue')]
    train_y = np.tile(CLASS_IDS, 4)
    eval_p, eval_h = (['item red', 'item green', 'item blue'], ['red', 'green', 'blue'])
    encoder, tokens = fit_pair_features(train_p, train_h, config['tokenizer'], config['bigram_features'])
    matrices = encoder.representations(encoder.count_matrix(tokens))
    checks, bundles = ([], {})
    original_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        systems = [(representation, kind, 'pair') for representation in REPRESENTATIONS for kind in CLASSIFIERS]
        diagnostic = config['diagnostics']['hypothesis_only']
        systems += [('unigram_counts', 'class_prior', 'pair'), (diagnostic['representation'], diagnostic['classifier'], diagnostic['input_mode'])]
        for representation, kind, mode in systems:
            X = select_input_features(matrices[representation], mode)
            classifier = fit_classifier(kind, X, train_y, config, num_epochs=2)
            full_eval = encoder.transform(eval_p, eval_h, representation)
            probabilities = classifier.predict_proba(select_input_features(full_eval, mode))
            scores = score_probabilities(CLASS_IDS, probabilities)
            name = f'{representation}__{kind}__{mode}'
            directory = run_dir / name
            save_prediction_bundle(directory, classifier, encoder, representation, input_mode=mode, context={'purpose': 'synthetic_software_check_not_snli', 'config_sha256': workspace['manifest']['config_sha256'], 'labels': 'Arbitrary red/green/blue software-check targets, not NLI annotations.'})
            restored = load_prediction_bundle(directory)
            reloaded = restored.predict_proba(eval_p, eval_h)
            difference = float(np.max(np.abs(probabilities - reloaded)))
            if not np.allclose(probabilities, reloaded, rtol=0, atol=1e-07):
                raise AssertionError('Saved bundle did not reproduce its probabilities.')
            checks.append({'representation': representation, 'classifier': kind, 'input_mode': mode, 'input_dim': classifier.input_dim, 'parameters': classifier.parameter_count, 'probabilities_valid': True, 'finite_nll': bool(np.isfinite(scores['nll'])), 'reload_max_abs_difference': difference, 'logistic_converged': classifier.fit_info.get('converged'), 'epochs': classifier.fit_info.get('epochs_completed')})
            bundles[name] = directory
    finally:
        torch.set_num_threads(original_threads)
    table = pd.DataFrame(checks)
    table.to_csv(run_dir / 'checks.csv', index=False)
    (run_dir / 'smoke.json').write_text(json.dumps({'purpose': 'synthetic_software_check_not_snli', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'training_rows': len(train_y), 'evaluation_rows': len(eval_h), 'neural_epochs_for_check_only': 2, 'dataset_revision': config['dataset']['revision'], 'repository': workspace['manifest']['repository'], 'snli_rows_used': 0, 'reported_cv_results': False, 'checks': checks}, indent=2) + '\n', encoding='utf-8')
    return {'checks': table, 'run_dir': run_dir, 'bundles': bundles}

In [ ]:
CV_VERSION = "v1"
PROBABILITY_COLUMNS = [f"p_{name}" for name in LABEL_NAMES]
PREDICTION_ROWS = [*ROW_COLUMNS, "partition"]

### Reading the existing baseline

Read immutable baseline tables/checkpoints and verify their hashes. The
complete fitting/CV orchestration is defined in section 7.

In [ ]:
def _experiments_read(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

In [ ]:
def _experiments_verify(directory: Path, hashes: dict[str, str]) -> None:
    for name, expected in hashes.items():
        path = directory / name
        if not path.is_file() or _features_sha256(path) != expected:
            raise ValueError(f'CV artifact changed or is missing: {path}')

In [ ]:
def _experiments_load_report(marker: Path) -> dict[str, Any]:
    manifest = _experiments_read(marker)
    if manifest['status'] != 'complete':
        raise ValueError(f'CV is {manifest['status']}: {manifest['completed_units']}/{manifest['expected_units']} units complete. Resume the runner.')
    _experiments_verify(marker.parent, manifest['artifact_sha256'])
    for name in manifest['artifact_sha256']:
        if name.endswith('/complete.json'):
            unit = marker.parent / name
            result = _experiments_read(unit)
            _experiments_verify(unit.parent / result['attempt'], result['artifact_sha256'])
    config_path = Path(manifest['configuration_path'])
    if _features_sha256(config_path) != manifest['specification']['configuration_sha256']:
        raise ValueError('The frozen CV configuration changed.')
    directory = marker.parent
    return {'manifest': manifest, 'manifest_path': marker, 'run_dir': directory, 'tables': {name: pd.read_csv(directory / f'{name}.csv') for name in ('fold_metrics', 'summary', 'learning_curves', 'paired_accuracy', 'convergence')}, 'figures': {name: directory / f'{name}.png' for name in ('fold_scores', 'learning_curves')}}

In [ ]:
def load_cv_report(config_path: str | Path=FROZEN_CONFIG) -> dict[str, Any]:
    """Read/check the completed comparison; no training or setup side effects."""
    config_path = REPO_ROOT / config_path
    config = _experiments_read(config_path)
    pointer = REPO_ROOT / config['paths']['output_dir'] / 'manifests/cross_validation.json'
    if not pointer.is_file():
        raise FileNotFoundError('Run python -m part1.snli_experiments before inspecting section 7.')
    report = _experiments_load_report(Path(_experiments_read(pointer)['manifest_path']))
    if report['manifest']['specification']['configuration_sha256'] != _features_sha256(config_path):
        raise ValueError('Saved CV results use a different configuration.')
    return report

In [ ]:
cv_report = load_cv_report()
cv_manifest, cv_tables = cv_report["manifest"], cv_report["tables"]
cv_metrics, cv_summary = cv_tables["fold_metrics"], cv_tables["summary"]
logistic_row = cv_metrics.loc[
    cv_metrics["seed"].eq(0) & cv_metrics["fold"].eq(0)
    & cv_metrics["representation"].eq("unigram_counts")
    & cv_metrics["classifier"].eq("logistic") & cv_metrics["input_mode"].eq("pair")
].iloc[0]
logistic_bundle = load_prediction_bundle(logistic_row["checkpoint"])
print("Weights / bias:", logistic_bundle.classifier.estimator.coef_.shape,
      logistic_bundle.classifier.estimator.intercept_.shape)

In [ ]:
example = training_examples.iloc[0]
example_x = logistic_bundle.encoder.transform([example["premise"]], [example["hypothesis"]], "unigram_counts")
estimator = logistic_bundle.classifier.estimator
logits = np.asarray(example_x @ estimator.coef_.T + estimator.intercept_)
from scipy.special import softmax
manual_probabilities = softmax(logits, axis=1)
probabilities = logistic_bundle.predict_proba([example["premise"]], [example["hypothesis"]])
np.testing.assert_allclose(manual_probabilities, probabilities, rtol=1e-6, atol=1e-7)
display(pd.DataFrame({"label": LABEL_NAMES, "probability": probabilities[0]}))
print("Prediction:", label_names[int(probabilities.argmax())], "| gold:", label_names[int(example["label"])])

### A one-hidden-layer MLP

The first hidden layer computes $h=\mathrm{ReLU}(xW_1+b_1)$; an output layer then
maps $h$ to three class scores. The nonlinear layer can combine input features.
It still cannot reconstruct distinctions absent from the input vector.

The baseline uses 128 hidden units and dropout 0.1. Let's inspect the actual
network and make the sparse first multiplication explicit.

In [ ]:
set_random_seeds(0)
D = pair_encoder.num_features("unigram_counts")
one_hidden = build_neural_model("mlp_one_hidden", D, baseline_config["neural_training"])
one_hidden.eval()
print(one_hidden)

example_batch = scipy_batch_to_torch(feature_matrices["train"]["unigram_counts"][:2])
with torch.no_grad():
    hidden = torch.relu(torch.sparse.mm(example_batch, one_hidden.weight) + one_hidden.bias)
    manual_logp = one_hidden.output(one_hidden.dropout(hidden))
    np.testing.assert_allclose(manual_logp.numpy(), one_hidden(example_batch).numpy())
print("Input → hidden → log-probabilities:", tuple(example_batch.shape), tuple(hidden.shape), tuple(manual_logp.shape))

### A two-hidden-layer MLP

The deeper baseline inserts a second affine/ReLU/dropout stage with 64 units.
Its first layer still has 128 units. More capacity can help, but it can also
increase overfitting. We will compare evidence rather than assume depth helps.

In [ ]:
two_hidden = build_neural_model("mlp_two_hidden", D, baseline_config["neural_training"])
print(two_hidden)
display(pd.DataFrame([
    {"model": "logistic", "parameters": 3 * D + 3},
    {"model": "one hidden layer", "parameters": sum(p.numel() for p in one_hidden.parameters())},
    {"model": "two hidden layers", "parameters": sum(p.numel() for p in two_hidden.parameters())},
]))

### Loss and the decision rule

NLL is $-\log p(y_{\mathrm{gold}}\mid x)$. Accuracy depends only on the chosen
label. Improving the probability of the correct label can improve NLL without
changing the prediction. Our networks return **log-probabilities**, so training
uses `NLLLoss`.

In [ ]:
toy_gold = np.array([0, 1, 2])
toy_probabilities = np.array([[.6, .3, .1], [.2, .5, .3], [.2, .2, .6]])
toy_scores = score_probabilities(toy_gold, toy_probabilities)
manual_nll = -np.log(toy_probabilities[np.arange(3), toy_gold]).mean()
assert np.isclose(manual_nll, toy_scores["nll"])
print("Accuracy:", toy_scores["accuracy"], "| NLL:", round(manual_nll, 4))

### One gradient update, then a short learning trace

Training repeats four operations: clear gradients, run the model, backpropagate
the loss, and update parameters. Here we show 30 updates on **16 training
examples** to make those operations visible. This is a teaching demonstration,
not a fitted candidate or a reported SNLI score. It uses no validation/test rows.
The full experiment loop is defined above in `fit_classifier` and `_models_fit_classifier`.

In [ ]:
set_random_seeds(0)
demo_model = build_neural_model("mlp_one_hidden", D, baseline_config["neural_training"])
demo_x = scipy_batch_to_torch(feature_matrices["train"]["unigram_counts"][:16])
demo_y = torch.as_tensor(fold_features["rows"]["train"]["label"].head(16).to_numpy(copy=True), dtype=torch.long)
demo_optimizer = torch.optim.AdamW(demo_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.NLLLoss()
demo_losses = []
original_threads = torch.get_num_threads()
torch.set_num_threads(1)
try:
    demo_model.train()
    for step in range(30):
        demo_optimizer.zero_grad(set_to_none=True)
        log_probabilities = demo_model(demo_x)
        loss = criterion(log_probabilities, demo_y)
        loss.backward()
        demo_optimizer.step()
        demo_losses.append(float(loss.detach()))
finally:
    torch.set_num_threads(original_threads)
assert np.isfinite(demo_losses).all()

In [ ]:
from io import BytesIO

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, 31), demo_losses)
ax.set(xlabel="Update on the same 16 training examples", ylabel="Training NLL",
       title="Teaching example: learning to fit one small batch")
fig.tight_layout()
figure_buffer = BytesIO()
fig.savefig(figure_buffer, format="png", dpi=120, bbox_inches="tight")
display(Image(data=figure_buffer.getvalue()))
plt.close(fig)

A falling training loss shows that parameters can fit these examples. It does
not establish generalization. We need separate validation measurements to
choose settings, which is the next experiment.

### Simple baselines

A training-prior baseline predicts from label frequencies. A hypothesis-only
logistic model drops the premise input block while keeping the same learned
preprocessing. It tests whether this classifier benefits from reading the
premise; strong performance alone does not establish the cause of that signal.

In [ ]:
display(cv_summary.loc[~cv_summary["primary"], [
    "classifier", "input_mode", "accuracy_mean", "nll_mean", "macro_f1_mean",
]].round(4))

## 6. Choosing tokenizer and model hyperparameters

The default search now has a **15-minute compute budget**, following the group's
revised request. It compares three joint tokenizer/model recipes across all nine
systems, keeps all eligible validation rows, and reuses checked completed baseline
observations. The short runner is in section 6.9; common preprocessing, fitting
and persistence code appears below. The earlier large search was stopped and is
retained as an optional extended policy. New tuned outer CV remains a separate
experiment; the historical baseline CV/Bayesian results are still in sections 7–8.

**Completed:** all 27 observations are available (9 reused, 18 new fits), in
**7 minutes 58 seconds** for the short pass. This is a limited validation search;
new tuned CV/Bayesian evaluation remains pending.

### 6.1 Saved default and optional extended policies

`snli_tuning.json` is now the 900-second default: baseline, compact/regularized,
and larger-vocabulary/faster-learning recipes. These vary BPE size, case and merge
frequency, bigram inventory and model settings jointly. This small search does
not isolate the effect of each setting. The previously planned extensive policy
is saved separately as `snli_tuning_extended.json`; its candidate/refinement and
full-data helpers are retained for optional later use.

In [ ]:
TUNING_CONFIG = REPO_ROOT / "part1/configs/snli_tuning.json"
EXTENDED_TUNING_CONFIG = REPO_ROOT / "part1/configs/snli_tuning_extended.json"


def tuning_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def tuning_write_json(path, value):
    """Atomic replacement leaves a readable report if execution is interrupted."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def tuning_source_sha256():
    """Hash definitions/imports/constants; changing a run switch does not fork a run."""
    import ast
    definitions = []
    for cell in nbformat.read(NOTEBOOK_PATH, 4).cells:
        if cell.cell_type != "code":
            continue
        for node in ast.parse(cell.source).body:
            if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef)):
                definitions.append(ast.dump(node, include_attributes=False))
            elif isinstance(node, ast.Assign) and all(
                isinstance(target, ast.Name) and target.id.isupper()
                and not target.id.startswith("RUN_") for target in node.targets
            ):
                definitions.append(ast.dump(node, include_attributes=False))
    return tuning_digest(definitions)


def tuning_candidate(representation, kind, config, *, origin="coarse", parent=None):
    """Identity contains the complete predictive recipe, not its search position."""
    recipe = {
        "representation": representation, "classifier": kind,
        "tokenizer": deepcopy(config["tokenizer"]),
        "bigram_features": deepcopy(config["bigram_features"]),
        "model": deepcopy(config["logistic_training"] if kind == "logistic"
                          else config["neural_training"]),
    }
    if kind != "logistic":
        recipe["model"]["hidden_dims"] = {kind: recipe["model"]["hidden_dims"][kind]}
        recipe["model"].pop("num_epochs", None)
    # Unigram recipes do not have a tunable bigram inventory.
    if representation != "bigram_counts":
        recipe["bigram_features"] = {"min_frequency": 2, "max_features": 1}
    return {"candidate_id": tuning_digest(recipe)[:16], **recipe,
            "origin": origin, "parent": parent}


def tuning_model_config(candidate, base):
    config = deepcopy(base)
    config["tokenizer"] = deepcopy(candidate["tokenizer"])
    config["bigram_features"] = deepcopy(candidate["bigram_features"])
    target = "logistic_training" if candidate["classifier"] == "logistic" else "neural_training"
    config[target].update(deepcopy(candidate["model"]))
    return config

In [ ]:
tuning_policy = json.loads(TUNING_CONFIG.read_text())
display(pd.DataFrame(tuning_policy["candidates"]))
print("Budget:", tuning_policy["time_budget_seconds"], "seconds | 3 recipes × 9 systems")
print("Training: complete-group sample targeting 100,000 rows | validation: all eligible rows")

### 6.2 Optional extended candidates and refinement

Each system tries all 24 vocabulary × merge-frequency × normalization combinations. Classifier settings use balanced, seeded draws, with baseline hyperparameters included first (neural epochs are now selected by validation). This is a joint random search, not an exhaustive Cartesian product. Up to three one-setting neighbours of each screening winner refine regularization/learning rate and vocabulary size; predefined boundary extensions are recorded as new candidates.

In [ ]:
def make_tuning_candidates(base, policy):
    """Balanced random search: every BPE combination once per system, baseline first."""
    from itertools import product
    space = policy["space"]
    tokenizers_grid = list(product(space["vocab_size"], space["merge_min_frequency"],
                                   space["normalization"]))
    baseline_tuple = tuple(base["tokenizer"][key] for key in
                           ("vocab_size", "min_frequency", "normalization"))
    tokenizers_grid.remove(baseline_tuple)
    rng = np.random.default_rng(policy["search_seed"])
    rng.shuffle(tokenizers_grid)
    tokenizers_grid.insert(0, baseline_tuple)
    candidates = []
    for representation in REPRESENTATIONS:
        for kind in CLASSIFIERS:
            # Independent shuffled cycles ensure each model-setting value is tried.
            choices = {}
            for name in ("C", "learning_rate", "dropout", "weight_decay", "batch_size",
                         "one_hidden", "two_hidden", "bigram_cap", "bigram_min_frequency"):
                values = space[name]
                indices = np.resize(np.arange(len(values)), len(tokenizers_grid))
                rng.shuffle(indices)
                choices[name] = [deepcopy(values[int(j)]) for j in indices]
            for index, (vocab, frequency, normalization) in enumerate(tokenizers_grid):
                config = deepcopy(base)
                config["tokenizer"].update(vocab_size=vocab, min_frequency=frequency,
                                            normalization=normalization)
                if index:
                    if representation == "bigram_counts":
                        config["bigram_features"].update(
                            max_features=choices["bigram_cap"][index],
                            min_frequency=choices["bigram_min_frequency"][index])
                    if kind == "logistic":
                        config["logistic_training"]["C"] = choices["C"][index]
                    else:
                        config["neural_training"].update({key: choices[key][index] for key in
                            ("learning_rate", "dropout", "weight_decay", "batch_size")})
                        config["neural_training"]["hidden_dims"][kind] = (
                            [choices["one_hidden"][index]] if kind == "mlp_one_hidden"
                            else choices["two_hidden"][index])
                candidate = tuning_candidate(representation, kind, config)
                candidate["baseline_hyperparameters"] = index == 0
                candidates.append(candidate)
    assert len(candidates) == len(tokenizers_grid) * 9
    assert len({candidate["candidate_id"] for candidate in candidates}) == len(candidates)
    return candidates


def tuning_refinements(candidates, records, base, policy):
    """Three one-setting neighbours of each screening winner, including boundaries."""
    output = []
    seen = {candidate["candidate_id"] for candidate in candidates}
    lookup = {candidate["candidate_id"]: candidate for candidate in candidates}
    for representation in REPRESENTATIONS:
        for kind in CLASSIFIERS:
            eligible = [row for row in records if row["representation"] == representation
                        and row["classifier"] == kind and row["eligible"]]
            best = min(eligible, key=lambda row: (-row["accuracy"], row["nll"], row["candidate_id"]))
            parent = lookup[best["candidate_id"]]
            original = tuning_model_config(parent, base)
            modifications = []
            if kind == "logistic":
                current = original["logistic_training"]["C"]
                modifications.extend(("logistic_training", "C", current * factor)
                                     for factor in (0.1, 10.0))
            else:
                current = original["neural_training"]["learning_rate"]
                modifications.extend(("neural_training", "learning_rate", current * factor)
                                     for factor in (1/3, 3.0))
            vocab = original["tokenizer"]["vocab_size"]
            vocab_values = policy["space"]["vocab_size"]
            neighbour = (1000 if vocab == min(vocab_values) else
                         30000 if vocab == max(vocab_values) else
                         vocab_values[vocab_values.index(vocab) + 1])
            modifications.append(("tokenizer", "vocab_size", neighbour))
            for section, key, value in modifications:
                config = deepcopy(original)
                config[section][key] = value
                candidate = tuning_candidate(representation, kind, config,
                                             origin="refinement", parent=parent["candidate_id"])
                if candidate["candidate_id"] not in seen:
                    output.append(candidate)
                    seen.add(candidate["candidate_id"])
    return output

### 6.3 Train-only preprocessing and coverage

A BPE normalizer is serialized inside its tokenizer, so lowercasing applies during both fitting and prediction. Punctuation and negation are retained. Token IDs never need to match across independently fitted pipelines. Tokenization is cached per training partition and tokenizer recipe; bigram inventories are learned from training pairs only. We construct only the sparse representation needed by a trial.

In [ ]:
def tuning_sparse_matrix(encoder, tokens, representation):
    """Construct only the requested role blocks, without a combined dense/cache matrix."""
    width = encoder.vocab_size
    blocks = []
    inventory = np.array([left * width + right for left, right in encoder.bigrams], dtype=np.int64)
    permutation = np.argsort(inventory)
    sorted_inventory = inventory[permutation]
    for ids, offsets in tokens.roles():
        if representation in ("indicators", "unigram_counts"):
            rows = np.repeat(np.arange(len(tokens), dtype=np.int32), np.diff(offsets))
            block = sparse.csr_matrix((np.ones(len(ids), dtype=np.float32), (rows, ids)),
                                      shape=(len(tokens), width))
            if representation == "indicators":
                block.data.fill(1)
        elif representation == "bigram_counts":
            # Remove each sentence's final position before forming adjacent pairs.
            positions = np.arange(max(0, len(ids) - 1), dtype=np.int64)
            valid = np.ones(len(positions), dtype=bool)
            boundaries = offsets[1:-1] - 1
            boundaries = boundaries[(boundaries >= 0) & (boundaries < len(valid))]
            valid[boundaries] = False
            positions = positions[valid]
            packed = ids[positions].astype(np.int64) * width + ids[positions + 1]
            locations = np.searchsorted(sorted_inventory, packed)
            retained = locations < len(sorted_inventory)
            if len(sorted_inventory):
                retained[retained] &= sorted_inventory[locations[retained]] == packed[retained]
            rows = np.searchsorted(offsets[1:], positions[retained], side="right")
            columns = permutation[locations[retained]]
            block = sparse.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, columns)),
                                      shape=(len(tokens), len(inventory)))
        else:
            raise ValueError(f"Unknown representation: {representation}")
        blocks.append(block)
    matrix = sparse.hstack(blocks, format="csr", dtype=np.float32)
    matrix.sort_indices()
    return matrix


def tuning_coverage(encoder, tokens, matrix, representation):
    count = sum(len(ids) for ids, _ in tokens.roles())
    unknown = encoder.tokenizer.token_to_id(encoder.unk_token)
    possible_bigrams = sum(int(np.maximum(np.diff(offsets) - 1, 0).sum())
                           for _, offsets in tokens.roles())
    return {
        "actual_bpe_vocabulary": encoder.vocab_size, "bigram_inventory": len(encoder.bigrams),
        "pieces_per_pair": count / max(1, len(tokens)),
        "unknown_piece_rate": sum(int((ids == unknown).sum()) for ids, _ in tokens.roles()) / max(1, count),
        "retained_bigram_fraction": float(matrix.sum()) / max(1, possible_bigrams)
                                     if representation == "bigram_counts" else None,
        "empty_pair_fraction": float((matrix.getnnz(axis=1) == 0).mean()),
        "features": matrix.shape[1], "nnz": matrix.nnz,
        "csr_bytes": matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes,
    }


def tuning_features(directory, training, validation, candidate):
    """Cache BPE tokenization by training-row identity and tokenizer settings only."""
    tokenizer_id = tuning_digest(candidate["tokenizer"])[:16]
    cache = directory / "tokenizers" / tokenizer_id
    metadata_path = cache / "cache.json"
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
        if metadata["tokenizer"] != candidate["tokenizer"]:
            raise ValueError("Tokenizer cache configuration mismatch.")
        for name, digest in metadata["sha256"].items():
            if _features_sha256(cache / name) != digest:
                raise ValueError(f"Changed tokenizer cache: {cache / name}")
        encoder = PairFeatureEncoder.load(cache / "encoder")
        train_tokens = TokenizedPairs.load(cache / "train_tokens.npz")
        validation_tokens = TokenizedPairs.load(cache / "validation_tokens.npz")
    else:
        cache.mkdir(parents=True, exist_ok=True)
        # No validation text enters BPE training or the bigram frequency counts.
        encoder, train_tokens = fit_pair_features(
            training["premise"], training["hypothesis"], candidate["tokenizer"],
            {"min_frequency": 1, "max_features": 1})
        validation_tokens = tokenize_pairs(encoder.tokenizer, validation["premise"], validation["hypothesis"])
        encoder.save(cache / "encoder")
        train_tokens.save(cache / "train_tokens.npz")
        validation_tokens.save(cache / "validation_tokens.npz")
        metadata = {"tokenizer": candidate["tokenizer"], "training_rows": len(train_tokens),
                    "validation_rows": len(validation_tokens), "sha256": {
                        str(path.relative_to(cache)): _features_sha256(path)
                        for path in cache.rglob("*") if path.is_file()}}
        tuning_write_json(metadata_path, metadata)
    if len(train_tokens) != len(training["label"]) or len(validation_tokens) != len(validation["label"]):
        raise ValueError("Cached tokenization row counts differ from the frozen selection.")
    bigrams, counts = learn_bigrams(train_tokens, **candidate["bigram_features"])
    encoder = PairFeatureEncoder(encoder.tokenizer, bigrams, counts, encoder.unk_token)
    representation = candidate["representation"]
    train_matrix = tuning_sparse_matrix(encoder, train_tokens, representation)
    validation_matrix = tuning_sparse_matrix(encoder, validation_tokens, representation)
    coverage = {"training": tuning_coverage(encoder, train_tokens, train_matrix, representation),
                "validation": tuning_coverage(encoder, validation_tokens, validation_matrix, representation)}
    return encoder, train_matrix, validation_matrix, coverage

### 6.4 Select a checkpoint for each candidate

Accuracy is primary, with validation NLL breaking exact ties. The default short policy uses at most 8 epochs and patience 3. The optional extended policy starts at 30 epochs and may continue to 60. We restore the best weights, rather than using the final epoch. Runs still improving at the absolute ceiling are flagged for review. Logistic convergence is checked separately from predictive tuning.

In [ ]:
def tuning_validation_metrics(labels, probabilities):
    probabilities = validate_probabilities(probabilities, len(labels))
    labels = _models_labels(labels, len(labels))
    return {"accuracy": float((probabilities.argmax(1) == labels).mean()),
            "nll": float(-np.log(np.clip(probabilities[np.arange(len(labels)), labels],
                                         np.finfo(np.float64).eps, 1)).mean()),
            "macro_f1": float(f1_score(labels, probabilities.argmax(1), labels=CLASS_IDS,
                                       average="macro", zero_division=0))}


def fit_tuning_candidate(kind, X, y, X_validation, y_validation, config, policy, *, seed):
    """Fit on training labels; only independent validation chooses the checkpoint."""
    if kind == "logistic":
        working = deepcopy(config)
        attempts = []
        while True:
            model = fit_classifier(kind, X, y, working, seed=seed)
            attempts.append({"max_iter": working["logistic_training"]["max_iter"],
                             **model.fit_info})
            if model.fit_info["converged"] or working["logistic_training"]["max_iter"] >= policy["logistic_max_iter"]:
                break
            working["logistic_training"]["max_iter"] = min(
                policy["logistic_max_iter"], working["logistic_training"]["max_iter"] * 2)
        model.fit_info["convergence_attempts"] = attempts
        model.fit_info["selection_eligible"] = model.fit_info["converged"]
        return model
    X, X_validation = _models_matrix(X), _models_matrix(X_validation)
    y = _models_labels(y, X.shape[0], require_all=True)
    y_validation = _models_labels(y_validation, X_validation.shape[0], require_all=True)
    if X.shape[1] != X_validation.shape[1]:
        raise ValueError("Training and validation coordinates differ.")
    settings = deepcopy(config["neural_training"])
    original_threads = torch.get_num_threads()
    torch.set_num_threads(settings["num_threads"])
    started = perf_counter()
    try:
        set_random_seeds(seed)
        model = build_neural_model(kind, X.shape[1], settings).to(config["device"])
        optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"],
                                     weight_decay=settings["weight_decay"])
        criterion = nn.NLLLoss()
        rng = np.random.default_rng(seed)
        history, best_state = [], None
        best_key, best_epoch, stale = None, 0, 0
        ceiling = policy["initial_max_epochs"]
        epoch = 0
        while epoch < ceiling:
            epoch += 1
            epoch_started = perf_counter()
            model.train()
            order = rng.permutation(len(y))
            loss_sum, correct = 0.0, 0
            for start in range(0, len(y), settings["batch_size"]):
                indices = order[start:start + settings["batch_size"]]
                target = torch.as_tensor(y[indices], dtype=torch.long, device=config["device"])
                features = scipy_batch_to_torch(X[indices], config["device"])
                optimizer.zero_grad(set_to_none=True)
                output = model(features)
                loss = criterion(output, target)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Nonfinite neural training loss.")
                loss.backward()
                optimizer.step()
                loss_sum += float(loss.item()) * len(indices)
                correct += int((output.argmax(1) == target).sum())
            fitted = FittedClassifier(kind, model, X.shape[1], settings, history, {})
            scores = tuning_validation_metrics(y_validation, fitted.predict_proba(X_validation))
            # Accuracy first; NLL breaks exact ties. Both rules are fixed before search.
            key = (scores["accuracy"], -scores["nll"])
            if best_key is None or key > best_key:
                best_key, best_epoch, stale = key, epoch, 0
                best_state = {name: tensor.detach().cpu().clone()
                              for name, tensor in model.state_dict().items()}
            else:
                stale += 1
            history.append({"epoch": epoch, "train_nll_online": loss_sum / len(y),
                            "train_accuracy_online": correct / len(y),
                            "validation_accuracy": scores["accuracy"], "validation_nll": scores["nll"],
                            "seconds": perf_counter() - epoch_started})
            if stale >= policy["patience"]:
                break
            if epoch == ceiling and stale < policy["patience"]:
                ceiling = min(policy["absolute_max_epochs"], ceiling + policy["initial_max_epochs"])
        model.load_state_dict(best_state)
        model.eval()
        settings["num_epochs"] = best_epoch
        return FittedClassifier(kind, model, X.shape[1], settings, history, {
            "seed": seed, "training_rows": len(y), "validation_rows": len(y_validation),
            "class_order": CLASS_IDS.tolist(), "best_epoch": best_epoch,
            "epochs_completed": epoch, "checkpoint_policy": "best_validation_accuracy_then_nll",
            "stopping_reason": "patience" if stale >= policy["patience"] else "absolute_epoch_ceiling",
            "needs_ceiling_review": epoch == policy["absolute_max_epochs"] and stale < policy["patience"],
            "selection_eligible": True, "fit_seconds": perf_counter() - started,
            "torch_cpu_threads": settings["num_threads"], "weight_decay_scope": "all parameters including biases"})
    finally:
        torch.set_num_threads(original_threads)

### 6.5 Save each completed trial

Every trial saves its recipe, validation probabilities, history, coverage, cost and a complete prediction bundle. A reload check predicts from raw validation text. Checksums and independently recomputed metrics guard reuse. Completed trials are reused on resume; an interrupted in-progress trial restarts. One worker fits one model at a time.

In [ ]:
def load_tuning_trial(directory):
    directory = Path(directory)
    path = directory / "trial.json"
    record = json.loads(path.read_text())
    for name, digest in record["artifact_sha256"].items():
        if not (directory / name).is_file() or _features_sha256(directory / name) != digest:
            raise ValueError(f"Changed or missing tuning artifact: {directory / name}")
    with np.load(directory / "validation_predictions.npz", allow_pickle=False) as stored:
        scores = tuning_validation_metrics(stored["label"], stored["probabilities"])
    for metric, value in scores.items():
        if not np.isclose(record[metric], value, rtol=0, atol=1e-12):
            raise ValueError(f"Saved tuning metric disagrees with probabilities: {metric}")
    return record


def run_tuning_trial(directory, candidate, base, policy, training, validation,
                     encoder, X, X_validation, coverage, *, seed, stage):
    directory = Path(directory)
    if (directory / "trial.json").exists():
        record = load_tuning_trial(directory)
        if record["candidate_id"] != candidate["candidate_id"] or record["seed"] != seed:
            raise ValueError("Trial identity mismatch.")
        return record
    temporary = directory.with_name(directory.name + ".working")
    # A previous interruption may leave a partial bundle. Completed trials are immutable.
    if temporary.exists():
        import shutil
        shutil.rmtree(temporary)
    temporary.mkdir(parents=True)
    started = perf_counter()
    config = tuning_model_config(candidate, base)
    model = fit_tuning_candidate(candidate["classifier"], X, training["label"],
                                 X_validation, validation["label"], config, policy, seed=seed)
    probabilities = model.predict_proba(X_validation)
    scores = tuning_validation_metrics(validation["label"], probabilities)
    save_prediction_bundle(temporary / "bundle", model, encoder, candidate["representation"],
                           context={"candidate": candidate, "stage": stage, "model_seed": seed,
                                    "selection_data": "eligible official validation; no test evaluation"})
    replay = load_prediction_bundle(temporary / "bundle").predict_proba(
        validation["premise"][:16], validation["hypothesis"][:16])
    difference = float(np.max(np.abs(probabilities[:16] - replay)))
    if difference > 1e-7:
        raise ValueError("Saved preprocessing/model does not reproduce raw-text validation predictions.")
    np.savez_compressed(temporary / "validation_predictions.npz",
                        example_id=np.asarray(validation["example_id"], dtype=str),
                        label=np.asarray(validation["label"]), probabilities=probabilities)
    tuning_write_json(temporary / "candidate.json", candidate)
    record = {"candidate_id": candidate["candidate_id"], "representation": candidate["representation"],
              "classifier": candidate["classifier"], "origin": candidate["origin"],
              "stage": stage, "seed": seed, "training_rows": len(training["label"]),
              "validation_rows": len(validation["label"]), **scores,
              "eligible": model.fit_info["selection_eligible"],
              "best_epoch": model.fit_info.get("best_epoch"),
              "needs_ceiling_review": model.fit_info.get("needs_ceiling_review", False),
              "fit_info": model.fit_info, "history": model.history, "coverage": coverage,
              "parameters": model.parameter_count, "total_seconds": perf_counter() - started,
              "raw_text_reload_max_abs_difference": difference,
              "test_scored": False, "artifact_sha256": {
                  str(path.relative_to(temporary)): _features_sha256(path)
                  for path in temporary.rglob("*") if path.is_file()}}
    tuning_write_json(temporary / "trial.json", record)
    temporary.replace(directory)
    print(f"{stage}: {candidate['representation']} / {candidate['classifier']} "
          f"{candidate['candidate_id']} seed={seed}: val={scores['accuracy']:.4f}, "
          f"NLL={scores['nll']:.4f}, epochs={record['best_epoch']}, "
          f"{record['total_seconds']:.1f}s", flush=True)
    return record

### 6.6 Reports and optional extended seeded selection

Full-data neural candidates are ranked by mean validation accuracy across all three predeclared seeds, then mean NLL. We select a recipe, never whichever seed happens to score highest. The report can also show incomplete searches without labelling the current leader as an optimized model.

In [ ]:
def tuning_results_table(records):
    fields = ["stage", "representation", "classifier", "candidate_id", "seed", "training_rows",
              "validation_rows", "accuracy", "nll", "macro_f1", "best_epoch", "eligible",
              "needs_ceiling_review", "total_seconds", "parameters"]
    return pd.DataFrame([{name: row[name] for name in fields} for row in records], columns=fields)


def tuning_selection(records, policy):
    """Rank full-data recipes by mean score across ALL predeclared model seeds."""
    selected = []
    for representation in REPRESENTATIONS:
        for kind in CLASSIFIERS:
            seeds = [policy["model_seeds"][0]] if kind == "logistic" else policy["model_seeds"]
            full = [row for row in records if row["stage"] == "full_training"
                    and row["representation"] == representation and row["classifier"] == kind]
            options = []
            for candidate_id in sorted({row["candidate_id"] for row in full}):
                rows = [row for row in full if row["candidate_id"] == candidate_id]
                if {row["seed"] for row in rows} != set(seeds) or not all(row["eligible"] for row in rows):
                    continue
                options.append({"representation": representation, "classifier": kind,
                                "candidate_id": candidate_id, "model_seeds": seeds,
                                "mean_validation_accuracy": float(np.mean([row["accuracy"] for row in rows])),
                                "sd_validation_accuracy": float(np.std([row["accuracy"] for row in rows], ddof=1))
                                                           if len(rows) > 1 else 0.0,
                                "mean_validation_nll": float(np.mean([row["nll"] for row in rows])),
                                "best_epochs": [row["best_epoch"] for row in rows],
                                "needs_ceiling_review": any(row["needs_ceiling_review"] for row in rows)})
            if options:
                selected.append(min(options, key=lambda row: (-row["mean_validation_accuracy"],
                                                              row["mean_validation_nll"], row["candidate_id"])))
    return selected


def load_tuning_report(policy_path=TUNING_CONFIG):
    policy = json.loads(Path(policy_path).read_text())
    if policy.get("mode") == "budgeted":
        return load_budgeted_tuning_report(policy_path)
    output = REPO_ROOT / policy["output_dir"]
    pointer = output / "latest.json"
    if not pointer.exists():
        return None
    directory = Path(json.loads(pointer.read_text())["run_dir"])
    manifest = json.loads((directory / "manifest.json").read_text())
    if manifest["specification"]["policy"] != policy:
        raise ValueError("Latest tuning run uses a different policy; start or resume the current policy explicitly.")
    records = [load_tuning_trial(path.parent) for path in sorted(directory.glob("*/trials/*/trial.json"))]
    return {"run_dir": directory, "manifest": manifest, "records": records,
            "table": tuning_results_table(records), "selected": tuning_selection(records, policy)}

### 6.7 Optional extended staged runner (superseded as the default)

Screen on a fixed prefix of complete premise groups reaching 100,000 training rows, using **all 9,839 eligible validation rows**. Refine the winners, then retrain the top two candidates per system on **all 549,253 eligible training rows**. Full-data neural finalists use seeds 0, 3 and 5. Tokenizers are refitted on the training partition at each data size.

The optional `training_frame` accepts one outer training fold, with complete groups; the same search can run inside that fold without receiving its held-out data. **Tuned outer CV and its Bayesian comparisons remain a subsequent experiment.** Development-selected candidates must not be reused in outer CV and presented as an unbiased assessment of the tuning procedure.

In [ ]:
def run_tokenizer_model_search(policy_path=TUNING_CONFIG, *, max_new_trials=None,
                               training_frame=None, context="development"):
    """Resumable search. Optional outer-training frame must be supplied BEFORE selection.

    An outer fold calls this with its own eligible training frame and unique context;
    its held-out examples/labels never enter this function. Development winners
    must not be transplanted into CV and described as an unbiased tuning procedure.
    """
    if max_new_trials is not None and (not isinstance(max_new_trials, int) or isinstance(max_new_trials, bool) or max_new_trials < 1):
        raise ValueError("max_new_trials must be a positive integer or None.")
    policy = json.loads(Path(policy_path).read_text())
    if policy.get("mode") == "budgeted":
        if training_frame is not None or context != "development":
            raise ValueError("The quick policy is development selection only; tuned outer CV is a later experiment.")
        return run_budgeted_tuning_search(policy_path, max_new_trials=max_new_trials)
    base = json.loads(FROZEN_CONFIG.read_text())
    prepared = REPO_ROOT / base["paths"]["data_dir"] / "snli" / base["dataset"]["revision"] / f"cleaned-{CLEANING_VERSION}"
    cleaned = load_from_disk(str(prepared))
    pools = make_development_pools(cleaned)
    eligible = pools["cv_pool"]
    if training_frame is not None:
        if context == "development" or training_frame["example_id"].duplicated().any():
            raise ValueError("Outer-training input requires unique row IDs and a distinct context.")
        selected_ids = set(training_frame["example_id"])
        if not selected_ids.issubset(set(eligible["example_id"])):
            raise ValueError("Outer training contains examples outside the eligible training pool.")
        eligible = eligible.loc[eligible["example_id"].isin(selected_ids)].copy().reset_index(drop=True)
        if set(eligible["example_id"]) != selected_ids:
            raise ValueError("Outer training IDs do not match eligible source rows.")
        # Reject partial premise groups, even if callers accidentally split one.
        full_groups = pools["cv_pool"].groupby("premise_group").size()
        counts = eligible.groupby("premise_group").size()
        if not counts.equals(full_groups.loc[counts.index]):
            raise ValueError("Outer training must contain complete premise groups.")
    screening = select_complete_groups(eligible, min(policy["screening_target_rows"], len(eligible)),
                                       policy["sampling_seed"])
    validation_rows = pools["membership"].loc[pools["membership"]["pilot_validation"]].copy().reset_index(drop=True)
    selection_id = tuning_digest({"training": eligible["example_id"].tolist(),
                                  "validation": validation_rows["example_id"].tolist()})
    specification = {"policy": policy, "base_config": base, "context": context,
                     "selection_sha256": selection_id,
                     "training_fingerprint": cleaned["train"]._fingerprint,
                     "validation_fingerprint": cleaned["validation"]._fingerprint,
                     "source_sha256": tuning_source_sha256(),
                     "packages": {name: version(name) for name in
                                  ("numpy", "scipy", "scikit-learn", "torch", "tokenizers", "datasets")}}
    run_id = tuning_digest(specification)[:16]
    output = REPO_ROOT / policy["output_dir"]
    directory = output / run_id
    directory.mkdir(parents=True, exist_ok=True)
    lock = (directory / ".lock").open("w")
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        lock.close()
        raise RuntimeError(f"Another process is already running search {run_id}.") from None
    records, new_trials = [], 0
    manifest_path = directory / "manifest.json"
    manifest = {"run_id": run_id, "specification": specification,
                "status": "incomplete", "eligible_training_rows": len(eligible),
                "screening_rows": len(screening), "validation_rows": len(validation_rows),
                "test_scored": False, "started_at_utc": datetime.now(timezone.utc).isoformat()}
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text())
        if previous["specification"] != specification:
            lock.close()
            raise ValueError("Saved search specification changed.")
        manifest["started_at_utc"] = previous["started_at_utc"]
    for name, frame in (("eligible_training", eligible), ("screening", screening), ("validation", validation_rows)):
        selection_path = directory / f"{name}_rows.parquet"
        if selection_path.exists():
            if not pd.read_parquet(selection_path).equals(frame):
                lock.close()
                raise ValueError(f"Changed frozen row selection: {selection_path}")
        else:
            frame.to_parquet(selection_path, index=False)
    def update(status="incomplete", stage=None):
        manifest.update(status=status, stage=stage, completed_trials=len(records),
                        updated_at_utc=datetime.now(timezone.utc).isoformat(),
                        selected=tuning_selection(records, policy))
        tuning_write_json(manifest_path, manifest)
        tuning_results_table(records).to_csv(directory / "trial_metrics.csv", index=False)
        if context == "development":
            tuning_write_json(output / "latest.json", {"run_id": run_id, "run_dir": str(directory)})
    def data_for(frame, split):
        selected = cleaned[split].select(frame["clean_index"].tolist())
        return {name: selected[name].to_pylist() if hasattr(selected[name], "to_pylist")
                else list(selected[name]) for name in ("premise", "hypothesis", "label", "example_id")}
    validation = data_for(validation_rows, "validation")
    def run_stage(name, plan, train_frame, seeds):
        nonlocal new_trials
        stage_dir = directory / name
        stage_dir.mkdir(exist_ok=True)
        plan_path = stage_dir / "plan.json"
        if plan_path.exists() and json.loads(plan_path.read_text()) != plan:
            raise ValueError("Saved stage plan changed; completed results were preserved.")
        tuning_write_json(plan_path, plan)
        train = None
        feature_key, features = None, None
        # Group candidates so preprocessing is shared between compatible classifier fits.
        ordered = sorted(plan, key=lambda c: (not c.get("baseline_hyperparameters", False), tuning_digest(c["tokenizer"]),
                         tuning_digest(c["bigram_features"]), c["representation"], c["classifier"]))
        for candidate in ordered:
            model_seeds = seeds if candidate["classifier"] != "logistic" else seeds[:1]
            for seed in model_seeds:
                trial_dir = stage_dir / "trials" / f"{candidate['candidate_id']}-seed{seed}"
                if (trial_dir / "trial.json").exists():
                    record = load_tuning_trial(trial_dir)
                    if (record["candidate_id"] != candidate["candidate_id"] or record["seed"] != seed
                        or record["stage"] != name or record["training_rows"] != len(train_frame)):
                        raise ValueError("Saved trial does not match its planned identity.")
                    with np.load(trial_dir / "validation_predictions.npz", allow_pickle=False) as saved:
                        if (saved["example_id"].tolist() != validation["example_id"]
                            or saved["label"].tolist() != validation["label"]):
                            raise ValueError("Saved validation predictions do not match frozen rows/labels.")
                    records.append(record)
                    continue
                if max_new_trials is not None and new_trials >= max_new_trials:
                    update(stage=name)
                    return False
                if train is None:
                    train = data_for(train_frame, "train")
                key = tuning_digest({k: candidate[k] for k in
                                     ("tokenizer", "bigram_features", "representation")})
                if feature_key != key:
                    features = encoder = X = X_validation = None
                    gc.collect()
                    # Refinement uses the same screening partition/token cache.
                    cache_dir = directory / ("full_training" if name == "full_training" else "screening_cache")
                    features = tuning_features(cache_dir, train, validation, candidate)
                    feature_key = key
                encoder, X, X_validation, coverage = features
                record = run_tuning_trial(trial_dir, candidate, base, policy, train, validation,
                                          encoder, X, X_validation, coverage, seed=seed, stage=name)
                records.append(record)
                new_trials += 1
                update(stage=name)
        return True
    original_threads = torch.get_num_threads()
    torch.set_num_threads(base["neural_training"]["num_threads"])
    try:
        update(stage="coarse")
        coarse = make_tuning_candidates(base, policy)
        if not run_stage("coarse", coarse, screening, policy["model_seeds"][:1]):
            return {"run_dir": directory, "manifest": manifest, "records": records}
        refinement_path = directory / "refinement" / "plan.json"
        refinement = (json.loads(refinement_path.read_text()) if refinement_path.exists() else
                      tuning_refinements(coarse, records, base, policy))
        if not run_stage("refinement", refinement, screening, policy["model_seeds"][:1]):
            return {"run_dir": directory, "manifest": manifest, "records": records}
        lookup = {candidate["candidate_id"]: candidate for candidate in coarse + refinement}
        full_plan_path = directory / "full_training" / "plan.json"
        if full_plan_path.exists():
            finalists = json.loads(full_plan_path.read_text())
        else:
            finalists = []
            for representation in REPRESENTATIONS:
                for kind in CLASSIFIERS:
                    choices = [row for row in records if row["eligible"]
                               and row["representation"] == representation and row["classifier"] == kind]
                    ranked = sorted(choices, key=lambda row: (-row["accuracy"], row["nll"], row["candidate_id"]))
                    if len(ranked) < policy["full_training_finalists_per_system"]:
                        raise ValueError("Insufficient converged candidates for full-training promotion.")
                    finalists.extend(lookup[row["candidate_id"]] for row in
                                     ranked[:policy["full_training_finalists_per_system"]])
        if not run_stage("full_training", finalists, eligible, policy["model_seeds"]):
            return {"run_dir": directory, "manifest": manifest, "records": records}
        if len(tuning_selection(records, policy)) != 9:
            raise ValueError("The search lacks a fully assessed recipe for each of the nine systems.")
        update("complete", stage="full_training")
        tuning_write_json(directory / "selected_recipes.json", [
            {**selection, "recipe": lookup[selection["candidate_id"]]}
            for selection in manifest["selected"]])
        return {"run_dir": directory, "manifest": manifest, "records": records}
    except BaseException:
        update(stage=manifest.get("stage"))
        raise
    finally:
        torch.set_num_threads(original_threads)
        fcntl.flock(lock, fcntl.LOCK_UN)
        lock.close()

### 6.8 Plot saved validation measurements

The plotting function works with both policies. The default run cell follows the
short-policy implementation in section 6.9. Keep `RUN_JOINT_SEARCH = False` to
read results; set it true only to start or explicitly resume a 15-minute budget.

In [ ]:
def plot_tuning_results(report):
    """Development validation results: a learning/search diagnostic, never CV/test accuracy."""
    table = report["table"]
    if table.empty:
        print("No completed trials yet.")
        return
    display(table.groupby(["stage", "representation", "classifier"], sort=False).agg(
        trials=("accuracy", "size"), best_validation_accuracy=("accuracy", "max"),
        total_seconds=("total_seconds", "sum")).round(4))
    figures, axes = plt.subplots(1, 2, figsize=(12, 4))
    for representation in REPRESENTATIONS:
        rows = table.query("representation == @representation").reset_index(drop=True)
        axes[0].plot(np.arange(1, len(rows) + 1), rows["accuracy"].cummax(), label=representation)
    axes[0].set(xlabel="Completed trials within representation", ylabel="Best validation accuracy",
                title="Search progress (selection data)")
    axes[0].yaxis.set_major_formatter(PercentFormatter(1))
    axes[0].legend()
    best = max(report["records"], key=lambda row: row["accuracy"])
    if best["history"]:
        curves = pd.DataFrame(best["history"])
        axes[1].plot(curves["epoch"], curves["train_nll_online"], label="Training (online)")
        axes[1].plot(curves["epoch"], curves["validation_nll"], label="Validation")
        axes[1].axvline(best["best_epoch"], linestyle="--", color="gray", label="Saved checkpoint")
        axes[1].set(xlabel="Epoch", ylabel="NLL", title="Best completed neural trial")
        axes[1].legend()
    else:
        axes[1].axis("off")
    figures.tight_layout()
    plt.show()
    if report["selected"]:
        display(pd.DataFrame(report["selected"]))

### 6.9 Default: a 15-minute development search

Following the revised compute budget, the default policy uses **three joint
recipes per system**, at most 8 epochs, patience 3 and one model seed. It retains
the fixed 100,001-row whole-group training sample and all 9,839 validation rows.
The 27 planned observations include nine compatible baseline observations already
completed; their original files/provenance remain unchanged and are referenced
explicitly. Imported trials must match data fingerprints, rows, recipe, seed and
a stopping history compatible with the new ceiling.

The active work has a **900-second wall-clock limit**, including preprocessing.
At the deadline, the current trial may be interrupted and only completed trials
are retained. The notebook reports partial coverage and `budget_exhausted` if
needed. Resuming explicitly gives another 15-minute allowance; it never resumes
automatically beyond the budget. There is no full-data promotion, extra refinement
stage or three-seed confirmation in this policy. Results identify the **best
of a small tested set**, not thoroughly optimized pipelines.

The older, expensive functions remain available above for reference. Their
24-config search policy is archived in `snli_tuning_extended.json` and requires
passing that path explicitly. The default switch below runs the short policy.


The completed pass is `4baddae6266ac867`; it finished all 27 planned observations
before the deadline. No extra training stages are queued.

In [ ]:
class TuningBudgetExpired(TimeoutError):
    """The user-selected wall-clock budget has elapsed."""


@contextmanager
def tuning_time_limit(seconds):
    """Bound work on this Linux notebook kernel; restore any previous timer."""
    import signal
    if seconds <= 0 or threading.current_thread() is not threading.main_thread():
        raise ValueError("Use a positive budget on the notebook kernel's main thread.")
    started = perf_counter()
    previous_handler = signal.getsignal(signal.SIGALRM)
    def expired(signum, frame):
        raise TuningBudgetExpired("The tuning wall-clock budget has elapsed.")
    signal.signal(signal.SIGALRM, expired)
    previous_timer = signal.setitimer(signal.ITIMER_REAL, seconds)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, previous_handler)
        if previous_timer[0]:
            signal.setitimer(signal.ITIMER_REAL,
                             max(.001, previous_timer[0] - (perf_counter() - started)), previous_timer[1])


def make_budgeted_candidates(base, policy):
    """Three explicit joint recipes per system; no Cartesian product/refinement."""
    candidates = []
    for variant in policy["candidates"]:
        config = deepcopy(base)
        for section in ("tokenizer", "bigram_features", "logistic_training", "neural_training"):
            config[section].update(deepcopy(variant.get(section, {})))
        for representation in REPRESENTATIONS:
            for kind in CLASSIFIERS:
                candidate = tuning_candidate(representation, kind, config, origin=variant["name"])
                candidate["variant"] = variant["name"]
                candidates.append(candidate)
    if len({row["candidate_id"] for row in candidates}) != len(candidates):
        raise ValueError("The budgeted plan has duplicate recipes.")
    return candidates


def budgeted_selection(records, candidates):
    selected = []
    for representation in REPRESENTATIONS:
        for kind in CLASSIFIERS:
            options = [row for row in records if row["representation"] == representation
                       and row["classifier"] == kind and row["eligible"]]
            if not options:
                continue
            best = min(options, key=lambda row: (-row["accuracy"], row["nll"], row["candidate_id"]))
            planned = sum(c["representation"] == representation and c["classifier"] == kind
                          for c in candidates)
            selected.append({"representation": representation, "classifier": kind,
                             "candidate_id": best["candidate_id"], "validation_accuracy": best["accuracy"],
                             "validation_nll": best["nll"], "best_epoch": best["best_epoch"],
                             "needs_ceiling_review": best["needs_ceiling_review"],
                             "evaluated_candidates": len(options), "planned_candidates": planned,
                             "scope": "best tested: 100k training subset, one seed; no full-data confirmation"})
    return selected


def budgeted_record(path, *, reused=False):
    record = load_tuning_trial(path)
    return {**record, "stage": "budgeted", "trial_dir": str(Path(path).resolve()), "reused": reused}


def budgeted_training_signature(path):
    """Compare the actual fitting/feature definitions before importing observations."""
    import ast
    required = {"train_tokenizer", "fit_pair_features", "learn_bigrams", "tokenize_pairs",
                "TokenizedPairs", "PairFeatureEncoder", "tuning_sparse_matrix",
                "fit_tuning_candidate", "fit_classifier", "_models_fit_classifier",
                "build_neural_model", "PairMLP", "SparseMLP", "FittedClassifier",
                "scipy_batch_to_torch", "set_random_seeds", "validate_probabilities",
                "_models_matrix", "_models_labels", "tuning_candidate", "tuning_model_config",
                "tuning_validation_metrics"}
    definitions = {}
    for cell in nbformat.read(path, 4).cells:
        if cell.cell_type != "code":
            continue
        for node in ast.parse(cell.source).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in required:
                definitions[node.name] = ast.dump(node, include_attributes=False)
    if set(definitions) != required:
        raise ValueError("Source snapshot lacks training/feature definitions needed to verify reuse.")
    return tuning_digest(definitions)


def budgeted_imports(directory, policy, candidates, screening, validation_rows, validation, specification):
    """Reuse exact, checked observations with explicit source lineage; never rename their provenance."""
    path = directory / "reused_trials.json"
    if path.exists():
        imports = json.loads(path.read_text())
        for imported in imports:
            if _features_sha256(Path(imported["directory"]) / "trial.json") != imported["trial_sha256"]:
                raise ValueError("An imported trial record changed.")
        return imports
    source = REPO_ROOT / policy["reuse_run_dir"] if policy.get("reuse_run_dir") else None
    imports = []
    if source is not None and (source / "manifest.json").exists():
        manifest = json.loads((source / "manifest.json").read_text())
        old = manifest["specification"]
        expected = specification
        source_snapshot = source / "archived_search_execution.ipynb"
        same_implementation = (source_snapshot.exists() and
            budgeted_training_signature(source_snapshot) == budgeted_training_signature(NOTEBOOK_PATH))
        same_data = all(old.get(key) == expected.get(key) for key in
                        ("base_config", "training_fingerprint", "validation_fingerprint", "packages"))
        same_rows = (pd.read_parquet(source / "screening_rows.parquet").equals(screening)
                     and pd.read_parquet(source / "validation_rows.parquet").equals(validation_rows))
        if same_implementation and same_data and same_rows and old["policy"]["patience"] == policy["patience"]:
            requested = {c["candidate_id"] for c in candidates}
            for trial_path in sorted(source.glob("coarse/trials/*/trial.json")):
                if trial_path.parent.name.endswith(".working"):
                    continue
                record = load_tuning_trial(trial_path.parent)
                if record["candidate_id"] not in requested or record["seed"] != policy["model_seeds"][0]:
                    continue
                info = record["fit_info"]
                compatible_stop = (record["classifier"] == "logistic" or
                    (info["epochs_completed"] <= policy["absolute_max_epochs"] and
                     (info["stopping_reason"] == "patience" or
                      info["epochs_completed"] == policy["absolute_max_epochs"])))
                if not compatible_stop or not record["eligible"]:
                    continue
                with np.load(trial_path.parent / "validation_predictions.npz", allow_pickle=False) as stored:
                    if (stored["example_id"].tolist() != validation["example_id"]
                        or stored["label"].tolist() != validation["label"]):
                        raise ValueError("Imported validation rows/labels differ.")
                imports.append({"candidate_id": record["candidate_id"],
                                "directory": str(trial_path.parent.resolve()),
                                "trial_sha256": _features_sha256(trial_path),
                                "source_run_id": manifest["run_id"],
                                "compatibility": "same training implementation, data, rows, recipe, seed and compatible stopping history"})
    tuning_write_json(path, imports)
    return imports


def load_budgeted_tuning_report(policy_path=TUNING_CONFIG):
    policy = json.loads(Path(policy_path).read_text())
    output = REPO_ROOT / policy["output_dir"]
    if not (output / "latest.json").exists():
        return None
    directory = Path(json.loads((output / "latest.json").read_text())["run_dir"])
    manifest = json.loads((directory / "manifest.json").read_text())
    if manifest["specification"]["policy"] != policy:
        return None  # The old, stopped large search is separate historical evidence.
    records = []
    for imported in json.loads((directory / "reused_trials.json").read_text()):
        if _features_sha256(Path(imported["directory"]) / "trial.json") != imported["trial_sha256"]:
            raise ValueError("An imported trial record changed.")
        records.append(budgeted_record(imported["directory"], reused=True))
    records.extend(budgeted_record(path.parent) for path in sorted(directory.glob("budgeted/trials/*/trial.json"))
                   if not path.parent.name.endswith(".working"))
    candidates = json.loads((directory / "plan.json").read_text())
    return {"run_dir": directory, "manifest": manifest, "records": records,
            "table": tuning_results_table(records), "selected": budgeted_selection(records, candidates)}

In [ ]:
def run_budgeted_tuning_search(policy_path=TUNING_CONFIG, *, max_new_trials=None):
    """At most 15 minutes per invocation; completed trials and reused observations survive."""
    started = perf_counter()
    if max_new_trials is not None and (not isinstance(max_new_trials, int)
        or isinstance(max_new_trials, bool) or max_new_trials < 1):
        raise ValueError("max_new_trials must be a positive integer or None.")
    policy = json.loads(Path(policy_path).read_text())
    if policy["mode"] != "budgeted" or policy["model_seeds"] != [0]:
        raise ValueError("This budgeted development search uses one predeclared seed, 0.")
    if not 0 < policy["time_budget_seconds"] <= 900:
        raise ValueError("The requested quick-search budget must be at most 900 seconds.")
    base = json.loads(FROZEN_CONFIG.read_text())
    prepared = (REPO_ROOT / base["paths"]["data_dir"] / "snli" /
                base["dataset"]["revision"] / f"cleaned-{CLEANING_VERSION}")
    cleaned = load_from_disk(str(prepared))
    pools = make_development_pools(cleaned)
    screening = select_complete_groups(pools["cv_pool"], min(policy["screening_target_rows"], len(pools["cv_pool"])),
                                       policy["sampling_seed"])
    validation_rows = pools["membership"].loc[pools["membership"]["pilot_validation"]].copy().reset_index(drop=True)
    def data_for(frame, split):
        data = cleaned[split].select(frame["clean_index"].tolist())
        return {name: list(data[name]) for name in ("premise", "hypothesis", "label", "example_id")}
    validation = data_for(validation_rows, "validation")
    specification = {"policy": policy, "base_config": base,
                     "training_fingerprint": cleaned["train"]._fingerprint,
                     "validation_fingerprint": cleaned["validation"]._fingerprint,
                     "selection_sha256": tuning_digest({"training": screening["example_id"].tolist(),
                                                         "validation": validation["example_id"]}),
                     "source_sha256": tuning_source_sha256(),
                     "packages": {name: version(name) for name in
                                  ("numpy", "scipy", "scikit-learn", "torch", "tokenizers", "datasets")}}
    run_id = tuning_digest(specification)[:16]
    output = REPO_ROOT / policy["output_dir"]
    directory = output / run_id
    directory.mkdir(parents=True, exist_ok=True)
    lock = (directory / ".lock").open("w")
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        lock.close()
        raise RuntimeError("This budgeted search is already running.") from None
    original_threads = torch.get_num_threads()
    records, new_trials = [], 0
    manifest = {"run_id": run_id, "specification": specification, "status": "incomplete",
                "planned_trials": len(policy["candidates"]) * 9,
                "screening_rows": len(screening), "eligible_training_rows": len(pools["cv_pool"]),
                "validation_rows": len(validation_rows), "test_scored": False,
                "full_training_confirmation": False, "multiple_seed_confirmation": False,
                "started_at_utc": datetime.now(timezone.utc).isoformat()}
    def update(status):
        manifest.update(status=status, completed_trials=len(records),
                        reused_trials=sum(row["reused"] for row in records),
                        new_trials_this_invocation=new_trials,
                        elapsed_seconds_this_invocation=perf_counter() - started,
                        updated_at_utc=datetime.now(timezone.utc).isoformat(),
                        selected=budgeted_selection(records, candidates))
        tuning_write_json(directory / "manifest.json", manifest)
        tuning_results_table(records).to_csv(directory / "trial_metrics.csv", index=False)
        if status in ("complete", "budget_exhausted"):
            tuning_write_json(directory / "selected_recipes.json", [
                {**row, "recipe": next(c for c in candidates if c["candidate_id"] == row["candidate_id"])}
                for row in manifest["selected"]])
        tuning_write_json(output / "latest.json", {"run_id": run_id, "run_dir": str(directory)})
    try:
        torch.set_num_threads(base["neural_training"]["num_threads"])
        candidates = make_budgeted_candidates(base, policy)
        existing_manifest = directory / "manifest.json"
        if existing_manifest.exists():
            previous = json.loads(existing_manifest.read_text())
            if previous["specification"] != specification:
                raise ValueError("Saved quick-search specification changed.")
            manifest["started_at_utc"] = previous["started_at_utc"]
        for name, frame in (("screening", screening), ("validation", validation_rows)):
            path = directory / f"{name}_rows.parquet"
            if path.exists() and not pd.read_parquet(path).equals(frame):
                raise ValueError("Saved quick-search row selection changed.")
            if not path.exists():
                frame.to_parquet(path, index=False)
        snapshot = directory / "source_notebook.ipynb"
        if not snapshot.exists():
            nbformat.write(nbformat.read(NOTEBOOK_PATH, 4), snapshot)
        plan_path = directory / "plan.json"
        if plan_path.exists() and json.loads(plan_path.read_text()) != candidates:
            raise ValueError("Saved quick-search candidate plan changed.")
        tuning_write_json(plan_path, candidates)
        imports = budgeted_imports(directory, policy, candidates, screening, validation_rows, validation, specification)
        records.extend(budgeted_record(item["directory"], reused=True) for item in imports)
        completed = {row["candidate_id"] for row in records}
        for path in sorted(directory.glob("budgeted/trials/*/trial.json")):
            if path.parent.name.endswith(".working"):
                continue
            row = budgeted_record(path.parent)
            if row["candidate_id"] in completed:
                raise ValueError("A reused observation also exists as a new trial.")
            with np.load(path.parent / "validation_predictions.npz", allow_pickle=False) as stored:
                if stored["example_id"].tolist() != validation["example_id"] or stored["label"].tolist() != validation["label"]:
                    raise ValueError("Saved quick-trial validation rows changed.")
            records.append(row)
            completed.add(row["candidate_id"])
        update("incomplete")
        print(f"Quick search: {len(candidates)} planned observations; {len(imports)} reused; "
              f"budget {policy['time_budget_seconds']}s. No full-data or multi-seed stage.", flush=True)
        if len(records) == len(candidates):
            update("complete")
            return {"run_dir": directory, "manifest": manifest, "records": records}
        remaining = policy["time_budget_seconds"] - (perf_counter() - started)
        if remaining <= 0:
            update("budget_exhausted")
            return {"run_dir": directory, "manifest": manifest, "records": records}
        train, features, feature_key = None, None, None
        try:
            with tuning_time_limit(remaining):
                for candidate in candidates:
                    if candidate["candidate_id"] in completed:
                        continue
                    if max_new_trials is not None and new_trials >= max_new_trials:
                        update("incomplete")
                        break
                    # Avoid starting another fit at the very end; interrupt active work at the deadline.
                    if policy["time_budget_seconds"] - (perf_counter() - started) < policy["minimum_start_seconds"]:
                        raise TuningBudgetExpired("Insufficient time for another complete trial.")
                    if train is None:
                        train = data_for(screening, "train")
                    key = tuning_digest({name: candidate[name] for name in
                                         ("tokenizer", "bigram_features", "representation")})
                    if key != feature_key:
                        features = encoder = X = X_validation = None
                        gc.collect()
                        features = tuning_features(directory / "budgeted", train, validation, candidate)
                        feature_key = key
                    encoder, X, X_validation, coverage = features
                    trial_dir = directory / "budgeted/trials" / f"{candidate['candidate_id']}-seed0"
                    run_tuning_trial(trial_dir, candidate, base, policy, train, validation,
                                     encoder, X, X_validation, coverage, seed=0, stage="budgeted")
                    records.append(budgeted_record(trial_dir))
                    completed.add(candidate["candidate_id"])
                    new_trials += 1
                    update("incomplete")
                else:
                    update("complete")
        except TuningBudgetExpired as error:
            update("budget_exhausted")
            print(str(error) + " Completed trials are preserved.", flush=True)
        tuning_write_json(directory / "selected_recipes.json", [
            {**row, "recipe": next(c for c in candidates if c["candidate_id"] == row["candidate_id"])}
            for row in manifest["selected"]])
        return {"run_dir": directory, "manifest": manifest, "records": records}
    except BaseException:
        if "candidates" in locals():
            update("interrupted")
        raise
    finally:
        torch.set_num_threads(original_threads)
        fcntl.flock(lock, fcntl.LOCK_UN)
        lock.close()

In [ ]:
RUN_JOINT_SEARCH = False

if RUN_JOINT_SEARCH:
    tuning_run = run_tokenizer_model_search()

tuning_report = load_tuning_report()
if tuning_report is None:
    print("No joint-search trials saved yet. Run section 6.7 to start the search.")
else:
    print("Search:", tuning_report["manifest"]["run_id"],
          "| status:", tuning_report["manifest"]["status"],
          "| completed trials:", len(tuning_report["records"]))
    # Plot actual completion order, rather than alphabetical artifact IDs.
    tuning_report["records"].sort(key=lambda row: json.loads(
        (Path(row.get("trial_dir", str(tuning_report["run_dir"] / row["stage"] / "trials" /
         f"{row['candidate_id']}-seed{row['seed']}"))) / "bundle/bundle.json").read_text()
    )["created_at_utc"])
    tuning_report["table"] = tuning_results_table(tuning_report["records"])
    plot_tuning_results(tuning_report)

The short search evaluates a small set of complete pipelines on one training
subset and one seed. Report incomplete candidate coverage and epoch-ceiling flags.
New fold-local CV/Bayesian comparisons and any final full-data fitting remain later
steps; the short search never scores the official test split.

## 7. Cross-validation experiments

The complete baseline runner is defined here. Each fitted pipeline uses only
its training fold, and completed held-out predictions are saved for paired
comparison. Its fixed schedule is the historical baseline recipe; the revised
validation-driven development search in section 6 has completed. A new tuned
outer-CV experiment still needs to evaluate the complete selection procedure.

### Cross-validation execution

The actual runner is defined below. It fits training-fold preprocessing and
models, saves held-out predictions, verifies reloads and assembles paired scores.
Defining these functions does not launch the large experiment.

In [ ]:
class SampledMemory:
    """Sample whole-process RSS every 20 ms; not allocator-only/isolated model RAM."""

    def __enter__(self):
        self.process = psutil.Process()
        self.peak_bytes = self.process.memory_info().rss
        self.stop = threading.Event()
        self.thread = threading.Thread(target=self._sample, daemon=True)
        self.thread.start()
        return self

    def _sample(self):
        while not self.stop.wait(0.02):
            self.peak_bytes = max(self.peak_bytes, self.process.memory_info().rss)

    def __exit__(self, *exception):
        self.peak_bytes = max(self.peak_bytes, self.process.memory_info().rss)
        self.stop.set()
        self.thread.join()

### CV persistence and planned systems

Completed fits are checked and reused; interrupted attempts remain separate.

In [ ]:
def _experiments_write(path: Path, value) -> None:
    """Commit a small state file atomically, including after an interruption."""
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)

In [ ]:
def _experiments_now() -> str:
    return datetime.now(timezone.utc).isoformat()

In [ ]:
def _experiments_hash_tree(directory: Path) -> dict[str, str]:
    return {str(p.relative_to(directory)): _features_sha256(p) for p in sorted(directory.rglob('*')) if p.is_file()}

In [ ]:
@contextmanager
def _experiments_lock(directory: Path):
    with (directory / '.lock').open('a') as stream:
        try:
            fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('Another process is already running this CV experiment.') from error
        try:
            yield
        finally:
            fcntl.flock(stream, fcntl.LOCK_UN)

In [ ]:
def planned_systems(config: dict[str, Any]) -> list[dict[str, Any]]:
    """Nine primary systems and two fixed baselines, with unique stable IDs."""
    if config['representations'] != list(REPRESENTATIONS) or config['classifiers'] != list(CLASSIFIERS):
        raise ValueError('CV requires the three current representations and three model families.')
    if config['status'] != 'frozen_after_pilot_before_reported_cv':
        raise ValueError('Use the pilot-frozen configuration for reported CV.')
    epochs = config['neural_training']['num_epochs']
    if not isinstance(epochs, int) or isinstance(epochs, bool) or epochs < 1:
        raise ValueError('Reported CV needs a positive frozen epoch count.')
    systems = [{'representation': representation, 'classifier': kind, 'input_mode': 'pair', 'primary': True} for representation in REPRESENTATIONS for kind in CLASSIFIERS]
    systems.append({'representation': 'unigram_counts', 'classifier': 'class_prior', 'input_mode': 'pair', 'primary': False})
    diagnostic = config['diagnostics']['hypothesis_only']
    systems.append({key: diagnostic[key] for key in ('representation', 'classifier', 'input_mode')} | {'primary': False})
    for system in systems:
        system['system_id'] = '__'.join((system[key] for key in ('representation', 'classifier', 'input_mode')))
    if len({s['system_id'] for s in systems}) != 11:
        raise ValueError('Expected eleven distinct primary/baseline systems.')
    return systems

In [ ]:
def validate_predictions(predictions: pd.DataFrame, rows: pd.DataFrame) -> dict[str, Any]:
    """Check exact held-out IDs/order/labels and score saved probability columns."""
    if not set([*PREDICTION_ROWS, 'prediction', 'stage', *PROBABILITY_COLUMNS]).issubset(predictions.columns):
        raise ValueError('Saved CV predictions have missing provenance/probability columns.')
    if not predictions[PREDICTION_ROWS].reset_index(drop=True).equals(rows[PREDICTION_ROWS].reset_index(drop=True)):
        raise ValueError('Saved CV predictions have wrong row order, IDs, groups or gold labels.')
    if predictions['example_id'].duplicated().any() or not predictions['stage'].eq('outer_cv_heldout').all():
        raise ValueError('CV predictions must contain each held-out example once with the correct stage.')
    if not predictions['source_split'].eq('train').all() or not predictions['partition'].eq('heldout').all():
        raise ValueError('Reported CV must use only held-out rows from the official training split.')
    probabilities = validate_probabilities(predictions[PROBABILITY_COLUMNS].to_numpy(), len(rows))
    if not np.array_equal(predictions['prediction'], probabilities.argmax(axis=1)):
        raise ValueError('Saved decisions disagree with the frozen argmax/tie convention.')
    return score_probabilities(rows['label'].to_numpy(), probabilities)

In [ ]:
def _experiments_load_unit(directory: Path, identity: dict[str, Any], rows: pd.DataFrame) -> dict[str, Any]:
    result = _experiments_read(directory / 'complete.json')
    if result['status'] != 'complete' or result['identity'] != identity:
        raise ValueError('Completed CV unit has a different fold/configuration/system identity.')
    attempt = directory / result['attempt']
    _experiments_verify(attempt, result['artifact_sha256'])
    scores = validate_predictions(pd.read_parquet(attempt / 'predictions.parquet'), rows)
    row = result['metrics']
    for name in ('accuracy', 'nll', 'macro_f1'):
        if not np.isclose(row[name], scores[name], rtol=0, atol=1e-12):
            raise ValueError('Saved CV metrics disagree with the checked held-out predictions.')
    return result

In [ ]:
def _experiments_fit_unit(directory, identity, system, features, heldout_text, config, feature_seconds):
    directory.mkdir(parents=True, exist_ok=True)
    attempt = directory / f'attempt_{time_ns()}'
    attempt.mkdir()
    state = {'status': 'running', 'started_at_utc': _experiments_now(), 'identity': identity}
    _experiments_write(attempt / 'state.json', state)
    started = perf_counter()
    try:
        representation, mode = (system['representation'], system['input_mode'])
        train, heldout = (features['rows']['train'], features['rows']['heldout'])
        X = select_input_features(features['matrices']['train'][representation], mode)
        Xheldout = select_input_features(features['matrices']['heldout'][representation], mode)
        with SampledMemory() as memory:
            classifier = fit_classifier(system['classifier'], X, train['label'].to_numpy(), config)
            evaluation_started = perf_counter()
            probabilities = classifier.predict_proba(Xheldout)
            scores = score_probabilities(heldout['label'].to_numpy(), probabilities)
            predictions = heldout[PREDICTION_ROWS].copy()
            predictions['prediction'] = probabilities.argmax(axis=1)
            for label, name in enumerate(PROBABILITY_COLUMNS):
                predictions[name] = probabilities[:, label]
            predictions['stage'] = 'outer_cv_heldout'
            predictions['run_id'] = identity['run_id']
            predictions['system_id'] = identity['system_id']
            predictions['seed'] = identity['seed']
            predictions['fold'] = identity['fold']
            validate_predictions(predictions, heldout)
            predictions.to_parquet(attempt / 'predictions.parquet', index=False)
            _experiments_write(attempt / 'scores.json', scores)
            save_prediction_bundle(attempt / 'bundle', classifier, features['encoder'], representation, input_mode=mode, context={**identity, 'stage': 'reported_cv', 'test_scored': False})
            loaded = load_prediction_bundle(attempt / 'bundle')
            reproduced = loaded.predict_proba(heldout_text['premise'].tolist(), heldout_text['hypothesis'].tolist())
            difference = float(np.max(np.abs(probabilities - reproduced)))
            if difference > 1e-06:
                raise ValueError('CV probabilities changed after raw-text bundle reload.')
            evaluation_seconds = perf_counter() - evaluation_started
            pd.DataFrame(classifier.history).to_csv(attempt / 'history.csv', index=False)
            _experiments_write(attempt / 'fit_info.json', classifier.fit_info)
            model_bytes = sum((p.stat().st_size for p in (attempt / 'bundle').iterdir() if p.suffix in ('.pt', '.joblib', '.npy')))
            row = {**identity, **system, 'training_rows': len(train), 'heldout_rows': len(heldout), 'model_seed': config['seeds']['model'], 'input_dim': X.shape[1], 'training_nonzeros': X.nnz, 'heldout_nonzeros': Xheldout.nnz, 'training_csr_bytes': X.data.nbytes + X.indices.nbytes + X.indptr.nbytes, 'parameters': classifier.parameter_count, 'model_bytes': model_bytes, 'accuracy': scores['accuracy'], 'nll': scores['nll'], 'macro_f1': scores['macro_f1'], 'training_seconds': classifier.fit_info['fit_seconds'], 'evaluation_reload_seconds': evaluation_seconds, 'feature_preparation_seconds': feature_seconds, 'epochs': classifier.fit_info.get('epochs_completed'), 'converged': classifier.fit_info.get('converged'), 'iterations': max(classifier.fit_info.get('iterations', [0])), 'convergence_warnings': classifier.fit_info.get('convergence_warnings', []), 'reload_max_abs_difference': difference, 'checkpoint': str(attempt / 'bundle'), 'predictions': str(attempt / 'predictions.parquet'), 'feature_id': features['manifest']['feature_id'], 'feature_manifest': str(features['manifest_path']), 'feature_manifest_sha256': _features_sha256(features['manifest_path'])}
        row['sampled_peak_process_rss_bytes'] = memory.peak_bytes
        row['unit_wall_seconds'] = perf_counter() - started
        _experiments_write(attempt / 'row.json', row)
        _experiments_write(attempt / 'state.json', {**state, 'status': 'complete', 'finished_at_utc': _experiments_now()})
        result = {'status': 'complete', 'identity': identity, 'attempt': attempt.name, 'metrics': row, 'artifact_sha256': _experiments_hash_tree(attempt)}
        _experiments_write(directory / 'complete.json', result)
        return result
    except BaseException as error:
        _experiments_write(attempt / 'state.json', {**state, 'status': 'interrupted' if isinstance(error, KeyboardInterrupt) else 'failed', 'finished_at_utc': _experiments_now(), 'error_type': type(error).__name__, 'error': str(error), 'traceback': traceback.format_exc()})
        raise

In [ ]:
def _experiments_prepare_features(config_path, protocol, seed, fold, verbose):
    try:
        return prepare_fold_features(config_path, seed=seed, fold=fold, verbose=verbose)
    except ValueError as error:
        match = re.fullmatch('Incomplete feature cache: (.+)\\. Existing files were not overwritten\\.', str(error))
        if not match:
            raise
        path = Path(match.group(1)).resolve()
        config = _experiments_read(REPO_ROOT / config_path)
        expected = REPO_ROOT / config['paths']['output_dir'] / 'features' / protocol['manifest']['protocol_id'] / f'seed_{seed}_fold_{fold}'
        if path.parent != expected.resolve() or not re.fullmatch('[0-9a-f]{16}', path.name):
            raise
        destination = path.parent / 'interrupted' / f'{path.name}_{time_ns()}'
        destination.parent.mkdir(exist_ok=True)
        path.rename(destination)
        if verbose:
            print(f'Preserved incomplete feature cache at {destination}; rebuilding this fold.', flush=True)
        return prepare_fold_features(config_path, seed=seed, fold=fold, verbose=verbose)

In [ ]:
def summarize_metrics(metrics: pd.DataFrame, systems, seeds, folds: int) -> pd.DataFrame:
    expected = {(system['system_id'], seed, fold) for system in systems for seed in seeds for fold in range(folds)}
    keys = list(zip(metrics['system_id'], metrics['seed'], metrics['fold']))
    if len(keys) != len(set(keys)) or set(keys) != expected:
        raise ValueError('A complete paired comparison needs every system on exactly the same saved folds.')
    summary = metrics.groupby(['system_id', 'representation', 'classifier', 'input_mode', 'primary'], as_index=False).agg(folds=('fold', 'size'), accuracy_mean=('accuracy', 'mean'), accuracy_std=('accuracy', 'std'), accuracy_min=('accuracy', 'min'), accuracy_max=('accuracy', 'max'), nll_mean=('nll', 'mean'), nll_std=('nll', 'std'), macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'), training_seconds_total=('training_seconds', 'sum'), training_seconds_mean=('training_seconds', 'mean'), parameters_min=('parameters', 'min'), parameters_max=('parameters', 'max'), input_dim_min=('input_dim', 'min'), input_dim_max=('input_dim', 'max'), model_bytes_mean=('model_bytes', 'mean'))
    return summary.sort_values(['primary', 'accuracy_mean', 'system_id'], ascending=[False, False, True]).reset_index(drop=True)

In [ ]:
def _experiments_plots(metrics, summary, curves, directory):
    names = summary['system_id'].tolist()
    labels = [name.replace('__pair', '').replace('__', ' / ') for name in names]
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
    jitter = np.random.default_rng(0).uniform(-0.12, 0.12, len(metrics))
    positions = metrics['system_id'].map({name: i for i, name in enumerate(names)}).to_numpy()
    for ax, score, title in zip(axes, ('accuracy', 'nll'), ('Held-out accuracy', 'Held-out NLL (nats)')):
        ax.scatter(metrics[score], positions + jitter, s=16, alpha=0.55, label='Saved fold scores')
        ax.scatter(summary[f'{score}_mean'], np.arange(len(names)), marker='D', color='black', s=28, label='Mean of folds')
        ax.set_xlabel(title)
        ax.grid(axis='x', alpha=0.25)
    axes[0].set_yticks(np.arange(len(names)), labels)
    axes[0].invert_yaxis()
    axes[1].legend(fontsize=8)
    fig.suptitle('Paired repeated grouped CV — overlapping folds are not independent')
    fig.tight_layout()
    fig.savefig(directory / 'fold_scores.png', dpi=160)
    plt.close(fig)
    fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
    for column, representation in enumerate(REPRESENTATIONS):
        for kind in CLASSIFIERS[1:]:
            rows = curves.loc[(curves['representation'] == representation) & (curves['classifier'] == kind)]
            grouped = rows.groupby('epoch')
            for line, metric in enumerate(('train_nll_online', 'train_accuracy_online')):
                means, stds = (grouped[metric].mean(), grouped[metric].std().fillna(0))
                axes[line, column].plot(means.index, means, marker='o', label=kind)
                axes[line, column].fill_between(means.index, means - stds, means + stds, alpha=0.15)
                axes[line, column].grid(alpha=0.2)
        axes[0, column].set_title(representation)
        axes[1, column].set_xlabel('Fixed training epoch')
    axes[0, 0].set_ylabel('Online training NLL')
    axes[1, 0].set_ylabel('Online training accuracy')
    axes[0, -1].legend(fontsize=8)
    fig.suptitle('Training histories: fold mean ± descriptive SD; no outer-fold checkpoint selection')
    fig.tight_layout()
    fig.savefig(directory / 'learning_curves.png', dpi=160)
    plt.close(fig)

In [ ]:
def run_cross_validation(config_path: str | Path=FROZEN_CONFIG, *, max_fits: int | None=None, verbose: bool=False):
    """Run/resume all fits; max_fits bounds new work while keeping status incomplete."""
    if max_fits is not None and (not isinstance(max_fits, int) or isinstance(max_fits, bool) or max_fits < 1):
        raise ValueError('max_fits must be a positive integer or None.')
    config_path = REPO_ROOT / config_path
    workspace = prepare_workspace(config_path)
    config = workspace['config']
    systems = planned_systems(config)
    protocol = prepare_evaluation_protocol(config_path, verbose=verbose)
    code_names = ('practical_1.ipynb',)
    specification = {'cv_version': CV_VERSION, 'configuration': config, 'configuration_sha256': _features_sha256(config_path), 'protocol_id': protocol['manifest']['protocol_id'], 'protocol_sha256': _features_sha256(protocol['manifest_path']), 'source_sha256': {'practical_1.ipynb': notebook_source_sha256()}, 'packages': workspace['manifest']['packages'], 'systems': systems, 'checkpoint_policy': 'last_fixed_epoch_no_outer_holdout_selection'}
    run_id = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:16]
    directory = workspace['paths']['output'] / 'cross_validation' / run_id
    directory.mkdir(parents=True, exist_ok=True)
    original_threads = torch.get_num_threads()
    torch.set_num_threads(config['neural_training']['num_threads'])
    try:
        with _experiments_lock(directory):
            return _experiments_run(directory, run_id, specification, workspace, protocol, systems, config_path, max_fits, verbose)
    finally:
        torch.set_num_threads(original_threads)

In [ ]:
def _experiments_run(directory, run_id, specification, workspace, protocol, systems, config_path, max_fits, verbose):
    started = perf_counter()
    config = workspace['config']
    marker = directory / 'manifest.json'
    pointer = workspace['paths']['manifests'] / 'cross_validation.json'
    if marker.is_file():
        previous = _experiments_read(marker)
        if previous['specification'] != specification:
            raise ValueError('Saved CV specification differs from this run.')
        if previous['status'] == 'complete':
            _experiments_write(pointer, {'run_id': run_id, 'manifest_path': str(marker)})
            return _experiments_load_report(marker)
    else:
        previous = {}
    expected = len(systems) * len(config['seeds']['cv_splits']) * config['cross_validation']['num_folds']
    manifest = {'stage': 'reported_cross_validation', 'status': 'running', 'run_id': run_id, 'created_at_utc': previous.get('created_at_utc', _experiments_now()), 'specification': specification, 'configuration_path': str(config_path), 'protocol_manifest': str(protocol['manifest_path']), 'expected_units': expected, 'completed_units': previous.get('completed_units', 0), 'repository': workspace['manifest']['repository'], 'hardware': workspace['manifest']['hardware'], 'python': workspace['manifest']['python'], 'execution_history': previous.get('execution_history', [])}
    invocation = {'started_at_utc': _experiments_now(), 'new_fits': 0, 'reused_fits': 0}
    manifest['execution_history'].append(invocation)
    _experiments_write(marker, manifest)
    _experiments_write(pointer, {'run_id': run_id, 'manifest_path': str(marker)})
    rows, curves, unit_hashes = ([], [], {})
    try:
        source = None
        for repetition, seed in enumerate(config['seeds']['cv_splits']):
            for fold in range(config['cross_validation']['num_folds']):
                heldout = protocol['assignments'].loc[protocol['assignments'][f'fold_seed_{seed}'].eq(fold), ROW_COLUMNS].reset_index(drop=True)
                heldout['partition'] = 'heldout'
                features, text = (None, None)
                for system in systems:
                    identity = {'run_id': run_id, 'protocol_id': specification['protocol_id'], 'config_sha256': specification['configuration_sha256'], 'dataset_revision': config['dataset']['revision'], 'system_id': system['system_id'], 'repetition': repetition, 'seed': seed, 'fold': fold}
                    unit = directory / 'units' / f'seed_{seed}' / f'fold_{fold}' / system['system_id']
                    complete = unit / 'complete.json'
                    if complete.is_file():
                        result = _experiments_load_unit(unit, identity, heldout)
                        invocation['reused_fits'] += 1
                    else:
                        if max_fits is not None and invocation['new_fits'] >= max_fits:
                            manifest['status'] = 'incomplete'
                            invocation.update({'finished_at_utc': _experiments_now(), 'wall_seconds': perf_counter() - started, 'status': 'bounded_stop'})
                            _experiments_write(marker, manifest)
                            return {'manifest': manifest, 'manifest_path': marker, 'run_dir': directory}
                        if features is None:
                            feature_started = perf_counter()
                            features = _experiments_prepare_features(config_path, protocol, seed, fold, verbose)
                            feature_seconds = perf_counter() - feature_started
                            if not features['rows']['heldout'][PREDICTION_ROWS].equals(heldout[PREDICTION_ROWS]):
                                raise ValueError('Feature row order disagrees with the frozen protocol.')
                            if source is None:
                                source = load_from_disk(protocol['manifest']['prepared_dataset'])['train']
                            text = source.select(heldout['clean_index'].tolist()).select_columns(['premise', 'hypothesis']).to_pandas()
                        if verbose:
                            print(f'CV {run_id}: fit {len(rows) + 1}/{expected}, seed {seed}, fold {fold}, {system['system_id']}', flush=True)
                        result = _experiments_fit_unit(unit, identity, system, features, text, config, feature_seconds)
                        invocation['new_fits'] += 1
                        if verbose:
                            r = result['metrics']
                            print(f'  accuracy={r['accuracy']:.4f}, NLL={r['nll']:.4f}, train={r['training_seconds']:.2f}s, converged={r['converged']}', flush=True)
                    rows.append(result['metrics'])
                    history_path = unit / result['attempt'] / 'history.csv'
                    if system['classifier'] in CLASSIFIERS[1:]:
                        curves.append(pd.read_csv(history_path).assign(**identity, representation=system['representation'], classifier=system['classifier']))
                    unit_hashes[str(complete.relative_to(directory))] = _features_sha256(complete)
                    manifest['completed_units'] = len(rows)
                    _experiments_write(marker, manifest)
                del features, text
                gc.collect()
        metrics = pd.DataFrame(rows).sort_values(['repetition', 'fold', 'system_id']).reset_index(drop=True)
        summary = summarize_metrics(metrics, systems, config['seeds']['cv_splits'], config['cross_validation']['num_folds'])
        histories = pd.concat(curves, ignore_index=True)
        metrics.to_csv(directory / 'fold_metrics.csv', index=False)
        summary.to_csv(directory / 'summary.csv', index=False)
        histories.to_csv(directory / 'learning_curves.csv', index=False)
        metrics.pivot(index=['repetition', 'seed', 'fold'], columns='system_id', values='accuracy').sort_index().to_csv(directory / 'paired_accuracy.csv')
        convergence = metrics.loc[metrics['classifier'].eq('logistic'), ['system_id', 'repetition', 'seed', 'fold', 'converged', 'iterations', 'convergence_warnings']]
        convergence.to_csv(directory / 'convergence.csv', index=False)
        _experiments_plots(metrics, summary, histories, directory)
        manifest.update({'status': 'complete', 'finished_at_utc': _experiments_now(), 'completed_units': len(metrics), 'primary_fits': int(metrics['primary'].sum()), 'baseline_fits': int((~metrics['primary']).sum()), 'nonconverged_logistic_fits': int(convergence['converged'].eq(False).sum()), 'checks': {'matching_complete_fold_grid': True, 'original_ids_and_gold_order_checked': True, 'finite_normalized_probabilities': True, 'raw_text_bundle_reloads_match': True, 'fixed_training_length_no_outer_selection': True, 'test_not_scored': True}, 'limitations': ['Fold means and SDs are descriptive; overlapping repeated folds are dependent.', 'Online training curves use changing weights/dropout, not end-of-epoch evaluation.', 'CV uses the fixed premise-group subset, not full-SNLI training; test remains unscored.']})
        invocation.update({'finished_at_utc': _experiments_now(), 'wall_seconds': perf_counter() - started, 'status': 'complete'})
        manifest['artifact_sha256'] = unit_hashes | {p.name: _features_sha256(p) for p in directory.iterdir() if p.suffix in ('.csv', '.png')}
        _experiments_write(marker, manifest)
        return _experiments_load_report(marker)
    except BaseException as error:
        invocation.update({'finished_at_utc': _experiments_now(), 'wall_seconds': perf_counter() - started, 'status': 'interrupted' if isinstance(error, KeyboardInterrupt) else 'failed', 'error_type': type(error).__name__, 'error': str(error)})
        manifest['status'] = invocation['status']
        _experiments_write(marker, manifest)
        raise

The following switch is explicit because a full CV run performs 165 fits.
Leaving it false reads the existing baseline later; setting it true runs the
notebook-defined implementation and creates a distinct source-identified run.
It uses the frozen baseline recipe. The joint tokenizer/model search has its
own switch in section 6; new tuned outer CV is a subsequent experiment.

When regenerating CV, also enable `RUN_BAYESIAN_ANALYSIS` in section 8 to
regenerate the matching analysis before reading it.

In [ ]:
RUN_BASELINE_CV = False
if RUN_BASELINE_CV:
    notebook_cv_report = run_cross_validation(FROZEN_CONFIG, verbose=True)

### Reading the baseline experiments

The following is the **completed baseline**, not the proposed tuned comparison.
All nine primary systems were fitted on the same 60,005-row premise-group subset,
using 15 paired partitions and three fixed neural epochs. The model seed was
0; `[0, 3, 5]` are split seeds. They do not measure initialization variability.

The report loader checks saved artifacts and predictions. No full CV training
runs when this notebook is opened or executed.

In [ ]:
assert cv_manifest["status"] == "complete"
assert cv_manifest["primary_fits"] == 135 and cv_manifest["baseline_fits"] == 30
assert all(cv_manifest["checks"].values())
print("Baseline CV:", cv_manifest["run_id"], "| completed fits:", cv_manifest["completed_units"])
primary_summary = cv_summary.loc[cv_summary["primary"]]
display((100 * primary_summary.pivot(index="representation", columns="classifier", values="accuracy_mean")).round(2))

### Accuracy and probability quality

The highest mean baseline accuracy is **62.75%** for indicators with two hidden
layers. The lowest mean NLL is **0.8466** for unigram counts with one hidden layer.
These criteria need not select the same system. Fold SDs are descriptive; they
are not independent error bars because training sets overlap.

In [ ]:
display(cv_summary[[
    "representation", "classifier", "input_mode", "accuracy_mean", "accuracy_std",
    "nll_mean", "macro_f1_mean",
]].round(4))
display(Image(filename=str(cv_report["figures"]["fold_scores"])))

### Learning curves

These are online training measurements with changing weights and dropout,
not end-of-epoch held-out scores. Lower training loss by itself is not evidence
that a longer schedule will improve validation performance.

In [ ]:
display(Image(filename=str(cv_report["figures"]["learning_curves"])))
assert len(cv_tables["learning_curves"]) == 90 * 3

### Inspecting predictions

Saved predictions retain original example IDs, gold labels and the three
probabilities. This gives us the material for error analysis without repeatedly
scoring the official test. The following rows illustrate the format, rather
than a selection of errors.

In [ ]:
example_predictions = pd.read_parquet(Path(cv_metrics.iloc[0]["predictions"]))
display(example_predictions[[
    "example_id", "label", "prediction", "p_entailment", "p_neutral", "p_contradiction",
]].head())
assert example_predictions["source_split"].eq("train").all()

## 8. Bayesian comparison and practical equivalence

Is an accuracy difference large enough to matter? The baseline uses the
preselected ROPE **±0.005 accuracy = ±0.5 percentage points**. It expresses our
practical judgement about accuracy, not a sampling error bar.

We compare matching `(repetition, seed, fold)` scores with the tutorial's
`BenavoliCorrelatedT` helper, explicitly using **`runs=3`**. Every difference is
**B−A**: positive favors B. The report contains all 18 comparisons that change
one factor, plus 10 separate prior/ablation comparisons.

In [ ]:
ANALYSIS_POLICY = REPO_ROOT / "part1/configs/snli_analysis.json"
KEYS = ["repetition", "seed", "fold"]
TABLES = ("paired_accuracy", "contrasts", "paired_differences", "posteriors", "rope_sensitivity")
FAMILIES = ("representation", "architecture", "class_prior", "hypothesis_ablation")
LABELS = {"indicators": "Indicators", "unigram_counts": "Unigrams", "bigram_counts": "Bigrams",
          "logistic": "LR", "mlp_one_hidden": "MLP1", "mlp_two_hidden": "MLP2"}

### The Bayesian correlated t model

The supplied tutorial wrapper is defined here. It delegates the statistical
distribution to the third-party baycomp/SciPy libraries.

In [ ]:
class BenavoliCorrelatedT:
    """
    A convenient wrapper around baycomp.

    Estimates the posterior mean difference (B-A) in performance between system A and system B.
    """

    def __init__(self, sys_a, sys_b, runs=1, names=['sys_a', 'sys_b']):
        self._ct = baycomp.CorrelatedTTest(np.array(sys_a), np.array(sys_b), rope=0.0, runs=runs)
        self._names = names

    @property
    def samples(self):
        """Sampled mean differences"""
        return self._ct.sample

    @property
    def mean_diff(self):
        """Posterior mean difference"""
        return self._ct.mean

    @property
    def std(self):
        """Posterior standard deviation"""
        return np.sqrt(self._ct.var)

    @property
    def df(self):
        """Posterior degrees of freedom"""
        return self._ct.df

    @property
    def effect_size(self):
        """
        Return mean difference divided by posterior std.
        This can be interpreted as improvement in "units of stds".
        There is no universal way to read this, 
         but some people like to think that < 0.2 indicates a minor effect
         more than 0.8 indicates a large effect
         and in between indicates moderate effect
        (Don't take these 'rules' too seriously)
        """
        return self.mean_diff / self.std

    def plot(self, rope_param, hdi=0.95, wider_hdi=0.999, num_points=1000, rope_factor=2.5):
        """
        Plot the posterior for the mean difference (B-A), the ROPE and the region probs (ie, A wins, Tie, B wins).
        Return fig and ax.
        
        rope_param: the ROPE is treated as the interval (-rope_param, rope_param)
        hdi: draw a high posterior interval of size 100*hdi% 
        wider_hdi: this is used to control the size of the plot
        num_points: how many evaluates of the posterior density
        rope_factor: this is also used to control the size of the plot, the larger this is
         the more we zoom out
        """
        assert wider_hdi > hdi, 'wider_hdi should be wider than hdi'
        t = st.t(self.df, loc=self.mean_diff, scale=self.std)
        fig, ax = plt.subplots()
        ci = t.interval(wider_hdi)
        x = np.linspace(min(ci[0], -rope_param * rope_factor), max(ci[1], rope_param * rope_factor), num_points)
        _ = ax.set_xlabel('Mean difference: $\\mu$')
        _ = ax.plot(x, t.pdf(x), label='$p(\\mu|\\mathcal{D})$')
        _ = ax.axvspan(-rope_param, rope_param, color='teal', alpha=0.25, label=f'ROPE ({rope_param})')
        transform = ax.get_xaxis_transform()
        _ = ax.text(-rope_param, 0.2, f'{t.cdf(-rope_param) * 100:.1f}% \nA>B ', ha='right', va='center', color='gray', transform=transform, fontsize=9)
        _ = ax.text(0, 0.5, f'{(t.cdf(rope_param) - t.cdf(-rope_param)) * 100:.1f}%\nTie', ha='center', va='center', color='gray', transform=transform, fontsize=9)
        _ = ax.text(rope_param, 0.7, f' {(1 - t.cdf(rope_param)) * 100:.1f}%\n B>A', ha='left', va='center', color='gray', transform=transform, fontsize=9)
        ci = t.interval(hdi)
        _ = ax.hlines(y=0.02, xmin=ci[0], xmax=ci[1], color='black', linewidth=2, label=f'{hdi * 100}% hdi')
        _ = ax.plot(t.median(), 0.02, 'd', color='black', label='median')
        _ = ax.set_title(f'A={self._names[0]} vs. B={self._names[1]}')
        _ = ax.legend()
        return (fig, ax)

    def sampled_probs(self, rope_size: float):
        """
        rope_size: defines a ROPE around 0 (ie, -rope_size, +rope_size)
        
        Return 
            - prob that mean diff is to the left of the ROPE (ie, A wins)
            - prob that mean diff is inside the ROPE (ie, tie)
            - prob that mean diff is to the right of the ROPE (ie, B wins)
        """
        p_A = np.mean(self.samples < -rope_size)
        p_tie = np.mean(np.abs(self.samples) < rope_size)
        p_B = np.mean(self.samples > rope_size)
        return (p_A, p_tie, p_B)

    def probs(self, rope_param: float):
        """
        rope_param: defines a ROPE around 0 (ie, -rope_param, +rope_param)
        
        Return 
            - prob that mean diff is to the left of the ROPE (ie, A wins)
            - prob that mean diff is inside the ROPE (ie, tie)
            - prob that mean diff is to the right of the ROPE (ie, B wins)
        """
        p_A = st.t.cdf(-rope_param, self.df, loc=self.mean_diff, scale=self.std)
        p_tie = st.t.cdf(rope_param, self.df, loc=self.mean_diff, scale=self.std) - st.t.cdf(-rope_param, self.df, loc=self.mean_diff, scale=self.std)
        p_B = 1 - st.t.cdf(rope_param, self.df, loc=self.mean_diff, scale=self.std)
        return (p_A, p_tie, p_B)

### Pairing, posterior computation and plots

Define the planned contrasts, validate partition keys, calculate corrected
posteriors and save labelled plots/tables.

In [ ]:
def _analysis_read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

In [ ]:
def _analysis_sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

In [ ]:
def _analysis_write(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + '\n', encoding='utf-8')

In [ ]:
def planned_contrasts(config):
    """Choose comparisons from design factors, never from observed rankings.

    A/B order follows the frozen lists. All 18 one-factor contrasts are primary;
    nine comparisons to the prior and one fixed input ablation provide context.
    """
    systems = planned_systems(config)
    rows = []

    def add(family, a, b, label):
        rows.append({'contrast_id': f'{family}_{sum((r['family'] == family for r in rows)) + 1:02d}', 'family': family, 'primary': family in FAMILIES[:2], 'system_a': a, 'system_b': b, 'label': label})
    for classifier in config['classifiers']:
        for a, b in combinations(config['representations'], 2):
            add('representation', f'{a}__{classifier}__pair', f'{b}__{classifier}__pair', f'{LABELS[classifier]}: {LABELS[a]} → {LABELS[b]}')
    for representation in config['representations']:
        for a, b in combinations(config['classifiers'], 2):
            add('architecture', f'{representation}__{a}__pair', f'{representation}__{b}__pair', f'{LABELS[representation]}: {LABELS[a]} → {LABELS[b]}')
    prior = next((s['system_id'] for s in systems if s['classifier'] == 'class_prior'))
    for system in systems:
        if system['primary']:
            add('class_prior', prior, system['system_id'], f'Prior → {LABELS[system['representation']]} {LABELS[system['classifier']]}')
    diagnostic = next((s for s in systems if s['input_mode'] == 'hypothesis_only'))
    add('hypothesis_ablation', f'{diagnostic['representation']}__{diagnostic['classifier']}__pair', diagnostic['system_id'], 'Unigram LR: pair → hypothesis only')
    return pd.DataFrame(rows)

In [ ]:
def paired_scores(metrics, config):
    """Validate the exact grid and align each system by partition identifiers."""
    systems = planned_systems(config)
    ids = [s['system_id'] for s in systems]
    seeds, folds = (config['seeds']['cv_splits'], config['cross_validation']['num_folds'])
    required = {*KEYS, 'system_id', 'accuracy', 'training_rows', 'heldout_rows', 'feature_id', 'run_id', 'protocol_id', 'config_sha256', 'model_seed', 'representation', 'classifier', 'input_mode', 'primary'}
    if not required.issubset(metrics.columns):
        raise ValueError('Missing CV pairing/provenance columns.')
    if metrics[list(required)].isna().any().any():
        raise ValueError('Missing values in CV pairing/provenance columns.')
    expected = pd.MultiIndex.from_tuples([(r, seed, fold, system) for r, seed in enumerate(seeds) for fold in range(folds) for system in ids], names=[*KEYS, 'system_id'])
    actual = pd.MultiIndex.from_frame(metrics[[*KEYS, 'system_id']])
    if actual.has_duplicates or len(actual) != len(expected) or set(actual) != set(expected):
        raise ValueError('Expected a complete paired grid with unique repetition/seed/fold/system keys.')
    accuracy = metrics['accuracy'].to_numpy(dtype=float)
    if not np.isfinite(accuracy).all() or np.any((accuracy < 0) | (accuracy > 1)):
        raise ValueError('Accuracies must be finite fractions in [0, 1], not percentages.')
    for field in ('run_id', 'protocol_id', 'config_sha256', 'model_seed'):
        if metrics[field].nunique() != 1:
            raise ValueError(f'Mixed CV provenance: {field}.')
    if not metrics['model_seed'].eq(config['seeds']['model']).all():
        raise ValueError('Model seed differs from the frozen design.')
    for system in systems:
        subset = metrics.loc[metrics['system_id'].eq(system['system_id'])]
        if any((not subset[key].eq(system[key]).all() for key in ('representation', 'classifier', 'input_mode', 'primary'))):
            raise ValueError('System metadata disagrees with its planned identity.')
    for field in ('feature_id', 'training_rows', 'heldout_rows'):
        if not metrics.groupby(KEYS)[field].nunique().eq(1).all():
            raise ValueError(f'Systems do not share paired fold {field}.')
    partitions = metrics.drop_duplicates(KEYS)
    if (partitions[['training_rows', 'heldout_rows']] <= 0).any().any():
        raise ValueError('Each fold needs positive training and held-out sizes.')
    totals = partitions['training_rows'] + partitions['heldout_rows']
    if totals.nunique() != 1 or not partitions.groupby('repetition')['heldout_rows'].sum().eq(totals.iloc[0]).all():
        raise ValueError('Each repetition must cover the same CV pool once.')
    index = pd.MultiIndex.from_tuples([(r, s, f) for r, s in enumerate(seeds) for f in range(folds)], names=KEYS)
    return metrics.pivot(index=KEYS, columns='system_id', values='accuracy').reindex(index=index, columns=ids).reset_index()

In [ ]:
def correlated_posterior(a, b, *, runs, rope, credible_mass=0.95):
    """Analytic posterior for mean accuracy B−A, with repeated-CV correction.

    The tutorial wrapper's ``std`` is Student scale, not distribution SD.
    Constant differences are a point mass; do not call a zero-scale t CDF.
    """
    a, b = (np.asarray(a, dtype=float), np.asarray(b, dtype=float))
    if a.ndim != 1 or a.shape != b.shape or len(a) < 4:
        raise ValueError('Need matching one-dimensional score arrays with at least four folds.')
    if not isinstance(runs, int) or isinstance(runs, bool) or runs < 1 or len(a) % runs or (len(a) // runs < 2):
        raise ValueError('runs must specify complete repetitions with at least two folds each.')
    if not np.isfinite(a).all() or not np.isfinite(b).all() or np.any((a < 0) | (a > 1) | (b < 0) | (b > 1)):
        raise ValueError('Scores must be finite accuracy fractions in [0, 1].')
    if not np.isfinite(rope) or rope <= 0 or (not 0 < credible_mass < 1):
        raise ValueError('Need positive finite ROPE and credible mass strictly between zero and one.')
    posterior = BenavoliCorrelatedT(a, b, runs=runs, names=['A', 'B'])
    mean, scale, df = (float(posterior.mean_diff), float(posterior.std), int(posterior.df))
    if scale == 0:
        pa, pb = (float(mean < -rope), float(mean > rope))
        pe, low, high = (1 - pa - pb, mean, mean)
    else:
        distribution = t(df, loc=mean, scale=scale)
        pa, pb = (float(distribution.cdf(-rope)), float(distribution.sf(rope)))
        pe = float(distribution.sf(-rope) - distribution.sf(rope) if -rope > mean else distribution.cdf(rope) - distribution.cdf(-rope))
        low, high = map(float, distribution.interval(credible_mass))
        if not np.allclose([pa, pe, pb], posterior.probs(rope), rtol=0, atol=1e-12):
            raise ValueError('Analytic region probabilities disagree with the supplied tutorial helper.')
    probabilities = np.array([pa, pe, pb])
    if not np.isfinite(probabilities).all() or np.any(probabilities < 0) or (not np.isclose(probabilities.sum(), 1, rtol=0, atol=1e-12)):
        raise ValueError('Invalid posterior region probabilities.')
    n, k = (len(a), len(a) // runs)
    return {'n_scores': n, 'runs': runs, 'folds_per_run': k, 'df': df, 'rope_half_width': float(rope), 'credible_mass': float(credible_mass), 'mean_difference': mean, 'mean_difference_pp': mean * 100, 'sample_difference_variance': float(np.var(b - a, ddof=1)), 'correction_factor': 1 / n + 1 / (k - 1), 'student_scale': scale, 'posterior_std': scale * np.sqrt(df / (df - 2)), 'ci_low': low, 'ci_high': high, 'ci_low_pp': low * 100, 'ci_high_pp': high * 100, 'p_a_better': pa, 'p_equivalent': pe, 'p_b_better': pb, 'point_mass': scale == 0}

In [ ]:
def _analysis_plot_posterior(row, path):
    rope, mean, scale = (row['rope_half_width'] * 100, row['mean_difference_pp'], row['student_scale'] * 100)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.axvspan(-rope, rope, color='teal', alpha=0.18, label=f'ROPE ±{rope:g} pp')
    if row['point_mass']:
        ax.axvline(mean, color='navy', label='Point mass')
        ax.set_xlim(min(-rope * 2, mean - rope), max(rope * 2, mean + rope))
    else:
        distribution = t(row['df'], loc=mean, scale=scale)
        lo, hi = distribution.interval(0.999)
        x = np.linspace(min(lo, -rope * 1.5), max(hi, rope * 1.5), 1400)
        ax.plot(x, distribution.pdf(x), color='navy', label='Posterior density')
        ax.set_xlim(x[0], x[-1])
    ax.axvline(0, color='gray', linewidth=0.7)
    ax.plot([row['ci_low_pp'], row['ci_high_pp']], [0.035, 0.035], color='black', lw=3, transform=ax.get_xaxis_transform(), label=f'{row['credible_mass']:.0%} credible interval')
    ax.set(title=row['label'], xlabel='Mean accuracy difference B − A (percentage points)', ylabel='Density per percentage point')
    fig.text(0.5, 0.01, f'A: {row['system_a']}\nB: {row['system_b']}\nP(A better)={row['p_a_better']:.2%}   P(equivalent)={row['p_equivalent']:.2%}   P(B better)={row['p_b_better']:.2%}', ha='center', fontsize=9)
    ax.legend(loc='upper right', fontsize=8)
    fig.tight_layout(rect=(0, 0.15, 1, 1))
    fig.savefig(path, dpi=160)
    plt.close(fig)

In [ ]:
def _analysis_plot_overview(posteriors, path):
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    for ax, family in zip(axes.flat, FAMILIES):
        subset = posteriors.loc[posteriors['family'].eq(family)].reset_index(drop=True)
        y = np.arange(len(subset))
        rope = subset['rope_half_width'].iloc[0] * 100
        ax.axvspan(-rope, rope, color='teal', alpha=0.18)
        ax.axvline(0, color='gray', lw=0.7)
        mean = subset['mean_difference_pp'].to_numpy()
        ax.errorbar(mean, y, xerr=np.stack([mean - subset['ci_low_pp'], subset['ci_high_pp'] - mean]), fmt='o', markersize=4, capsize=3, color='navy')
        ax.set_yticks(y, subset['label'], fontsize=9)
        ax.invert_yaxis()
        ax.set_title(family.replace('_', ' ').title())
        ax.set_xlabel('B − A accuracy (percentage points)')
        ax.grid(axis='x', alpha=0.2)
        if len(subset) == 1:
            ax.set_ylim(0.5, -0.5)
    mass = posteriors['credible_mass'].iloc[0]
    rope = posteriors['rope_half_width'].iloc[0] * 100
    fig.suptitle(f'Paired Bayesian comparisons: mean and {mass:.0%} credible interval\nA → B follows each row label; shaded ROPE ±{rope:g} pp; panels have separate x scales', fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.savefig(path, dpi=170)
    plt.close(fig)

In [ ]:
def _analysis_context(config_path, policy_path):
    config_path, policy_path = (REPO_ROOT / config_path, REPO_ROOT / policy_path)
    config, policy = (_analysis_read(config_path), _analysis_read(policy_path))
    if policy['method'] != 'tutorial_BenavoliCorrelatedT' or policy['difference'] != 'B_minus_A' or policy['contrast_policy'] != 'all_one_factor_pairs_plus_prior_and_fixed_hypothesis_ablation':
        raise ValueError('Unsupported analysis policy.')
    mass = policy['credible_mass']
    widths = policy['sensitivity_rope_half_widths']
    rope = config['evaluation']['rope_half_width']
    if not 0 < mass < 1 or rope <= 0 or (not np.isfinite(rope)) or (not widths) or any((not np.isfinite(w) or w <= 0 for w in widths)):
        raise ValueError('Invalid credible mass or ROPE policy.')
    report = load_cv_report(config_path)
    metrics = report['tables']['fold_metrics']
    paired = paired_scores(metrics, config)
    expected = paired.set_index(KEYS)
    saved_paired = report['tables']['paired_accuracy']
    existing = saved_paired.set_index(KEYS)
    if existing.index.has_duplicates or len(existing) != len(expected) or set(existing.index) != set(expected.index):
        raise ValueError('Saved paired accuracy table has different partition keys.')
    existing = existing.reindex(expected.index)
    if set(existing.columns) != set(expected.columns) or not np.allclose(existing[expected.columns], expected, rtol=0, atol=1e-14):
        raise ValueError('Saved paired accuracy table disagrees with fold metrics.')
    for key, value in (('run_id', report['manifest']['run_id']), ('protocol_id', report['manifest']['specification']['protocol_id']), ('config_sha256', _analysis_sha(config_path))):
        if not metrics[key].eq(value).all():
            raise ValueError(f'Fold metrics disagree with the CV manifest: {key}.')
    spec = {'analysis_version': 'v1', 'source_run_id': report['manifest']['run_id'], 'source_cv_manifest': str(report['manifest_path']), 'source_cv_manifest_sha256': _analysis_sha(report['manifest_path']), 'source_tables_sha256': {name: _analysis_sha(report['run_dir'] / f'{name}.csv') for name in ('fold_metrics', 'paired_accuracy')}, 'configuration_sha256': _analysis_sha(config_path), 'policy_sha256': _analysis_sha(policy_path), 'policy': policy, 'primary_rope_half_width': rope, 'source_sha256': {'practical_1.ipynb': notebook_source_sha256(), 'baycomp/single.py': _analysis_sha(Path(baycomp.single.__file__))}, 'versions': {name: version(name) for name in ('baycomp', 'scipy', 'numpy', 'pandas', 'matplotlib')}}
    identity = hashlib.sha256(json.dumps(spec, sort_keys=True, allow_nan=False).encode()).hexdigest()[:16]
    output = REPO_ROOT / config['paths']['output_dir']
    return (config, policy, report, paired, spec, identity, output)

In [ ]:
def _analysis_load(directory, spec):
    manifest = _analysis_read(directory / 'manifest.json')
    if manifest['status'] != 'complete' or manifest['specification'] != spec:
        raise ValueError('Analysis is incomplete or uses different source results/code/policy. Run the analysis command.')
    for name, expected in manifest['artifact_sha256'].items():
        path = directory / name
        if not path.is_file() or _analysis_sha(path) != expected:
            raise ValueError(f'Analysis artifact changed or is missing: {path}')
    return {'manifest': manifest, 'manifest_path': directory / 'manifest.json', 'analysis_dir': directory, 'tables': {name: pd.read_csv(directory / f'{name}.csv') for name in TABLES}, 'figures': {'overview': directory / 'posterior_overview.png'}, 'posterior_figures': {row['contrast_id']: directory / 'posteriors' / f'{row['contrast_id']}.png' for row in manifest['contrasts']}}

In [ ]:
def run_bayesian_analysis(config_path=FROZEN_CONFIG, policy_path=ANALYSIS_POLICY):
    """Create immutable tables/plots, or check/reuse an identical completed run."""
    config, policy, cv, paired, spec, identity, output = _analysis_context(config_path, policy_path)
    root = output / 'analysis'
    root.mkdir(parents=True, exist_ok=True)
    directory = root / identity
    contrasts = planned_contrasts(config)
    if not directory.exists():
        with tempfile.TemporaryDirectory(prefix=f'.{identity}_', dir=root) as temporary:
            stage = Path(temporary)
            (stage / 'posteriors').mkdir()
            rows, sensitivity, differences = ([], [], [])
            runs = len(config['seeds']['cv_splits'])
            rope = config['evaluation']['rope_half_width']
            for contrast in contrasts.to_dict('records'):
                a, b = (paired[contrast['system_a']].to_numpy(), paired[contrast['system_b']].to_numpy())
                result = contrast | correlated_posterior(a, b, runs=runs, rope=rope, credible_mass=policy['credible_mass'])
                rows.append(result)
                for width in sorted(set([rope, *policy['sensitivity_rope_half_widths']])):
                    sensitivity.append(contrast | correlated_posterior(a, b, runs=runs, rope=width, credible_mass=policy['credible_mass']) | {'primary_rope': width == rope})
                difference = paired[KEYS].copy()
                difference['contrast_id'] = contrast['contrast_id']
                difference['accuracy_a'], difference['accuracy_b'], difference['difference_b_minus_a'] = (a, b, b - a)
                differences.append(difference)
                _analysis_plot_posterior(result, stage / 'posteriors' / f'{contrast['contrast_id']}.png')
            tables = {'paired_accuracy': paired, 'contrasts': contrasts, 'paired_differences': pd.concat(differences, ignore_index=True), 'posteriors': pd.DataFrame(rows), 'rope_sensitivity': pd.DataFrame(sensitivity)}
            for name, table in tables.items():
                table.to_csv(stage / f'{name}.csv', index=False)
            _analysis_plot_overview(tables['posteriors'], stage / 'posterior_overview.png')
            manifest = {'stage': 'bayesian_paired_cv_analysis', 'status': 'complete', 'analysis_id': identity, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'specification': spec, 'contrasts': contrasts.to_dict('records'), 'primary_contrasts': int(contrasts['primary'].sum()), 'context_contrasts': int((~contrasts['primary']).sum()), 'checks': {'source_complete_cv_checked': True, 'identical_fold_order': True, 'runs_equals_cv_repetitions': True, 'original_rope_preserved': True, 'test_not_scored': True}, 'limitations': ['Correlated-normal paired-score model with the tutorial/baycomp matching-prior posterior.', 'Approximate equal-fold correction: scale squared = sample variance * (1/N + 1/(K-1)).', 'Single corpus, fixed group subset, hyperparameters and model seed; not initialization or cross-corpus uncertainty.', 'Separate conditional pairwise probabilities, not a joint ranking or probability of being globally best.'], 'artifact_sha256': {str(p.relative_to(stage)): _analysis_sha(p) for p in sorted(stage.rglob('*')) if p.is_file()}}
            _analysis_write(stage / 'manifest.json', manifest)
            stage.rename(directory)
    result = _analysis_load(directory, spec)
    pointer = output / 'manifests/bayesian_analysis.json'
    pointer.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', dir=pointer.parent, delete=False, encoding='utf-8') as stream:
        json.dump({'analysis_id': identity, 'manifest_path': str(result['manifest_path'])}, stream, indent=2)
        temporary_pointer = Path(stream.name)
    temporary_pointer.replace(pointer)
    return result

In [ ]:
def load_analysis_report(config_path=FROZEN_CONFIG, policy_path=ANALYSIS_POLICY):
    """Read a checked report and independently recompute its posteriors here.

    Historical reports retain their original recorded Python-source identity;
    arithmetic is checked against the definitions executed in this notebook.
    """
    config_path, policy_path = REPO_ROOT / config_path, REPO_ROOT / policy_path
    configuration = _analysis_read(config_path)
    policy = _analysis_read(policy_path)
    cv = load_cv_report(config_path)
    pointer = REPO_ROOT / configuration["paths"]["output_dir"] / "manifests/bayesian_analysis.json"
    record = _analysis_read(pointer)
    marker = Path(record["manifest_path"])
    saved = _analysis_read(marker)
    spec = saved["specification"]
    expected = {
        "source_run_id": cv["manifest"]["run_id"],
        "source_cv_manifest_sha256": _analysis_sha(cv["manifest_path"]),
        "configuration_sha256": _analysis_sha(config_path),
        "policy_sha256": _analysis_sha(policy_path),
        "policy": policy,
        "primary_rope_half_width": configuration["evaluation"]["rope_half_width"],
    }
    if saved["analysis_id"] != record["analysis_id"] or any(spec[key] != value for key, value in expected.items()):
        raise ValueError("Analysis source CV, configuration or policy changed.")
    for name, digest in spec["source_tables_sha256"].items():
        if _analysis_sha(cv["run_dir"] / f"{name}.csv") != digest:
            raise ValueError("Analysis input scores changed.")
    result = _analysis_load(marker.parent, spec)
    paired = paired_scores(cv["tables"]["fold_metrics"], configuration)
    saved_pairs = result["tables"]["paired_accuracy"]
    pd.testing.assert_frame_equal(paired, saved_pairs[paired.columns], check_dtype=False,
                                  check_names=False, rtol=0, atol=1e-14)
    contrast_plan = planned_contrasts(configuration)
    pd.testing.assert_frame_equal(contrast_plan, result["tables"]["contrasts"], check_dtype=False)
    checked_fields = ["mean_difference", "student_scale", "posterior_std", "ci_low", "ci_high",
                      "p_a_better", "p_equivalent", "p_b_better"]
    for contrast in result["tables"]["posteriors"].to_dict("records"):
        recomputed = correlated_posterior(
            paired[contrast["system_a"]].to_numpy(), paired[contrast["system_b"]].to_numpy(),
            runs=len(configuration["seeds"]["cv_splits"]),
            rope=configuration["evaluation"]["rope_half_width"], credible_mass=policy["credible_mass"],
        )
        np.testing.assert_allclose([recomputed[k] for k in checked_fields],
                                   [contrast[k] for k in checked_fields], rtol=0, atol=1e-12)
    return result

The notebook can regenerate the analysis with the functions above. Run All
keeps the large-run switches off, loads the completed baseline and recomputes
all 28 posterior means, intervals and region probabilities for agreement.

In [ ]:
RUN_BAYESIAN_ANALYSIS = False
if RUN_BAYESIAN_ANALYSIS:
    notebook_bayesian_report = run_bayesian_analysis()

In [ ]:
bayesian_report = load_analysis_report()
bayesian_manifest, bayesian_tables = bayesian_report["manifest"], bayesian_report["tables"]
posteriors = bayesian_tables["posteriors"]
analysis_pairs = bayesian_tables["paired_accuracy"]
assert bayesian_manifest["specification"]["source_run_id"] == cv_manifest["run_id"]
assert len(analysis_pairs) == 15 and len(posteriors) == 28
assert posteriors["runs"].eq(3).all() and posteriors["rope_half_width"].eq(.005).all()
display(analysis_pairs.iloc[:5, :6])

### Posterior probabilities

For $d_i=\mathrm{accuracy}_{B,i}-\mathrm{accuracy}_{A,i}$, the implemented posterior
has mean $\bar d$, 14 degrees of freedom, and Student scale
$s_d\sqrt{1/15+1/4}$, where $s_d$ is the paired sample SD. Its scale squared is
4.75 times the naive independent-score value. This approximately accounts for
overlapping folds; it does not make them independent.

We report three probabilities: **A better**, **inside the ROPE**, and **B better**.
The bars below are 95% equal-tailed credible intervals for the mean difference,
conditional on the model. The tutorial helper's `std` is Student scale; the
actual posterior SD is `scale * sqrt(14/12)`. Both are recorded explicitly.

In [ ]:
probability_columns = ["p_a_better", "p_equivalent", "p_b_better"]
posterior_columns = ["label", "mean_difference_pp", "ci_low_pp", "ci_high_pp", *probability_columns]
assert np.allclose(posteriors[probability_columns].sum(axis=1), 1, rtol=0, atol=1e-12)
for family in ("representation", "architecture"):
    print(family.title())
    display(posteriors.loc[posteriors["family"].eq(family), posterior_columns].round(6))

In [ ]:
display(Image(filename=str(bayesian_report["figures"]["overview"])))

Indicators and unigrams have high probability of practical equivalence at each
architecture. Pure bigrams lose about **7–8 percentage points** under the baseline
settings. MLPs improve on logistic, but the benefit of an extra hidden layer is
less decisive. These observations concern the tested recipes, not the best
possible configurations in each family.

For example, unigram MLP2−MLP1 has mean **+0.266 pp** and **95.97%** probability
inside the ROPE. A slightly larger mean does not automatically justify more
complexity. The following curve makes that judgement visible.

In [ ]:
display(Image(filename=str(bayesian_report["posterior_figures"]["architecture_06"])))

### Hypothesis-only context and sensitivity

For unigram logistic, hypothesis-only minus pair input is **+0.258 pp**, with
**97.72%** probability inside the ROPE. This is an input ablation using shared
pair-trained preprocessing. It does not establish that premises are unnecessary
for NLI. Annotation cues and the model's limited interactions are hypotheses to
investigate through held-out errors.

In [ ]:
display(posteriors.loc[~posteriors["primary"], posterior_columns].round(6))
rope_sensitivity = bayesian_tables["rope_sensitivity"]
display(rope_sensitivity.loc[
    rope_sensitivity["contrast_id"].eq("architecture_06"),
    ["rope_half_width", "primary_rope", *probability_columns],
].round(6))
assert len(rope_sensitivity) == 84

Changing the ROPE changes the practical judgement, not the posterior. The
original half-width remains 0.005; 0.0025 and 0.01 are sensitivity checks.

The common-correlation, correlated-normal model and matching-prior posterior
are approximations. Grouped fold sizes differ slightly, accuracy is bounded,
and all scores come from one corpus/subset with fixed settings. These are
pairwise probabilities, not a joint ranking or a probability of being globally
best. As HC2a PDF page 47 emphasizes, fixing a seed records a run without removing
experimental variation or establishing independent replication.

## 9. Error analysis and choosing a system

**Pending.** The next tuning stage comes first. We then need approximately 30–50
traceable held-out errors, representative confusion matrices and examples of
shared failures/disagreements. Questions to investigate include:

- Are negation and neutral/contradiction distinctions handled correctly?
- Does high lexical overlap cause misleading predictions?
- When do retained bigrams distinguish role/order changes, and when are they lost?
- Does the hypothesis-only model reveal label cues unrelated to the premise?

Selection should combine tuned CV evidence, NLL, Bayesian uncertainty, simplicity,
cost and the linguistic information our features preserve or discard.

## 10. Final refitting and test evaluation

**Pending.** Freeze the selected pipeline and training schedule, refit on the
eligible train-plus-validation pool, and evaluate once on the official test.
Save probabilities, accuracy, NLL, macro-F1 and class-level errors. Neither this
notebook's demonstrations nor the baseline results are official test scores.

## 11. Findings and oral preparation

The baseline teaches us that simple token evidence is useful, local bigrams
alone lose important evidence under this inventory, and more depth need not
produce a practically meaningful gain. The effects of joint tokenizer/model
optimization remain an open experiment.

For the oral assessment, explain the complete path from a sentence pair to a
sparse vector, logits, probabilities, a decision and an evaluation score. Connect
those choices to [COURSE_METHODS.md](COURSE_METHODS.md). Final design conclusions
and the compact backup slides remain pending.

## Appendix A. Reproduction and implementation checks

The code used by this notebook is defined in its cells. The existing Python
modules remain the historical command-line implementation. Full
commands and environment instructions are in [README.md](README.md); measured
results and limitations are in [PROJECT1_FINDINGS.md](PROJECT1_FINDINGS.md).
The existing baseline can be reproduced from the repository root with:

```bash
python -m part1.snli_pilot
python -m part1.snli_experiments
python -m part1.snli_analysis
```

The first command recreates the historical baseline's development run. The
proposed joint search in section 6 has no runner yet. Notebook Run All performs
small teaching/software checks and reads saved experiment reports.

### Environment and outputs

In [ ]:
print("Python version:", manifest["python"]["version"])
print("Configured device:", manifest["hardware"]["configured_device"])
print("Platform:", manifest["hardware"]["platform"])
print("Logical CPUs:", manifest["hardware"]["logical_cpu_count"])
print("Git commit:", manifest["repository"]["commit"])
print("Working tree dirty:", manifest["repository"]["dirty"])
print("Config SHA-256:", manifest["config_sha256"])

core_packages = [
    "numpy", "scipy", "pandas", "matplotlib", "scikit-learn",
    "torch", "datasets", "tokenizers", "baycomp", "ipykernel",
]
display(pd.DataFrame(
    [(name, manifest["packages"][name]) for name in core_packages],
    columns=["Package", "Installed version"],
))

In [ ]:
display(pd.DataFrame(
    [(name, str(path.relative_to(REPO_ROOT))) for name, path in paths.items()],
    columns=["Artifact group", "Repository-relative directory"],
))
print("Setup manifest:", manifest["artifacts"]["setup_manifest"])
print("Current package snapshot:", manifest["artifacts"]["environment"])
print("Dataset loaded during workspace setup:", manifest["dataset_loaded"])

### Feature transformations and reloads

Empty input, unknown tokens, role swaps and serialization should behave
predictably. A transformation must preserve the fitted vocabulary/inventory.

In [ ]:
before_transform = pair_encoder.tokenizer.to_str()
unknown_examples = ["", "  ", "🦄"]
unknown_vectors = pair_encoder.transform(unknown_examples, ["", "", ""], "unigram_counts")
unknown_id = pair_encoder.tokenizer.token_to_id("[UNK]")
display(pd.DataFrame([
    {"premise": repr(text), "nonzero_coordinates": unknown_vectors.getrow(i).nnz,
     "unknown_count": float(unknown_vectors[i, unknown_id])}
    for i, text in enumerate(unknown_examples)
]))
assert unknown_vectors[:2].nnz == 0

original = pair_encoder.transform(["the dog dog"], ["the man"], "unigram_counts")
swapped = pair_encoder.transform(["the man"], ["the dog dog"], "unigram_counts")
V = pair_encoder.vocab_size
assert (original[:, :V] != swapped[:, V:]).nnz == 0
assert (original[:, V:] != swapped[:, :V]).nnz == 0
assert (original != swapped).nnz > 0
assert before_transform == pair_encoder.tokenizer.to_str()

restored_encoder = PairFeatureEncoder.load(fold_features["feature_dir"])
for name in REPRESENTATIONS:
    first = pair_encoder.transform(["the dog dog"], ["the man"], name)
    restored = restored_encoder.transform(["the dog dog"], ["the man"], name)
    assert (first != restored).nnz == 0
print("Empty inputs, role order, fixed vocabulary, and exact encoder reload verified.")

### Synthetic model checks

These small artificial examples verify all nine model/representation
combinations, valid probabilities and complete prediction-bundle reloads.
Their scores are software checks, not SNLI accuracy results.

In [ ]:
model_smoke = run_model_smoke()
display(model_smoke["checks"])
assert model_smoke["checks"]["probabilities_valid"].all()
assert model_smoke["checks"]["finite_nll"].all()
assert model_smoke["checks"]["reload_max_abs_difference"].le(1e-7).all()

In [ ]:
hypothesis_key = "unigram_counts__logistic__hypothesis_only"
hypothesis_bundle = load_prediction_bundle(model_smoke["bundles"][hypothesis_key])
first = hypothesis_bundle.predict_proba(["item red", "item green"], ["red", "green"])
changed_premise = hypothesis_bundle.predict_proba(["unrelated premise", "🦄"], ["red", "green"])
np.testing.assert_array_equal(first, changed_premise)
prior_bundle = load_prediction_bundle(model_smoke["bundles"]["unigram_counts__class_prior__pair"])
np.testing.assert_allclose(prior_bundle.classifier.estimator, [1/3, 1/3, 1/3])
print("Hypothesis-only predictions ignore changed premises; synthetic class priors verified.")

### Pairing and convergence checks

Partition IDs, shared per-fold preprocessing, solver convergence and reload
agreement make the baseline comparison reviewable. The neural history check
below belongs to the original fixed-three-epoch experiment.

In [ ]:
paired_accuracy = cv_tables["paired_accuracy"]
assert len(paired_accuracy) == 15
assert paired_accuracy.shape[1] == 14  # Three partition identifiers plus eleven systems.
assert not paired_accuracy.isna().any().any()
assert cv_metrics.groupby("system_id").size().eq(15).all()
assert cv_metrics.groupby(["seed", "fold"])["feature_id"].nunique().eq(1).all()
assert cv_metrics.groupby(["seed", "fold"])["heldout_rows"].nunique().eq(1).all()
assert cv_metrics["reload_max_abs_difference"].le(1e-6).all()
neural_metrics = cv_metrics.loc[cv_metrics["classifier"].isin(["mlp_one_hidden", "mlp_two_hidden"])]
assert neural_metrics["epochs"].eq(3).all()
print("Paired rows:")
display(paired_accuracy[["repetition", "seed", "fold"]])
print("Logistic convergence by system:")
display(cv_tables["convergence"].groupby("system_id").agg(
    fits=("converged", "size"), converged=("converged", "sum"),
    min_iterations=("iterations", "min"), max_iterations=("iterations", "max"),
))
print("Maximum full held-out raw-text reload difference:", cv_metrics["reload_max_abs_difference"].max())

In [ ]:
example_artifact = Path(cv_metrics.iloc[0]["predictions"])
example_predictions = pd.read_parquet(example_artifact)
display(example_predictions[[
    "example_id", "source_split", "label", "prediction", "p_entailment", "p_neutral", "p_contradiction",
]].head())
assert example_predictions["source_split"].eq("train").all()
print("Prediction artifact:", example_artifact)
print("Distinct fold-specific feature caches:", cv_metrics["feature_id"].nunique())
print("Total model training seconds:", cv_metrics["training_seconds"].sum())
print("Maximum sampled process RSS (MiB):", cv_metrics["sampled_peak_process_rss_bytes"].max() / 2**20)
print("Official test scored:", not cv_manifest["checks"]["test_not_scored"])
assert cv_metrics["feature_id"].nunique() == 15

## Appendix B. Historical development run

The original small pilot checked runtime and selected a common schedule; it
was **not comprehensive hyperparameter optimization**. We preserve it to explain
the baseline's origin. It used 6,001 training and 1,202 validation examples,
selected 3 from epoch candidates `[3, 5, 8, 12]`, and adjusted solver limits.
It did not search vocabulary size, `C`, learning rate, widths or regularization.
The next experiment in section 6 replaces this selection procedure.

### Historical pilot report reader

Only the historical reader is needed here. Its code is defined below; no
pilot is trained by this notebook.

In [ ]:
def _pilot_read(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

In [ ]:
def _pilot_verify(directory: Path, manifest: dict[str, Any]) -> None:
    for name, digest in manifest['artifact_sha256'].items():
        path = directory / name
        if not path.is_file() or _features_sha256(path) != digest:
            raise ValueError(f'Pilot artifact changed or is missing: {path}')

In [ ]:
def _pilot_load_report(marker: Path) -> dict[str, Any]:
    manifest = _pilot_read(marker)
    _pilot_verify(marker.parent, manifest)
    frozen = manifest['frozen_configuration']
    if _features_sha256(Path(frozen['path'])) != frozen['sha256']:
        raise ValueError('The frozen pilot-selected configuration changed.')
    directory = marker.parent
    names = ('metrics', 'curves', 'epoch_selection', 'runtime_projection', 'class_counts', 'matrix_summary', 'comparison_pool_summary', 'comparison_fold_summary')
    return {'manifest': manifest, 'manifest_path': marker, 'pilot_dir': directory, 'frozen_config': _pilot_read(Path(frozen['path'])), 'tables': {name: pd.read_csv(directory / f'{name}.csv') for name in names}, 'learning_curve_figure': directory / 'learning_curves.png'}

In [ ]:
def load_pilot_report(config_path: str | Path=DEFAULT_CONFIG) -> dict[str, Any]:
    """Inspect saved artifacts without repeating training or changing selection."""
    config = _pilot_read(REPO_ROOT / config_path)
    pointer = REPO_ROOT / config['paths']['output_dir'] / 'manifests/pilot.json'
    if not pointer.is_file():
        raise FileNotFoundError('Run python -m part1.snli_pilot before inspecting notebook section 6.')
    record = _pilot_read(pointer)
    marker = Path(record['pilot_manifest'])
    if _features_sha256(marker) != record['sha256']:
        raise ValueError('Pilot manifest changed.')
    report = _pilot_load_report(marker)
    if report['manifest']['specification']['base_configuration']['representations'] != config['representations']:
        raise ValueError('Saved pilot representations differ from the current configuration; run python -m part1.snli_pilot.')
    return report

In [ ]:
pilot_report = load_pilot_report()
pilot_manifest = pilot_report["manifest"]
pilot_tables = pilot_report["tables"]
frozen_config = pilot_report["frozen_config"]

print("Pilot ID:", pilot_manifest["pilot_id"])
print("Training rows/groups:", pilot_manifest["training_rows"], pilot_manifest["training_groups"])
print("Validation rows/groups:", pilot_manifest["validation_rows"], pilot_manifest["validation_groups"])
print("Pilot vocabulary/bigrams:", pilot_manifest["preprocessing"]["actual_vocab_size"], pilot_manifest["preprocessing"]["bigram_types"])
print("Whole pilot wall time (seconds):", round(pilot_manifest["elapsed_seconds"], 2))
display(pilot_tables["class_counts"])

### What the old validation curves show

Training NLL fell while validation NLL generally worsened with longer training.
This motivates checking generalization and regularization, rather than assuming
more updates will improve accuracy. The best schedule on 6,001 training examples
need not be the best schedule on full training data.

In [ ]:
display(pilot_tables["epoch_selection"].round(6))
display(Image(filename=str(pilot_report["learning_curve_figure"])))
training_curve_summary = pilot_tables["curves"].groupby(["representation", "classifier"]).agg(
    first_online_train_nll=("train_nll_online", "first"),
    last_online_train_nll=("train_nll_online", "last"),
    first_validation_nll=("validation_nll", "first"),
    last_validation_nll=("validation_nll", "last"),
)
display(training_curve_summary.round(4))

### Historical budget and configuration

The following projections explain why the baseline used 60,005 rows. They are
planning estimates, not timings for the proposed search. The saved frozen
configuration remains unchanged so the old results can still be checked.

In [ ]:
runtime_projection = pilot_tables["runtime_projection"].copy()
runtime_projection["estimated_minutes"] = runtime_projection["estimated_primary_seconds"] / 60
runtime_projection["minutes_with_margin"] = runtime_projection["conservative_primary_seconds"] / 60
display(runtime_projection[["target_rows", "estimated_minutes", "minutes_with_margin", "within_budget", "selected"]].round(2))

memory_table = pilot_tables["matrix_summary"].copy()
memory_table["csr_MiB"] = memory_table["csr_bytes"] / 2**20
display(memory_table[["partition", "representation", "rows", "columns", "nonzeros", "csr_MiB"]].round(3))
display(pilot_tables["comparison_pool_summary"])
first_fold = pilot_tables["comparison_fold_summary"].iloc[0]
comparison_counts = pd.DataFrame([
    {"label": label, "label_name": name,
     "rows": int(first_fold[f"train_{name}"] + first_fold[f"heldout_{name}"])}
    for label, name in enumerate(LABEL_NAMES)
])
comparison_counts["proportion"] = comparison_counts["rows"] / comparison_counts["rows"].sum()
display(comparison_counts)

In [ ]:
frozen_path = Path(pilot_manifest["frozen_configuration"]["path"])
frozen_evaluation = prepare_evaluation_protocol(frozen_path)
assert frozen_evaluation["reused_folds"]
assert len(frozen_evaluation["cv_pool"]) == pilot_manifest["comparison_actual_rows"]
assert all(frozen_evaluation["manifest"]["checks"].values())
print("Frozen configuration:", frozen_path)
print("Fixed neural epochs / threads:", frozen_config["neural_training"]["num_epochs"], frozen_config["neural_training"]["num_threads"])
print("Final CV protocol:", frozen_evaluation["manifest_path"])
print("Actual comparison rows:", len(frozen_evaluation["cv_pool"]))
print("Pilot evidence:", pilot_report["manifest_path"])
display(pilot_tables["comparison_fold_summary"][[
    "seed", "fold", "train_rows", "heldout_rows", "shared_premise_groups",
]])

## Sources and assistance

- [Project 1 requirements](project.ipynb), [tutorial 1](tutorial.ipynb), and the
  [four-deck method map](COURSE_METHODS.md).
- [SNLI dataset](https://huggingface.co/datasets/stanfordnlp/snli) and
  [SNLI paper](https://aclanthology.org/D15-1075/).
- [Course practical used as a presentation reference](https://github.com/S-Chaudhuri/uva_nlp1_2025/blob/main/NLP1_2025_Practical_2_%28student%29.ipynb): short explanations, worked inputs,
  visible computations, model progression and learning curves. Our text/code
  and SNLI experiments are developed for the local Project 1 scope.
- [Baycomp API](https://baycomp.readthedocs.io/en/latest/classes.html),
  [correlated-t implementation](https://baycomp.readthedocs.io/en/latest/_modules/baycomp/single.html),
  and [Benavoli et al., Section 3](https://www.jmlr.org/papers/volume18/16-305/16-305.pdf).
- [Hyperparameter search](https://scikit-learn.org/stable/modules/grid_search.html)
  and [selection/CV separation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).
- [Measured findings](PROJECT1_FINDINGS.md), [execution plan](PROJECT1_PLAN.md)
  and [AI-assistance record](AI_ASSISTANCE.md).

This notebook was developed with coding-assistant support. The group is
responsible for checking, understanding and explaining the implementation.